# Weakly-supervised HER2 localisation in gastro-oesophageal adenocarcinoma TMA cores, without pixel masks

**Dataset.** Pisula *et al.*, *Br J Cancer* 2023 (Zenodo 10.5281/zenodo.7031868, CC-BY 4.0). It has 1,598 training cores: a multi-spot TMA with
8 cores per tumour for 165 cases, and a single-spot TMA with 428 cases. The test set is 307 single-spot cores from **later** patients.
Each core has two labels: an IHC score (0–3) and the HER2 status (score 2 was resolved by FISH). The test set has **no IHC 1+ cores**.
There are **no patient identifiers and no pixel masks**.

## What the labels can and cannot teach (decided *before* any architecture choice)
| Target | Supervision available | How this notebook handles it | Status of the output |
|---|---|---|---|
| Core HER2 status / IHC score | Direct (image level) | Multiple-instance learning (MIL) on 20× tiles, evaluated on the temporally independent test set | **Validated** (classification metrics with CIs) |
| *Where* HER2-positive evidence lies | Indirect. Positive cores contain some positive tiles. IHC 0/1+ cores contain **no** 3+-type evidence, which is an exact negative constraint | Instance-level positivity, with the negative-bag constraint and a top-k positive constraint | Partially checkable: the false-positive rate on IHC-0 cores is a *real* error measure. Everything else is a proxy |
| DAB chromogen amount | Physics (Beer–Lambert colour deconvolution) | Absolute optical density with **one** dataset-level stain matrix and label-anchored thresholds | Measurement, not a prediction |
| *Membranous* versus cytoplasmic or nuclear DAB | **None.** This is a morphological judgement | Hessian line-likeness, a white top-hat local-prominence test, and haematoxylin-dominant nuclear exclusion, distilled into a CNN student trained with ignore regions and confidence weights | **Unverified hypothesis** until expert masks exist |
| Tumour versus stroma | **None.** HER2 labels say nothing about tissue type | Unsupervised tile clusters plus a gallery that a pathologist names in about 10 minutes | Unverified until the clusters are named |
| Cells and membrane completeness | **None** | Nuclei from haematoxylin, expanded into territories, then the fraction of each territory boundary that is DAB-membranous | Approximate morphometry and a proxy only |

**Final rule enforced by the code:** with no reference masks, the notebook never prints Dice or IoU as accuracy. Agreement with pseudo-labels is reported
under that name. Section 17 computes true Dice, IoU, HD95 and boundary F1 automatically once expert masks are supplied.

### Pathology conventions used
* Gastric/GEA HER2 scoring follows Hofmann/Rüschoff. Membranous reactivity can be **complete, basolateral or lateral (U-shaped)**, so circumferential staining is *not* required.
  Only membranous staining counts. Cytoplasmic and nuclear DAB are excluded from "membrane" by design.
* IHC 2+ is **equivocal**. Its status comes from FISH. The pipeline reports a status probability for 2+ cores but always routes them to "ISH required".
* The periphery of a TMA core is prone to edge artefact. A rim band is measured separately and ignored in the pseudo-labels.
* The weak/moderate/strong DAB bands below are *data-derived operating points*, not validated equivalents of a pathologist's intensity grading.

## Version 2: changes driven by the audit of the v1 run (full run on 1,905 cores, Tesla T4)
| # | v1 finding (evidence) | v2 change |
|---|---|---|
| 1 | **Bug.** The residual stain vector was abs(H×DAB), nearly parallel to H (cos 0.988; condition number 15). This inflated or zeroed the haematoxylin channel. | Correct H×DAB normal (condition number 3). All features are recomputed in a fresh cache (`her2_wsl_v2`). |
| 2 | **Bug.** The opening-based "thick DAB" test removed 30% of pixels in 3+ crops, so pseudo-labels taught that strong 3+ membranes are *not* membrane. IHC 3 cores showed less membrane than 2+. | A white top-hat local-prominence test plus a stricter line test. On 20 training crops, 3+ membrane coverage rose about 6× and 3+/0 contrast rose about 3×. |
| 3 | Dark DAB was read as nuclei, because nuclei were defined as "H ≥ t_h" only. | Nuclei must be haematoxylin-**dominant** (H ≥ t_h and H ≥ DAB). |
| 4 | The dark-artefact QC flag fired on 25% of test IHC 3 cores, because strong DAB looks "dark". | Dark artefact = very dark **and** near-neutral colour. |
| 5 | 5 test cores with fewer than 8 tissue tiles were silently dropped from evaluation. | Test cores are never dropped. They are kept, flagged low-tissue, and routed to "refer". |
| 6 | ISH routing with a fixed P(IHC2) ≥ 0.3 caught only 40% of IHC 2 test cores. | Threshold tuned on OOF predictions for 80% sensitivity. Test sensitivity and specificity are reported. |
| 7 | Cell coverage did not separate 2+ from 3+, because territories were fixed 4 µm expansions unrelated to membranes. | Membrane-guided watershed: nuclei are the seeds and the membrane map is the barrier. |
| 8 | Claims rested on single training runs. | Repeated CV seeds (`N_SEEDS`) for the key variants, reported as mean ± SD. |
| 9 | There was no CAM baseline (the approach of the original draft) and no pathology-domain encoder. | Added a Grad-CAM thumbnail CNN baseline, scored with identical metrics, and a Phikon (pathology foundation model) encoder ablation from the same pass. |
| 10 | Segmentation training froze twice in forked loader workers. The best epoch was not kept. | Single-process loading, per-epoch checkpoint and resume, and best-epoch selection. |
| 11 | A localisation sensitivity check was missing. | IHC 3+ core detection and tile AUROC (tiles from IHC 3 cores versus IHC 0 cores) added alongside the IHC 0 false-positive rate. |

The v1 architecture finding still stands. MSA-U-Net-TR did not beat a plain U-Net (same-budget high-confidence agreement 0.84 vs 0.87). Both are
kept as a reported ablation, and no architectural novelty is claimed.

## Version 2.1: restartability and additional evidence
**Restart from where it stopped.** Every expensive loop now saves each finished item at once.
* **How items are saved.** Most loops write each finished item to an append-only log (`cache/logs/*.jsonl`) and flush it to disk; feature extraction writes each core's `npz` file. Re-running a cell skips finished items and continues with the rest.
* **What is covered.** Feature extraction (per core), MIL runs (per variant, fold set and seed), Grad-CAM (per fold), pseudo-label crops, core and cell features, and whole-core test segmentation (per core). Segmentation training resumes per epoch.
* **Crash safety.** All cache files are written to a temporary name and then renamed, so a crash can never leave a half-written file that looks finished.
* **Same result after a restart.** Random draws are seeded per core, so resumed output does not depend on where the run stopped.
* **Within one session:** just run all cells again.
* **In a new Kaggle session:** use *Add Input*, pick the stopped notebook's output, then *Run All*. The first cell finds `her2_wsl_v2` and copies it into `/kaggle/working`, and the run continues from there. To point at it explicitly, set the `HER2_RESUME_FROM` environment variable.

**New evidence for the manuscript.** Each item below is computed. None of it is asserted.
| Item | Output |
|---|---|
| H1: paired per-core Wilcoxon and bootstrap tests of IHC-0 false-positive tiles, per seed, on the test set **and** on out-of-fold training cores; CI for the FP rate | `tables/h1_negative_constraint_tests.csv`, `fig11` |
| H3: single-feature AUROC for 3+ vs 2+ and for 2+/FISH+ vs 2+/FISH−; direction fixed on train; stratified bootstrap CI; Holm correction | `tables/h3_feature_separation_3v2_and_fish.csv`, `fig12` |
| Holm-corrected p-values for every paired AUROC comparison with the proposed model | `tables/paired_delta_auc.csv` |
| Convergence curves (MIL per fold; segmentation per epoch, selected epoch marked) | `figS2_convergence` |
| Computational cost: parameters, per-core inference time, wall-clock per section | `tables/compute_*.csv` |
| Claims ledger wording gated on pre-stated criteria (e.g. H1 is called "supported" only if every seed's CI lies below 0) | `RESULTS_SUMMARY.md` |

In [ ]:
# ============================== 0. ENVIRONMENT & CONFIG ==============================
import os, sys, json, math, time, random, glob, hashlib, warnings, gc, re, pickle, contextlib, subprocess
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
import cv2
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy import stats
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
                             balanced_accuracy_score, f1_score, matthews_corrcoef, confusion_matrix,
                             cohen_kappa_score, brier_score_loss)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.cluster import AgglomerativeClustering, KMeans, MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

def _pip(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
try:
    import timm
except ImportError:
    _pip("timm"); import timm
try:
    import skimage
    from skimage.feature import peak_local_max
    from skimage.segmentation import watershed, expand_labels, find_boundaries
except ImportError:
    _pip("scikit-image"); import skimage
    from skimage.feature import peak_local_max
    from skimage.segmentation import watershed, expand_labels, find_boundaries
from tqdm.auto import tqdm
# Keep scikit-learn / NumPy single-threaded for the whole notebook. After PyTorch's DataLoader workers have forked,
# two OpenMP runtimes can dead-lock scikit-learn on Kaggle (seen in k-means). GPU work is unaffected; CPU steps here are small.
from threadpoolctl import threadpool_limits
_THREADS = threadpool_limits(limits=1)

RUN_MODE = os.environ.get("HER2_RUN_MODE", "full").lower()   # 'full' | 'fast' | 'debug'
IS_KAGGLE = Path("/kaggle/input").exists()

class CFG:
    # ---------- paths (None = auto-discover under DATA_ROOT) ----------
    # Your full-resolution Zenodo images (training_dataset/, testing_dataset/, train.csv, test.csv).
    # Pointing at this folder (not all of /kaggle/input) stops other attached datasets with the same filenames being used.
    DATA_ROOT      = os.environ.get("HER2_DATA_ROOT", next((p for p in ["/kaggle/input/datasets/ashishpandharimohod/her2dataset",
                                                               "/kaggle/input/her2dataset"] if Path(p).exists()),
                                                      "/kaggle/input" if IS_KAGGLE else "./data"))
    MIN_CORE_PX    = 2000      # cores smaller than this cannot resolve membranes/cells -> stop with an explanation
    ALLOW_LOWRES   = False     # set True only for a classification-only experiment on downsampled images
    TRAIN_CSV      = None
    TEST_CSV       = None
    REF_MASK_DIR   = os.environ.get("HER2_REF_MASK_DIR") or None   # folder of expert masks (PNG, same stem as image or ROI) -> enables true Dice/IoU
    CLUSTER_NAMING_CSV = None  # filled cluster-naming sheet from a pathologist (Section 9) -> tumour-candidate maps
    OUT            = Path(os.environ.get("HER2_OUT", "/kaggle/working/her2_wsl_v2" if IS_KAGGLE else "./her2_wsl_out"))   # v2: fresh cache (stain matrix fixed)
    SEED           = 2024
    NUM_WORKERS    = 4

    # ---------- physical scale ----------
    NATIVE_MPP     = 0.23      # NanoZoomer S360 @40x (µm/px). Everything physical is derived from this.
    WORK_DS        = 2         # work level = native/2 ≈ 0.46 µm/px (≈20x): membranes are 2–4 px wide
    THUMB_REL      = 4         # thumbnail = work/4 ≈ 1.84 µm/px
    TILE           = 224       # MIL tile at work level ≈ 103 µm
    MIN_TILE_TISSUE= 0.5
    MAX_TILES_PER_IMAGE = 400
    MIN_TILES_KEEP = 8         # TRAIN cores with fewer tissue tiles are excluded; TEST cores are never dropped (>=1 tile kept, flagged)

    # ---------- stain / morphology ----------
    OD_TISSUE_RANGE= (0.05, 0.15)  # per-core Otsu threshold on max-channel OD, clipped to this range (tissue vs glass)
    OD_GAP         = 0.04      # at work level, pixels below this OD inside tissue are lumina/gaps
    OD_DARK        = 2.2       # summed OD above which a pixel may be a dark artefact ...
    DARK_MAX_CHROMA= 30        # ... but only if it is also near-neutral (max-min RGB <= this); strong brown DAB is NOT an artefact
    RIM_UM         = 60.0      # core periphery band (edge-artefact zone)
    RIDGE_SIGMAS_UM= (0.5, 0.75, 1.0)   # membrane half-widths probed by the Hessian
    RIDGE_LIN_MIN  = 0.75      # line-likeness (1-|λs|/|λb|) of a membrane ridge
    TOPHAT_R_UM    = 2.0       # white top-hat radius: membranes are thinner than ~4 µm
    TOPHAT_REL     = 0.55      # membrane DAB must exceed its ~4 µm surround by >= this fraction of its own OD
    TOPHAT_ABS     = 0.03      # ... and by >= this absolute OD
    NUC_DILATE_UM  = 0.8       # exclusion ring around haematoxylin-dominant nuclei
    # (v2) settings chosen by visual review of 20 TRAINING crops (4 per IHC group); see fig03. Unverified until expert review.
    Q_NEG          = 0.99      # t_weak = this quantile of membrane-shaped DAB OD in IHC-0 cores
    Q_BG           = 0.90      # t_bg   = this quantile (confident-negative level)
    N_CALIB_PER_SCORE = 40

    # ---------- encoder ----------
    ENCODER        = "timm:resnet50.tv_in1k"   # truncated after layer3 (CLAM convention, 1024-d)
    ENCODER_OUT_INDEX = 3
    ENCODER2       = "hf:owkin/phikon"         # pathology foundation model (ViT-B, open weights) for the encoder ablation; None to skip
    N_SEEDS        = 3         # repeated CV seeds for the key MIL variants (variability / CIs)
    RUN_GRADCAM    = True      # thumbnail-CNN + Grad-CAM baseline (the approach of the original draft / Pisula et al. ResNet)
    CAM_ARCH       = "resnet18.tv_in1k"
    CAM_EPOCHS     = 10
    CAM_BS         = 16
    ISH_ROUTE_TARGET_SENS = 0.80   # P(IHC2) routing threshold chosen on OOF to reach this sensitivity for IHC2 cores
    ENC_BATCH      = 128

    # ---------- MIL ----------
    N_FOLDS        = 5
    MIL_EPOCHS     = 40
    MIL_BS         = 16
    MIL_LR         = 2e-4
    MIL_WD         = 1e-3
    MIL_MAX_TILES_TRAIN = 256
    MIL_TILE_DROP  = 0.10
    TOPK_FRAC      = 0.05
    W_ORD, W_NEG, W_POS, W_CLAM = 0.5, 0.5, 0.5, 0.3
    MC_SAMPLES     = 10
    EXPECTED_PATIENTS = 593    # 165 multi-spot + 428 single-spot cases (paper)
    EXPECTED_CORES = 1748      # 165*8 + 428 before the authors discarded corrupted images

    # ---------- tissue clusters ----------
    N_TISSUE_CLUSTERS = 12

    # ---------- pseudo-labels & segmentation student ----------
    SEG_CROP       = 512
    CROPS_PER_IMAGE= 8
    SEG_ENCODER    = "resnet34.tv_in1k"
    SEG_EPOCHS     = 10
    SEG_ABL_EPOCHS = 4
    SEG_BS         = 8
    SEG_LR         = 3e-4
    EV_P_POS, EV_P_NEG, EV_STD_MAX = 0.80, 0.20, 0.15
    W_EV           = 0.5
    TTA            = 4
    MIN_MEM_OBJ_PX = 12
    FEAT_CROPS_PER_CORE = 12   # crops per core for core-level / cell-level features (same protocol train & test)
    CELL_NUC_MIN_UM= 3.0
    CELL_MAX_UM    = 8.0       # (v2) max cell radius from its nucleus for the membrane-guided watershed
    SAVE_FULLRES_MASKS = True

    # ---------- evaluation ----------
    N_BOOT         = 2000
    ABSTAIN_COVERAGE = 0.90
    N_ANNOT_ROIS_PER_CORE = 3

if RUN_MODE == "fast":
    CFG.N_SEEDS, CFG.CAM_EPOCHS = 2, 6
    CFG.MIL_EPOCHS, CFG.SEG_EPOCHS, CFG.SEG_ABL_EPOCHS = 25, 6, 2
    CFG.CROPS_PER_IMAGE, CFG.FEAT_CROPS_PER_CORE, CFG.MC_SAMPLES, CFG.N_BOOT = 6, 8, 5, 1000
if RUN_MODE == "debug":          # tiny synthetic run on CPU, only to prove the code path end-to-end
    CFG.TILE, CFG.SEG_CROP, CFG.MAX_TILES_PER_IMAGE, CFG.MIN_TILES_KEEP = 64, 128, 400, 2
    CFG.MIL_EPOCHS, CFG.SEG_EPOCHS, CFG.SEG_ABL_EPOCHS, CFG.N_FOLDS = 4, 2, 1, 3
    CFG.CROPS_PER_IMAGE, CFG.FEAT_CROPS_PER_CORE, CFG.MC_SAMPLES, CFG.N_BOOT = 3, 3, 3, 100
    CFG.N_CALIB_PER_SCORE, CFG.N_TISSUE_CLUSTERS, CFG.ENC_BATCH, CFG.SEG_BS = 6, 4, 32, 4
    CFG.NUM_WORKERS, CFG.TTA, CFG.RIM_UM = 0, 2, 20.0
    CFG.ENCODER, CFG.SEG_ENCODER, CFG.ENCODER2, CFG.CAM_ARCH = "timm:resnet18", "resnet18", None, "resnet18"
    CFG.N_SEEDS, CFG.CAM_EPOCHS, CFG.CAM_BS = 2, 2, 8

OUT = Path(CFG.OUT); CACHE = OUT / "cache"; FIG = OUT / "figures"; TAB = OUT / "tables"

# ---------------- cross-session resume ----------------
# A Kaggle session wipes /kaggle/working. To continue a stopped run in a NEW session: open the stopped/saved version, then
# "Add Input" -> "Your Work" -> this notebook's output. Any attached folder that contains her2_wsl_v2/cache/calibration.json
# is copied into OUT before anything runs, and every stage below skips the work already present.
CFG.RESUME_FROM = os.environ.get("HER2_RESUME_FROM", None)
def _find_previous_run():
    if CFG.RESUME_FROM: return Path(CFG.RESUME_FROM)
    if not IS_KAGGLE: return None
    for cal in Path("/kaggle/input").rglob("calibration.json"):
        root = cal.parent.parent
        if root.name == OUT.name and (root / "cache" / "feats").exists(): return root
    return None
_prev = _find_previous_run()
_copy_flag = OUT / ".resume_copy_in_progress"
if _prev is not None and _prev.resolve() != OUT.resolve() and (_copy_flag.exists() or not (OUT / "cache" / "calibration.json").exists()):
    import shutil
    OUT.mkdir(parents=True, exist_ok=True); _copy_flag.touch()          # an interrupted copy is simply redone next time
    print(f"RESUME: copying previous run {_prev} -> {OUT} (one-off, a few minutes)")
    shutil.copytree(_prev, OUT, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns("*.zip", ".resume_copy_in_progress"))   # the old review zip is rebuilt at the end
    _copy_flag.unlink(missing_ok=True)
    print("RESUME: done; finished stages will be loaded from cache, unfinished ones continue")

for d in [OUT, CACHE, FIG, TAB, CACHE / "feats", CACHE / "maps", CACHE / "thumbs", CACHE / "mil", CACHE / "logs",
          CACHE / "seg_crops", OUT / "models", OUT / "test_outputs", OUT / "annotation_worklist"]:
    d.mkdir(parents=True, exist_ok=True)

# ---------------- crash-safe persistence helpers ----------------
# Every cache file is written to a temporary name and atomically renamed, so an interrupted write can never leave a
# truncated file that a restart would mistake for finished work.
def atomic_write(path, write_fn):
    path = Path(path); tmp = path.with_name(path.name + ".part")
    with open(tmp, "wb") as f:
        write_fn(f); f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)
def atomic_pickle(obj, path): atomic_write(path, lambda f: pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL))
def atomic_npz(path, compressed=False, **arrays): atomic_write(path, lambda f: (np.savez_compressed if compressed else np.savez)(f, **arrays))
def atomic_torch_save(obj, path): atomic_write(path, lambda f: torch.save(obj, f))
def atomic_csv(df, path, **kw): atomic_write(path, lambda f: f.write(df.to_csv(**kw).encode()))
def load_pickle(path):
    try:
        with open(path, "rb") as f: return pickle.load(f)
    except Exception:
        return None

class ProgressLog:
    """Append-only JSON-lines log for long per-item loops: each finished item is flushed to disk immediately, so a restart
    skips everything already done. A truncated last line (crash mid-write) is ignored and that item is simply redone."""
    def __init__(self, name):
        self.path = CACHE / "logs" / f"{name}.jsonl"; self.items = {}
        if self.path.exists():
            for line in open(self.path):
                try: rec = json.loads(line); self.items[rec["_key"]] = rec
                except Exception: pass
    def __contains__(self, key): return key in self.items
    def __len__(self): return len(self.items)
    def add(self, key, rec):
        rec = dict(rec, _key=key)
        with open(self.path, "a") as f:
            f.write(json.dumps(rec, default=lambda o: o.item() if hasattr(o, "item") else str(o)) + "\n"); f.flush(); os.fsync(f.fileno())
        self.items[key] = rec
    def records(self): return list(self.items.values())

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def autocast():
    return torch.autocast(device_type="cuda", dtype=torch.float16) if DEVICE.type == "cuda" else contextlib.nullcontext()

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything(CFG.SEED)
torch.backends.cudnn.benchmark = True

WORK_MPP = CFG.NATIVE_MPP * CFG.WORK_DS
THUMB_MPP = WORK_MPP * CFG.THUMB_REL
def um2px(um, mpp=None):
    return um / (mpp or WORK_MPP)

_T0 = {"t": time.time(), "last": time.time()}
SESSION_ID = time.strftime("%Y%m%d-%H%M%S")   # timings from interrupted + resumed sessions are summed per section
# ---------------- memory / disk guard ----------------
# The 10 GB image set lives read-only in /kaggle/input and is streamed one core at a time: it never occupies RAM or the
# /kaggle/working quota. What does consume resources is logged per section (RAM, peak GPU memory, output-folder size).
try: import psutil
except ImportError: psutil = None
def free_disk_gb(path=None):
    import shutil as _sh; return _sh.disk_usage(str(path or OUT)).free / 1e9
def dir_size_gb(path):
    tot = 0
    for root, _, files in os.walk(path):
        for f in files:
            try: tot += os.path.getsize(os.path.join(root, f))
            except OSError: pass
    return tot / 1e9
def resource_report(tag="startup"):
    ram = psutil.virtual_memory() if psutil else None
    gpu = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
    print(f"[resources:{tag}] RAM total {ram.total / 1e9:.1f} GB, available {ram.available / 1e9:.1f} GB | " if ram else "[resources] ",
          f"GPU {gpu:.1f} GB | free disk in {OUT.parent}: {free_disk_gb():.1f} GB")
    if free_disk_gb() < 12: print("!! WARNING: less than 12 GB free disk; a full run writes about 6-8 GB. Full-resolution test masks "
                                  "will be skipped automatically if space runs low.")
    if ram and ram.total < 12e9: print("!! WARNING: less than 12 GB RAM; set CFG.NUM_WORKERS = 2 if the kernel restarts.")

def section_time(name):
    """Log wall-clock time, current RAM, peak GPU memory and output size per notebook section (computational-cost table),
    then release cached memory before the next section."""
    now = time.time(); dt = now - _T0["last"]; _T0["last"] = now
    gc.collect()
    ram = (psutil.Process().memory_info().rss / 1e9) if psutil else float("nan")
    vram = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
    if torch.cuda.is_available(): torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    disk = dir_size_gb(OUT)
    print(f"[time] {name}: {dt / 60:.1f} min (total {(now - _T0['t']) / 60:.1f} min) | RAM {ram:.1f} GB | "
          f"peak GPU {vram:.1f} GB | output {disk:.1f} GB, free disk {free_disk_gb():.1f} GB")
    import csv
    new = not (OUT / "timings.csv").exists()
    with open(OUT / "timings.csv", "a", newline="") as f:      # csv-quoted: section names contain commas
        w = csv.writer(f)
        if new: w.writerow(["section", "seconds", "session", "ram_gb", "peak_gpu_gb", "output_gb"])
        w.writerow([name, f"{dt:.1f}", SESSION_ID, f"{ram:.2f}", f"{vram:.2f}", f"{disk:.2f}"])

def gpu_safe(fn, x, min_bs=1):
    """Run fn on a batch; on CUDA out-of-memory, free the cache and retry in halves (results are concatenated in order).
    Turns a would-be crash into a slower but identical computation."""
    try: return fn(x)
    except torch.cuda.OutOfMemoryError:
        if len(x) <= min_bs: raise
        torch.cuda.empty_cache(); h = len(x) // 2
        print(f"  (GPU memory full: retrying a batch of {len(x)} as 2 x {h})")
        a, b = gpu_safe(fn, x[:h], min_bs), gpu_safe(fn, x[h:], min_bs)
        return tuple(torch.cat([u, v]) for u, v in zip(a, b)) if isinstance(a, tuple) else torch.cat([a, b])

def save_json(obj, path):
    txt = json.dumps(obj, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
    atomic_write(path, lambda f: f.write(txt.encode()))

# ---------- publication figure style (validated categorical order; text in neutral ink) ----------
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
GROUP_ORDER = ["0", "1", "2-", "2+", "3"]
GROUP_COL = {"0": "#2a78d6", "1": "#1baf7a", "2-": "#eda100", "2+": "#eb6834", "3": "#e34948"}
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9, "axes.titlesize": 10,
                     "axes.labelsize": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6,
                     "axes.edgecolor": "#52514e", "axes.labelcolor": "#0b0b0b", "xtick.color": "#52514e",
                     "ytick.color": "#52514e", "legend.frameon": False, "lines.linewidth": 2})
def savefig(fig, name):
    fig.tight_layout()
    fig.savefig(FIG / f"{name}.png", bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    if matplotlib.get_backend().lower() != "agg":
        plt.show()
    plt.close(fig)

print(f"RUN_MODE={RUN_MODE} | device={DEVICE} | torch={torch.__version__} | timm={timm.__version__}")
resource_report("startup")
print(f"work level {WORK_MPP:.2f} µm/px | tile {CFG.TILE}px = {CFG.TILE*WORK_MPP:.0f} µm | out={OUT}")

## 1. Data discovery and validation
This section looks for `train.csv`, `test.csv` and the image folders anywhere under `DATA_ROOT`, and accepts any common column spelling.
It then checks the following:
* every labelled file exists
* the labels obey the scoring rules: 0 and 1+ are negative, 3+ is positive, 2+ can be either
* the training filename prefix agrees with the IHC score
* the image geometry is consistent
* there are no exact duplicates within or across splits

Near-duplicate detection runs later, using thumbnail hashes.

In [ ]:
# ---------------- (debug only) synthetic HER2-like cores, so the full code path can be exercised without the real data ----
def make_synthetic_dataset(root, n_train=48, n_test=18, size=1024, seed=0):
    rng = np.random.default_rng(seed)
    root = Path(root); (root / "train").mkdir(parents=True, exist_ok=True); (root / "test").mkdir(parents=True, exist_ok=True)
    vH, vD = np.array([0.65, 0.70, 0.29]), np.array([0.27, 0.57, 0.78])
    from scipy.spatial import cKDTree
    def core(score):
        yy, xx = np.mgrid[:size, :size]
        r = size * 0.42; cy, cx = size / 2 + rng.normal(0, 10), size / 2 + rng.normal(0, 10)
        tissue = (yy - cy) ** 2 + (xx - cx) ** 2 < r ** 2
        pts = rng.uniform(0, size, (int(size * size / 900), 2))
        lab = cKDTree(pts).query(np.stack([yy.ravel(), xx.ravel()], 1))[1].reshape(size, size)
        epi = ndi.gaussian_filter(rng.normal(0, 1, (size // 32, size // 32)), 1.5)
        epi = cv2.resize(epi, (size, size)) > 0.0
        bnd = find_boundaries(lab, mode="thick") & epi & tissue
        dist = np.linalg.norm(np.stack([yy.ravel(), xx.ravel()], 1) - pts[lab.ravel()], axis=1).reshape(size, size)
        nuc = (dist < 7) & tissue
        cH = 0.16 * tissue + 0.55 * nuc + 0.03 * rng.random((size, size)) * tissue
        amp = {0: 0.0, 1: 0.18, 2: 0.45, 3: 0.95}[score]
        keep = ndi.gaussian_filter(rng.random((size, size)), 6) > ({0: 1, 1: 0.55, 2: 0.45, 3: 0.3}[score])
        cD = amp * ndi.gaussian_filter((bnd & keep).astype(float), 0.8) * 1.8 + 0.02 * tissue * rng.random((size, size))
        od = cH[..., None] * vH + cD[..., None] * vD
        rgb = np.clip(245 * np.exp(-od) + rng.normal(0, 2, od.shape), 0, 255).astype(np.uint8)
        return rgb
    rows = {"train": [], "test": []}
    for split, n in [("train", n_train), ("test", n_test)]:
        for i in range(n):
            score = int(rng.choice([0, 0, 1, 2, 3, 3])) if split == "train" else int(rng.choice([0, 0, 2, 3]))
            status = 0 if score < 2 else (1 if score == 3 else int(rng.random() < 0.5))
            name = f"{score}_{i:04d}.jpg" if split == "train" else f"TMA_SP_{i:03d}.jpg"
            cv2.imwrite(str(root / split / name), cv2.cvtColor(core(score), cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 92])
            rows[split].append({"filename": name, "ihc-score": score, "her2-status": status})
    pd.DataFrame(rows["train"]).to_csv(root / "train.csv", index=False)
    pd.DataFrame(rows["test"]).to_csv(root / "test.csv", index=False)

if RUN_MODE == "debug" and not (Path(CFG.DATA_ROOT).exists() and any(Path(CFG.DATA_ROOT).rglob("train.csv"))):
    print("DEBUG: generating synthetic cores at", CFG.DATA_ROOT)
    make_synthetic_dataset(CFG.DATA_ROOT)

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
def _norm(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

ARCHIVE_EXT = (".zip", ".tar", ".tar.gz", ".tgz", ".rar", ".7z")
EXTRACT_DIR = Path("/tmp/her2_extracted" if IS_KAGGLE else "./extracted")   # /tmp: does not count against the 20 GB /kaggle/working limit

def discover(root):
    """Walk `root` FOLLOWING symlinks (Kaggle mounts dataset sub-folders as links) and index CSVs, images, archives."""
    csvs, images, archives, seen = {}, defaultdict(list), [], set()
    for dp, dn, fns in os.walk(root, followlinks=True):
        real = os.path.realpath(dp)
        if real in seen or real.startswith(str(OUT.resolve())) or real.startswith("/kaggle/working/her2_wsl"):
            dn[:] = []; continue
        seen.add(real)
        for fn in fns:
            p = Path(dp) / fn; low = fn.lower()
            if low == "train.csv" and "train" not in csvs: csvs["train"] = p
            elif low == "test.csv" and "test" not in csvs: csvs["test"] = p
            elif p.suffix.lower() in IMG_EXT:
                images[low].append(p); images["stem::" + Path(low).stem].append(p)
            elif low.endswith(ARCHIVE_EXT): archives.append(p)
    return csvs, images, archives

def extract_archives(archives, dest):
    """Unpack image archives that Kaggle did not auto-extract (.zip/.tar natively; .rar/.7z via installed tools)."""
    import zipfile, tarfile, shutil
    dest.mkdir(parents=True, exist_ok=True)
    for a in archives:
        out = dest / re.sub(r"\.(zip|tar|tar\.gz|tgz|rar|7z)$", "", a.name, flags=re.I)
        if out.exists() and any(out.rglob("*")): continue
        print(f"extracting {a} -> {out} (one-off; may take several minutes)")
        low = a.name.lower()
        try:
            if low.endswith(".zip"): zipfile.ZipFile(a).extractall(out)
            elif low.endswith((".tar", ".tar.gz", ".tgz")): tarfile.open(a).extractall(out)
            else:
                if shutil.which("unrar") is None and shutil.which("7z") is None:
                    subprocess.run("apt-get -qq install -y unrar p7zip-full > /dev/null 2>&1", shell=True)
                out.mkdir(parents=True, exist_ok=True)
                cmd = ["unrar", "x", "-o+", "-inul", str(a), str(out) + "/"] if shutil.which("unrar") and low.endswith(".rar") else ["7z", "x", "-y", f"-o{out}", str(a)]
                subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL)
        except Exception as e:
            print(f"!! could not extract {a.name}: {e}. Extract it locally and upload the image folders instead.")

def standardize_labels(df, split):
    m = {_norm(c): c for c in df.columns}
    pick = lambda keys: next((m[k] for k in keys if k in m), None)
    fcol = pick(["filename", "file", "image", "imagename", "imageid", "path", "img", "name"])
    scol = pick(["ihcscore", "score", "ihc", "her2score"])
    tcol = pick(["her2status", "status", "her2", "label"])
    if fcol is None:
        raise ValueError(f"{split}.csv: no filename column in {list(df.columns)}")
    out = pd.DataFrame({"filename": df[fcol].astype(str).map(lambda s: Path(s).name)})
    out["ihc_score"] = pd.to_numeric(df[scol], errors="coerce") if scol else np.nan
    def st(v):
        if isinstance(v, str):
            v2 = v.strip().lower()
            if v2 in ("positive", "pos", "1", "true", "amplified"): return 1
            if v2 in ("negative", "neg", "0", "false", "not amplified"): return 0
            return np.nan
        return v
    out["her2_status"] = pd.to_numeric(df[tcol].map(st), errors="coerce") if tcol else np.nan
    out["split"] = split
    return out

csvs, IMG_INDEX, ARCHIVES = discover(CFG.DATA_ROOT)
TRAIN_CSV = Path(CFG.TRAIN_CSV) if CFG.TRAIN_CSV else csvs.get("train")
TEST_CSV = Path(CFG.TEST_CSV) if CFG.TEST_CSV else csvs.get("test")
assert TRAIN_CSV is not None and TEST_CSV is not None, f"train.csv/test.csv not found under {CFG.DATA_ROOT}"
def _match_rate():
    names = pd.concat([standardize_labels(pd.read_csv(p), s).filename for s, p in [("train", TRAIN_CSV), ("test", TEST_CSV)]])
    return np.mean([(n.lower() in IMG_INDEX) or (("stem::" + Path(n.lower()).stem) in IMG_INDEX) for n in names])
if _match_rate() < 0.5 and ARCHIVES:
    print(f"Only {_match_rate():.0%} of CSV filenames match loose images; found archive(s) {[a.name for a in ARCHIVES]} -> extracting")
    extract_archives(ARCHIVES, EXTRACT_DIR)
    _, extra, _ = discover(str(EXTRACT_DIR))
    for k, v in extra.items(): IMG_INDEX[k].extend(v)
n_img = sum(len(v) for k, v in IMG_INDEX.items() if not k.startswith("stem::"))
print("train csv:", TRAIN_CSV, "| test csv:", TEST_CSV, "| image files indexed:", n_img, f"| CSV match rate: {_match_rate():.1%}")

def resolve(fn, split):
    """Match a CSV filename (no path needed) to an indexed image: exact name (case-insensitive), then name without extension."""
    cands = IMG_INDEX.get(fn.lower(), []) or IMG_INDEX.get("stem::" + Path(fn.lower()).stem, [])
    if not cands: return None
    pref = [c for c in cands if split in str(c).lower()] or cands
    return str(max(pref, key=lambda c: os.path.getsize(c)))   # duplicate names: the largest (full-resolution) file wins

frames = []
for split, p in [("train", TRAIN_CSV), ("test", TEST_CSV)]:
    d = standardize_labels(pd.read_csv(p), split)
    d["path"] = [resolve(f, split) for f in d.filename]
    frames.append(d)
DF = pd.concat(frames, ignore_index=True)
if DF.path.notna().sum() == 0:
    tree = []
    for dp, dn, fns in os.walk(CFG.DATA_ROOT, followlinks=True):
        depth = dp[len(str(CFG.DATA_ROOT)):].count(os.sep)
        if depth <= 4: tree.append(f"{'  ' * depth}{os.path.basename(dp) or dp}/  ({len(fns)} files, e.g. {fns[:3]})")
    raise RuntimeError(
        f"None of the {len(DF)} filenames in the CSVs matched an image under {CFG.DATA_ROOT} "
        f"({n_img} images indexed; CSV example: {DF.filename.iloc[0]!r}; "
        f"indexed example: {next((k for k in IMG_INDEX if not k.startswith('stem::')), None)!r}; archives seen: {[a.name for a in ARCHIVES]}).\nFolder tree:\n" + "\n".join(tree[:60]) +
        "\nFix: attach the dataset that contains the train/test JPEGs (Add Input), or set CFG.DATA_ROOT to it.")
n_miss = int(DF.path.isna().sum())
if n_miss: print(f"WARNING: {n_miss} CSV rows have no matching image file (listed in data_validation.json)")
DF["image_id"] = DF.split + "__" + DF.filename.map(lambda s: Path(s).stem)

report = {}
report["n_rows"] = DF.groupby("split").size().to_dict()
report["missing_files"] = DF[DF.path.isna()].groupby("split").size().to_dict()
report["missing_labels"] = int(DF[["ihc_score", "her2_status"]].isna().any(axis=1).sum())
viol = DF[((DF.ihc_score <= 1) & (DF.her2_status == 1)) | ((DF.ihc_score == 3) & (DF.her2_status == 0))]
report["rule_violations(0/1+ positive or 3+ negative)"] = int(len(viol))
pref = DF[DF.split == "train"].filename.str.extract(r"^(\d)_")[0].astype(float)
report["train_prefix_vs_score_mismatch"] = int(((pref.notna()) & (pref != DF[DF.split == "train"].ihc_score)).sum())
DF = DF[DF.path.notna() & DF.ihc_score.notna() & DF.her2_status.notna()].reset_index(drop=True)
DF["ihc_score"] = DF.ihc_score.astype(int); DF["her2_status"] = DF.her2_status.astype(int)
DF["group"] = DF.ihc_score.astype(str)
DF.loc[DF.ihc_score == 2, "group"] = np.where(DF.loc[DF.ihc_score == 2, "her2_status"] == 1, "2+", "2-")

# geometry (header only) + quick content hash (size + first/last 1 MB) for exact duplicates
def header_and_hash(p):
    try:
        with Image.open(p) as im:
            w, h, fmt = im.size[0], im.size[1], im.format
    except Exception:
        return (np.nan, np.nan, None, None)
    sz = os.path.getsize(p); hsh = hashlib.md5(str(sz).encode())
    with open(p, "rb") as f:
        hsh.update(f.read(1 << 20)); f.seek(max(0, sz - (1 << 20))); hsh.update(f.read(1 << 20))
    return (w, h, fmt, hsh.hexdigest())
assert len(DF), "No labelled rows with a matching image remain - see the messages above."
meta = [header_and_hash(p) for p in tqdm(DF.path, desc="headers")]
DF["W"], DF["H"], DF["fmt"], DF["qhash"] = zip(*meta)
report["unreadable"] = int(DF.fmt.isna().sum())
DF = DF[DF.fmt.notna()].reset_index(drop=True)
_side = float(np.median(np.minimum(DF.W, DF.H)))
print(f"median core image size: {_side:.0f} px  ->  ≈{1200 / _side:.2f} µm/px if one image spans a 1.2 mm core")
if _side < CFG.MIN_CORE_PX and RUN_MODE != "debug" and not CFG.ALLOW_LOWRES:
    raise RuntimeError(
        f"The images are only ~{_side:.0f} px across (≈{1200 / _side:.1f} µm/px). At this scale a tumour cell is ~{12 * _side / 1200:.0f} px and a "
        "membrane is sub-pixel, so membrane/cell segmentation is physically impossible. Use the original full-resolution Zenodo cores "
        "(≈5468 px, 0.23 µm/px), or set CFG.ALLOW_LOWRES=True for a classification-only experiment.\nImages used came from e.g.: " + DF.path.iloc[0])
if abs(_side - 5468) / 5468 > 0.10 and _side >= CFG.MIN_CORE_PX:
    print(f"NOTE: cores are {_side:.0f} px, not the 5468 px in Pisula et al. If they were resized, set CFG.NATIVE_MPP = {0.23 * 5468 / _side:.3f}.")
report["image_sizes"] = DF.groupby(["W", "H"]).size().rename("n").reset_index().head(10).to_dict("records")
dup = DF[DF.duplicated("qhash", keep=False)]
report["exact_duplicate_files"] = int(len(dup))
report["exact_duplicates_across_splits"] = int(dup.groupby("qhash").split.nunique().gt(1).sum())
report["label_table"] = pd.crosstab(DF.split, DF.group).reindex(columns=GROUP_ORDER, fill_value=0).to_dict("index")
save_json(report, TAB / "data_validation.json")
print(json.dumps(report, indent=1, default=str))
display_tab = pd.crosstab(DF.split, DF.group).reindex(columns=GROUP_ORDER, fill_value=0)
print(display_tab)
if (display_tab.loc["test", "1"] if "test" in display_tab.index else 0) == 0:
    print("NOTE: the test set contains no IHC 1+ cores. Any 4-class test metric that reports a 1+ row is impossible.")
TRAIN = DF[DF.split == "train"].reset_index(drop=True)
TEST = DF[DF.split == "test"].reset_index(drop=True)

## 2. Image I/O and the stain/morphology engine
* **Reduced-resolution JPEG decoding.** PIL's `draft()` decodes the DCT directly at 1/2, 1/4 or 1/8 scale. A 5,468² core decodes at the
  ≈20× work level about 4× faster, with no resampling artefacts.
* **One dataset-level stain matrix.** It is estimated with Macenko on pooled pixels from a stratified sample and checked against Ruifrok's reference
  vectors. Background illumination is estimated per image. **DAB is never min–max normalised per image.** That normalisation would make an IHC-0
  core's faint background look like 3+ staining.
* **Membrane-shape test on the DAB optical-density image.** It combines three checks:
  1. *Ridge*: the multi-scale Hessian's dominant eigenvalue is negative and the structure is line-like.
  2. *Locally prominent*: a white top-hat (radius 2 µm) shows the DAB rises above its own surround by ≥55%. Diffuse
     cytoplasmic staining and solid blobs fail this; a membrane over weaker cytoplasm passes, even in dense 3+ tissue.
  3. *Not nuclear*: it lies outside haematoxylin-**dominant** nuclei (H ≥ t_h and H ≥ DAB).

  All three steps run on the GPU.

In [ ]:
def read_rgb(path, ds):
    """Decode at 1/ds of native resolution. Uses JPEG DCT scaling when possible."""
    im = Image.open(path)
    W, H = im.size
    tw, th = max(1, W // ds), max(1, H // ds)
    if ds > 1 and im.format == "JPEG":
        im.draft("RGB", (tw, th))
    im = im.convert("RGB")
    if im.size != (tw, th):
        im = im.resize((tw, th), Image.BILINEAR)
    return np.asarray(im)

RUIFROK_H = np.array([0.650, 0.704, 0.286]); RUIFROK_D = np.array([0.268, 0.570, 0.776])
def _unit(v): v = np.asarray(v, float); return v / (np.linalg.norm(v) + 1e-12)
def complete_matrix(h, d):
    """Third (residual) stain = unit normal of the H-DAB plane. v1 bug fixed: taking abs() of the cross product made the
    residual almost parallel to haematoxylin (cos 0.99, condition number 15 vs 3), corrupting the H channel."""
    h, d = _unit(h), _unit(d); r = _unit(np.cross(h, d))
    return np.stack([h, d, r]).astype(np.float32)

def estimate_I0(thumb):
    """Per-channel background illumination from the brightest (non-tissue) pixels."""
    flat = thumb.reshape(-1, 3).astype(np.float32)
    bright = flat[flat.sum(1) > np.percentile(flat.sum(1), 90)]
    return np.clip(np.median(bright, 0), 180, 255) if len(bright) else np.array([255., 255., 255.])

def od_np(rgb, I0):
    return np.maximum(-np.log((rgb.astype(np.float32) + 1.0) / (np.asarray(I0, np.float32) + 1.0)), 0)

def tissue_from_thumb(thumb, I0):
    od = od_np(thumb, I0); om = od.max(-1)
    # glass is the brightest mode and covers >=10% of a TMA-core image; threshold sits a fixed OD margin above it.
    # (Otsu was rejected: strongly DAB-stained cores pull it upward and erase pale stroma -> class-dependent tissue masks.)
    thr = float(np.clip(np.percentile(om, 10) + 0.045, *CFG.OD_TISSUE_RANGE))
    t = (om > thr).astype(np.uint8)
    k = max(3, int(round(um2px(8, THUMB_MPP))) | 1); se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    t = cv2.morphologyEx(cv2.morphologyEx(t, cv2.MORPH_CLOSE, se), cv2.MORPH_OPEN, se)
    n, lab, st, _ = cv2.connectedComponentsWithStats(t, 8)
    min_area = (50.0 / THUMB_MPP) ** 2            # drop specks < 50x50 µm
    keep = np.zeros(n, bool); keep[1:] = st[1:, cv2.CC_STAT_AREA] >= min_area
    t = keep[lab].astype(np.uint8)
    # core envelope = heavy closing + hole filling; rim = band inside the envelope boundary
    kc = max(3, int(round(um2px(120, THUMB_MPP))) | 1)
    env = cv2.morphologyEx(t, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kc, kc)))
    env = ndi.binary_fill_holes(env).astype(np.uint8)
    dist = cv2.distanceTransform(env, cv2.DIST_L2, 5) * THUMB_MPP
    rim = (dist > 0) & (dist < CFG.RIM_UM)
    chroma = thumb.max(-1).astype(np.int16) - thumb.min(-1).astype(np.int16)
    dark = (od.sum(-1) > CFG.OD_DARK) & (chroma <= CFG.DARK_MAX_CHROMA)   # neutral-dark only: strong (brown) DAB is not an artefact
    return t.astype(bool), rim, dark, env.astype(bool)

def _gauss_1d(sigma, device):
    r = int(math.ceil(3.5 * sigma)); x = torch.arange(-r, r + 1, dtype=torch.float32, device=device)
    g = torch.exp(-x ** 2 / (2 * sigma ** 2)); g = g / g.sum()
    g1 = -x / sigma ** 2 * g
    g2 = (x ** 2 / sigma ** 4 - 1 / sigma ** 2) * g
    return g, g1, g2

def sep_conv(img, kx, ky):
    """img (1,1,H,W); kx filters along width, ky along height."""
    px, py = kx.numel() // 2, ky.numel() // 2
    out = F.conv2d(F.pad(img, (px, px, 0, 0), mode="reflect"), kx.view(1, 1, 1, -1))
    return F.conv2d(F.pad(out, (0, 0, py, py), mode="reflect"), ky.view(1, 1, -1, 1))

def hessian_ridge(x, sigmas_px):
    """Bright-ridge detector. Returns (scale-normalised magnitude σ²|λ_big|, line-likeness ∈[0,1]); zero where not a ridge."""
    best_m = torch.zeros_like(x); best_l = torch.zeros_like(x)
    for s in sigmas_px:
        g, g1, g2 = _gauss_1d(s, x.device)
        Ixx = sep_conv(x, g2, g) * s * s; Iyy = sep_conv(x, g, g2) * s * s; Ixy = sep_conv(x, g1, g1) * s * s
        mean = (Ixx + Iyy) / 2; dev = torch.sqrt(((Ixx - Iyy) / 2) ** 2 + Ixy ** 2)
        l1, l2 = mean + dev, mean - dev
        big = torch.where(l1.abs() > l2.abs(), l1, l2); small = torch.where(l1.abs() > l2.abs(), l2, l1)
        ridge = big < 0
        mag = torch.where(ridge, -big, torch.zeros_like(big))
        lin = torch.where(ridge, 1 - small.abs() / (big.abs() + 1e-6), torch.zeros_like(big))
        upd = mag > best_m
        best_m = torch.where(upd, mag, best_m); best_l = torch.where(upd, lin, best_l)
    return best_m, best_l

def dilate(m, r):
    if r < 1: return m
    k = 2 * int(r) + 1
    return F.max_pool2d(m.float(), k, 1, k // 2) > 0.5
def erode(m, r):
    if r < 1: return m
    k = 2 * int(r) + 1
    return ~(F.max_pool2d((~m).float(), k, 1, k // 2) > 0.5)

class StainEngine:
    def __init__(self, M, calib=None):
        self.M = torch.tensor(M, dtype=torch.float32, device=DEVICE)
        self.Minv = torch.linalg.pinv(self.M)
        self.cal = calib
        self.sig = [um2px(s) for s in CFG.RIDGE_SIGMAS_UM]

    @torch.no_grad()
    def maps(self, rgb, I0, tissue_thumb=None, rim_thumb=None, need_ridge=True):
        """rgb: HxWx3 uint8 at work level. Returns dict of (1,1,H,W) tensors."""
        H, W = rgb.shape[:2]
        x = torch.from_numpy(np.ascontiguousarray(rgb)).to(DEVICE).float()
        I0t = torch.tensor(np.asarray(I0, np.float32), device=DEVICE)
        od = torch.clamp(-torch.log((x + 1) / (I0t + 1)), min=0)
        c = torch.clamp(od.reshape(-1, 3) @ self.Minv, min=0).reshape(H, W, 3)
        out = {"H": c[..., 0][None, None], "DAB": c[..., 1][None, None]}
        tis = od.max(-1).values[None, None] > CFG.OD_GAP
        if tissue_thumb is not None:
            tt = torch.from_numpy(tissue_thumb.astype(np.float32))[None, None].to(DEVICE)
            tis = tis & (F.interpolate(tt, size=(H, W), mode="nearest") > 0.5)
        out["tissue"] = tis
        if rim_thumb is not None:
            rt = torch.from_numpy(rim_thumb.astype(np.float32))[None, None].to(DEVICE)
            out["rim"] = (F.interpolate(rt, size=(H, W), mode="nearest") > 0.5) & tis
        else:
            out["rim"] = torch.zeros_like(tis)
        gray = (0.299 * x[..., 0] + 0.587 * x[..., 1] + 0.114 * x[..., 2])[None, None]
        out["gray"] = gray
        if not need_ridge:
            return out
        dab = out["DAB"]
        mag, lin = hessian_ridge(dab, self.sig)
        out["ridge_mag"], out["ridge_lin"] = mag, lin
        # white top-hat: DAB in excess of its ~4 um surround. Membranes rise above (weaker) cytoplasm; diffuse
        # cytoplasmic staining and solid blobs do not. (v1 used an opening-based "thick" test that erased 3+ membranes.)
        r = max(1, int(round(um2px(CFG.TOPHAT_R_UM)))); k = 2 * r + 1
        opened = F.max_pool2d(-F.max_pool2d(-dab, k, 1, r), k, 1, r)
        tophat = (dab - opened).clamp(min=0); out["tophat"] = tophat
        shape = (lin >= CFG.RIDGE_LIN_MIN) & (mag > 0) & (tophat >= CFG.TOPHAT_ABS) & (tophat >= CFG.TOPHAT_REL * dab) & tis
        g1 = _gauss_1d(1.0, DEVICE)[0]
        Hs, Ds = sep_conv(out["H"], g1, g1), sep_conv(dab, g1, g1)
        out["Hs"], out["Ds"] = Hs, Ds
        if self.cal is not None:
            nuc = (Hs >= self.cal["t_h"]) & (Hs >= Ds)          # haematoxylin-DOMINANT blobs only (dark DAB is not a nucleus)
            nuc = erode(dilate(nuc, 1), 1) & tis
            out["nuc"] = nuc
            mem_shape = shape & ~dilate(nuc, max(1, round(um2px(CFG.NUC_DILATE_UM))))
            out["mem_shape"] = mem_shape
            out["nonmem"] = (dab >= self.cal["t_weak"]) & tis & ~dilate(mem_shape, 1) & ~nuc   # cytoplasmic / blob DAB
            for kk in ("weak", "mod", "strong"):
                out[f"mem_{kk}"] = mem_shape & (dab >= self.cal[f"t_{kk}"])
        else:
            out["mem_shape"] = shape
        return out

## 3. Stain calibration (pass 0, stratified sample)
1. A Macenko stain matrix is estimated on pooled thumbnails, with up to `N_CALIB_PER_SCORE` cores per IHC score, and checked against Ruifrok.
2. The nuclear threshold `t_h` is the Otsu threshold of haematoxylin OD on tissue.
3. The DAB thresholds are **label-anchored**:
   * `t_weak` is the 99th percentile of membrane-shaped, non-nuclear DAB OD in **IHC-0** cores, i.e. "rarely reached by staining in pathologist-negative cores".
   * `t_bg` is the 90th percentile of the same values, the level below which a pixel is a confident negative.
   * `t_mod` and `t_strong` are the boundaries of a 3-component Gaussian mixture fitted to log-DAB above `t_weak`.

These thresholds are reproducible operating points. They are not a claim that they reproduce a pathologist's weak/moderate/strong grading.

In [ ]:
CALIB_PATH = CACHE / "calibration.json"
def thumb_of(work):
    h, w = work.shape[:2]
    return cv2.resize(work, (max(1, w // CFG.THUMB_REL), max(1, h // CFG.THUMB_REL)), interpolation=cv2.INTER_AREA)

def macenko_global(thumbs_I0, per_img=20000, rng=np.random.default_rng(0)):
    pool = []
    for th, I0 in thumbs_I0:
        od = od_np(th, I0).reshape(-1, 3)
        od = od[(od.max(1) > 0.15) & (od.sum(1) < CFG.OD_DARK)]
        if len(od) > per_img: od = od[rng.choice(len(od), per_img, replace=False)]
        pool.append(od)
    od = np.concatenate(pool)
    _, V = np.linalg.eigh(np.cov(od.T)); V = V[:, [2, 1]]
    V = V * np.sign(V.sum(0) + 1e-12)
    phi = np.arctan2(od @ V[:, 1], od @ V[:, 0])
    a1, a2 = np.percentile(phi, 1), np.percentile(phi, 99)
    v1 = _unit(V @ [np.cos(a1), np.sin(a1)]); v2 = _unit(V @ [np.cos(a2), np.sin(a2)])
    v1, v2 = np.abs(v1), np.abs(v2)
    h, d = (v1, v2) if v1 @ RUIFROK_H > v2 @ RUIFROK_H else (v2, v1)
    info = {"cos_H_vs_ruifrok": float(h @ RUIFROK_H), "cos_DAB_vs_ruifrok": float(d @ RUIFROK_D)}
    if info["cos_H_vs_ruifrok"] < 0.90: h = RUIFROK_H; info["H_fallback"] = True
    if info["cos_DAB_vs_ruifrok"] < 0.85: d = RUIFROK_D; info["DAB_fallback"] = True
    return complete_matrix(h, d), info

def gmm_boundaries(logv, k=3, seed=0):
    g = GaussianMixture(k, random_state=seed, n_init=3).fit(logv.reshape(-1, 1))
    order = np.argsort(g.means_.ravel())
    grid = np.linspace(logv.min(), logv.max(), 4000).reshape(-1, 1)
    post = g.predict_proba(grid)[:, order]
    lab = post.argmax(1)
    bnds = [grid[np.argmax(lab >= j), 0] for j in range(1, k)]
    return bnds, g

if CALIB_PATH.exists():
    CALIB = json.load(open(CALIB_PATH)); print("loaded calibration", CALIB_PATH)
else:
    cal_df = pd.concat([TRAIN[TRAIN.ihc_score == s].sample(min(CFG.N_CALIB_PER_SCORE, (TRAIN.ihc_score == s).sum()),
                                                              random_state=CFG.SEED) for s in range(4)]).reset_index(drop=True)
    thumbs = []
    for p in tqdm(cal_df.path, desc="calib thumbs"):
        th = read_rgb(p, CFG.WORK_DS * CFG.THUMB_REL); thumbs.append((th, estimate_I0(th)))
    M, minfo = macenko_global(thumbs)
    eng0 = StainEngine(M, None)
    rng = np.random.default_rng(CFG.SEED)
    Hs, samples = [], []
    for (th, I0), (_, r) in tqdm(list(zip(thumbs, cal_df.iterrows())), desc="calib work-level"):
        work = read_rgb(r.path, CFG.WORK_DS)
        tis_t, rim_t, _, _ = tissue_from_thumb(th, I0)
        mp = eng0.maps(work, I0, tis_t, rim_t)
        tis = mp["tissue"][0, 0]; interior = tis & ~mp["rim"][0, 0]; shp = mp["mem_shape"][0, 0] & interior
        hv = mp["H"][0, 0][tis].cpu().numpy(); Hs.append(hv[rng.choice(len(hv), min(20000, len(hv)), replace=False)] if len(hv) else hv)
        dv = mp["DAB"][0, 0][shp].cpu().numpy(); hv2 = mp["Hs"][0, 0][shp].cpu().numpy(); dsv = mp["Ds"][0, 0][shp].cpu().numpy()
        sel = rng.choice(len(dv), min(20000, len(dv)), replace=False) if len(dv) else np.array([], int)
        samples.append((int(r.ihc_score), r.image_id, dv[sel], hv2[sel], dsv[sel]))
        del mp, work
    from skimage.filters import threshold_otsu
    t_h = float(threshold_otsu(np.concatenate(Hs)))
    by_score = {s: [] for s in range(4)}
    for s, iid, dv, hv, dsv in samples:
        by_score[s].append(dv[~((hv >= t_h) & (hv >= dsv))])     # drop haematoxylin-dominant (nuclear) pixels
    neg = np.concatenate(by_score[0]) if by_score[0] else np.array([0.05])
    t_weak = float(np.quantile(neg, CFG.Q_NEG)); t_bg = float(np.quantile(neg, CFG.Q_BG))
    above = np.concatenate([v[v > t_weak] for s in (1, 2, 3) for v in by_score[s]] or [np.array([])])
    try:
        assert len(above) > 500
        (b1, b2), _ = gmm_boundaries(np.log(above))
        t_mod, t_strong = float(np.exp(b1)), float(np.exp(b2))
        assert t_weak < t_mod < t_strong
        how = "gmm3"
    except Exception:
        s3 = np.concatenate([v[v > t_weak] for v in by_score[3]] or [np.array([t_weak * 1.5, t_weak * 3])])
        t_mod, t_strong = float(np.quantile(s3, 1 / 3)), float(np.quantile(s3, 2 / 3)); how = "quantile_fallback"
        t_mod = max(t_mod, t_weak * 1.05); t_strong = max(t_strong, t_mod * 1.05)
    CALIB = dict(M=M.tolist(), macenko=minfo, t_h=t_h, t_bg=t_bg, t_weak=t_weak, t_mod=t_mod, t_strong=t_strong,
                 band_method=how, n_calib_images=len(cal_df), q_neg=CFG.Q_NEG)
    save_json(CALIB, CALIB_PATH)
    # figure: membrane-shaped DAB OD distribution by IHC score with operating points
    fig, ax = plt.subplots(figsize=(6.2, 3.4))
    for s in range(4):
        v = np.concatenate(by_score[s]) if by_score[s] else np.array([])
        v = v[v > 1e-3]
        if len(v): ax.hist(np.log10(v), bins=120, density=True, histtype="step", lw=1.6, color=GROUP_COL[str(s) if s != 2 else "2+"], label=f"IHC {s} (n={len(by_score[s])} cores)")
    for k, ls in [("t_bg", ":"), ("t_weak", "--"), ("t_mod", "-."), ("t_strong", "-")]:
        ax.axvline(np.log10(CALIB[k]), color="#52514e", ls=ls, lw=1); ax.text(np.log10(CALIB[k]), ax.get_ylim()[1] * 0.92, k, rotation=90, fontsize=7, ha="right", color="#52514e")
    ax.set_xlabel("log10 DAB optical density (membrane-shaped, non-nuclear pixels)"); ax.set_ylabel("density"); ax.legend(fontsize=7)
    ax.set_title("Label-anchored DAB operating points")
    savefig(fig, "fig02_dab_calibration")
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in CALIB.items() if k != "M"})
ENGINE = StainEngine(np.array(CALIB["M"]), CALIB)

## 4. Pass 1: one decode per core yields tissue, QC, stain features and 20× tile embeddings
For every core (train and test) this pass stores the following, and can resume from a partial run:
* the tissue mask, the rim band and dark artefacts at about 1.8 µm/px
* 224-px tiles (about 103 µm) at about 0.46 µm/px, kept when ≥50% of the tile is tissue
* one encoder embedding per tile
* 14 interpretable stain and morphology features per tile
* QC metrics
* low-resolution DAB and membrane maps for figures

In [ ]:
HAND_NAMES = ["tissue_frac", "dab_mean", "dab_std", "dab_ge_weak", "dab_ge_mod", "dab_ge_strong", "mem_weak", "mem_mod",
              "mem_strong", "nonmem_dab", "nuclear_dab", "nuc_frac", "focus", "rim_frac"]
IMNET_MEAN, IMNET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

class TileEncoder(nn.Module):
    def __init__(self, spec):
        super().__init__()
        self.kind, name = spec.split(":", 1)
        self.pretrained = True
        if self.kind == "timm":
            try:
                self.m = timm.create_model(name, pretrained=True, features_only=True, out_indices=(CFG.ENCODER_OUT_INDEX,))
            except Exception as e:
                print(f"!! WARNING: could not download pretrained '{name}' ({type(e).__name__}). Using RANDOM weights. "
                      "Enable Internet in Kaggle settings or attach the weights; results will be meaningless otherwise.")
                self.m = timm.create_model(name, pretrained=False, features_only=True, out_indices=(CFG.ENCODER_OUT_INDEX,))
                self.pretrained = False
            self.dim = self.m.feature_info.channels()[-1]
            pc = getattr(self.m, "pretrained_cfg", None) or {}
            mean, std = pc.get("mean", IMNET_MEAN), pc.get("std", IMNET_STD)
        else:
            from transformers import AutoModel
            self.m = AutoModel.from_pretrained(name); self.dim = self.m.config.hidden_size
            mean, std = IMNET_MEAN, IMNET_STD
        self.register_buffer("mean", torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(std).view(1, 3, 1, 1))

    def forward(self, x):
        x = (x - self.mean) / self.std
        if self.kind == "timm":
            return self.m(x)[-1].mean((2, 3))
        return self.m(pixel_values=x).last_hidden_state[:, 0]

ALL = pd.concat([TRAIN, TEST], ignore_index=True)
FEAT_DIR, MAP_DIR, THUMB_DIR = CACHE / "feats", CACHE / "maps", CACHE / "thumbs"
LAP = torch.tensor([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=torch.float32, device=DEVICE).view(1, 1, 3, 3)

def dhash(gray, n=16):
    g = cv2.resize(gray, (n + 1, n), interpolation=cv2.INTER_AREA)
    bits = (g[:, 1:] > g[:, :-1]).ravel()
    return "".join("%x" % int("".join(map(str, bits[i:i + 4].astype(int))), 2) for i in range(0, len(bits), 4))

class DecodeDS(torch.utils.data.Dataset):
    def __init__(self, df): self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        try: return i, read_rgb(self.df.path[i], CFG.WORK_DS)
        except Exception: return i, None

@torch.no_grad()
def featurize(row, work, enc, enc2=None):
    iid = row.image_id; T = CFG.TILE
    H, W = work.shape[:2]
    thumb = thumb_of(work); I0 = estimate_I0(thumb)
    tis_t, rim_t, dark_t, env_t = tissue_from_thumb(thumb, I0)
    mp = ENGINE.maps(work, I0, tis_t, rim_t)
    ny, nx = H // T, W // T
    cut = lambda m: m.float()[..., :ny * T, :nx * T]
    pool = lambda m: F.avg_pool2d(cut(m), T, T)[0, 0]
    tis = mp["tissue"]; tf = pool(tis); tisN = tf.clamp(min=1e-6)
    frac = lambda m: pool(m & tis) / tisN
    dab = mp["DAB"] * tis
    dmean = pool(dab) / tisN; dsq = pool(dab * dab) / tisN
    lap = F.conv2d(F.pad(mp["gray"], (1, 1, 1, 1), mode="reflect"), LAP)
    focus = pool(lap * lap) - pool(lap) ** 2
    feats = torch.stack([tf, dmean, torch.sqrt((dsq - dmean ** 2).clamp(min=0)),
                         frac(mp["DAB"] >= CALIB["t_weak"]), frac(mp["DAB"] >= CALIB["t_mod"]), frac(mp["DAB"] >= CALIB["t_strong"]),
                         frac(mp["mem_weak"]), frac(mp["mem_mod"]), frac(mp["mem_strong"]), frac(mp["nonmem"]),
                         frac(mp["nuc"] & (mp["DAB"] >= CALIB["t_weak"])), frac(mp["nuc"]), focus, pool(mp["rim"])], -1)
    keep = torch.nonzero(tf >= CFG.MIN_TILE_TISSUE).cpu().numpy()
    if len(keep) > CFG.MAX_TILES_PER_IMAGE:
        rs = np.random.default_rng(int(hashlib.md5(iid.encode()).hexdigest()[:8], 16))
        keep = keep[np.sort(rs.choice(len(keep), CFG.MAX_TILES_PER_IMAGE, replace=False))]
    hand = feats[keep[:, 0], keep[:, 1]].cpu().numpy().astype(np.float32) if len(keep) else np.zeros((0, len(HAND_NAMES)), np.float32)
    coords = (keep * T).astype(np.int32)
    xg = torch.from_numpy(work).to(DEVICE).permute(2, 0, 1)
    embs, embs2 = [], []
    for b in range(0, len(coords), CFG.ENC_BATCH):
        cb = coords[b:b + CFG.ENC_BATCH]
        batch = torch.stack([xg[:, y:y + T, x:x + T] for y, x in cb]).float() / 255.0
        with autocast():
            embs.append(gpu_safe(enc, batch).float().cpu())
            if enc2 is not None: embs2.append(gpu_safe(enc2, batch).float().cpu())
    emb = torch.cat(embs).numpy().astype(np.float16) if embs else np.zeros((0, enc.dim), np.float16)
    extra = {}
    if enc2 is not None:
        extra["emb2"] = torch.cat(embs2).numpy().astype(np.float16) if embs2 else np.zeros((0, enc2.dim), np.float16)
    # ---------- QC ----------
    tissue_px = max(int(tis_t.sum()), 1)
    full_lap = lap[0, 0][tis[0, 0]]
    dabpos = (mp["DAB"] >= CALIB["t_weak"]) & tis
    if int(dabpos.sum()) > 500:
        x = torch.from_numpy(work).to(DEVICE).float(); I0t = torch.tensor(I0, device=DEVICE, dtype=torch.float32)
        odv = torch.clamp(-torch.log((x + 1) / (I0t + 1)), min=0)[dabpos[0, 0]].mean(0).cpu().numpy()
        drift = float(_unit(odv) @ np.array(CALIB["M"])[1])
    else:
        drift = np.nan
    interior = tis & ~mp["rim"]
    mi = float((mp["mem_weak"] & interior).sum() / interior.sum().clamp(min=1))
    mr = float((mp["mem_weak"] & mp["rim"]).sum() / mp["rim"].sum().clamp(min=1))
    qc = dict(image_id=iid, n_tiles=int(len(coords)), tissue_mm2=float(tis_t.sum() * THUMB_MPP ** 2 / 1e6),
              dark_frac=float((dark_t & tis_t).sum() / tissue_px), focus_med=float(np.median(hand[:, 12])) if len(hand) else np.nan,
              global_lapvar=float(full_lap.var()) if full_lap.numel() > 10 else np.nan, I0=float(np.mean(I0)),
              dab_hue_cos=drift, mem_weak_interior=mi, mem_weak_rim=mr,
              dab_mean_core=float((mp["DAB"] * tis).sum() / tis.sum().clamp(min=1)), dhash=dhash(cv2.cvtColor(thumb, cv2.COLOR_RGB2GRAY)),
              work_h=H, work_w=W)

    R = CFG.THUMB_REL
    lr = lambda m: F.avg_pool2d(m.float()[..., :H // R * R, :W // R * R], R, R)[0, 0].cpu().numpy().astype(np.float16)
    atomic_npz(MAP_DIR / f"{iid}.npz", compressed=True, dab=lr(mp["DAB"] * tis), mem_weak=lr(mp["mem_weak"]), mem_mod=lr(mp["mem_mod"]),
               tissue=tis_t, rim=rim_t, I0=I0)
    ok, jpg = cv2.imencode(".jpg", cv2.cvtColor(thumb, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 90])
    atomic_write(THUMB_DIR / f"{iid}.jpg", lambda f: f.write(jpg.tobytes()))
    # the features file is written LAST: its presence marks the core as complete (maps and thumbnail already on disk)
    atomic_npz(FEAT_DIR / f"{iid}.npz", coords=coords, emb=emb, hand=hand, qc=json.dumps(qc), **extra)
    return qc

def _core_done(iid):
    """A core is complete only if all three files exist and the feature file opens (guards against truncated files)."""
    f = FEAT_DIR / f"{iid}.npz"
    if not (f.exists() and (MAP_DIR / f"{iid}.npz").exists() and (THUMB_DIR / f"{iid}.jpg").exists()): return False
    try:
        with np.load(f) as z: return {"coords", "emb", "hand", "qc"} <= set(z.files)
    except Exception:
        f.unlink(missing_ok=True); return False
todo = ALL[[not _core_done(i) for i in ALL.image_id]].reset_index(drop=True)
print(f"pass 1: {len(todo)} / {len(ALL)} cores to featurize")
ENC_INFO_PATH = CACHE / "encoder_info.json"
if len(todo):
    ENC = TileEncoder(CFG.ENCODER).to(DEVICE).eval()
    ENC2 = None
    if CFG.ENCODER2:
        try:
            ENC2 = TileEncoder(CFG.ENCODER2).to(DEVICE).eval()
            print(f"encoder ablation: {CFG.ENCODER2} ({ENC2.dim}-d) extracted in the same pass")
        except Exception as e:
            print(f"!! {CFG.ENCODER2} unavailable ({type(e).__name__}: {str(e)[:120]}); encoder ablation skipped")
    if not ENC_INFO_PATH.exists():
        save_json(dict(encoder=CFG.ENCODER, pretrained=ENC.pretrained, dim=ENC.dim,
                       encoder2=CFG.ENCODER2 if ENC2 is not None else None, dim2=ENC2.dim if ENC2 is not None else None), ENC_INFO_PATH)
    dl = torch.utils.data.DataLoader(DecodeDS(todo), batch_size=1, num_workers=CFG.NUM_WORKERS,
                                     collate_fn=lambda b: b[0], prefetch_factor=2 if CFG.NUM_WORKERS else None)
    t0 = time.time()
    ENC2_USE = ENC2 if (ENC2 is not None and json.load(open(ENC_INFO_PATH)).get("encoder2")) else None
    for i, work in tqdm(dl, total=len(todo), desc="featurize"):
        if work is None:
            print("unreadable:", todo.path[i]); continue
        featurize(todo.iloc[i], work, ENC, ENC2_USE)
    print(f"pass 1 done in {(time.time() - t0) / 60:.1f} min")
    del ENC, ENC2; gc.collect(); torch.cuda.empty_cache()
ENC_INFO = json.load(open(ENC_INFO_PATH)) if ENC_INFO_PATH.exists() else dict(pretrained=True, encoder2=None)
ENCODER_PRETRAINED = ENC_INFO["pretrained"]
CFG.ENCODER2 = ENC_INFO.get("encoder2")     # the encoder ablation runs only if every core carries the second embedding
section_time("pass 1 featurisation")

In [ ]:
# ---------------- membrane-detector QA figure (for pathologist review of the morphological prior) ----------------
def qa_membrane_figure(n=3):
    picks = []
    for s in [3, 2, 0]:
        sub = TRAIN[TRAIN.ihc_score == s]
        if len(sub): picks.append(sub.sample(1, random_state=CFG.SEED).iloc[0])
    fig, axes = plt.subplots(len(picks), 4, figsize=(11, 2.9 * len(picks)))
    axes = np.atleast_2d(axes)
    for ax_row, r in zip(axes, picks):
        work = read_rgb(r.path, CFG.WORK_DS); th = thumb_of(work); I0 = estimate_I0(th)
        tis_t, rim_t, _, _ = tissue_from_thumb(th, I0)
        mp = ENGINE.maps(work, I0, tis_t, rim_t)
        d = F.avg_pool2d(mp["DAB"] * mp["tissue"], CFG.SEG_CROP, CFG.SEG_CROP // 2)[0, 0]
        yy, xx = np.unravel_index(int(d.argmax()), d.shape); S = CFG.SEG_CROP
        y0, x0 = min(yy * S // 2, work.shape[0] - S), min(xx * S // 2, work.shape[1] - S)
        sl = (slice(y0, y0 + S), slice(x0, x0 + S))
        g = lambda k: mp[k][0, 0].cpu().numpy()[sl]
        ax_row[0].imshow(work[sl]); ax_row[0].set_title(f"{r.image_id[:22]}  IHC {r.ihc_score}")
        ax_row[1].imshow(g("DAB"), cmap="magma", vmin=0, vmax=CALIB["t_strong"] * 1.5); ax_row[1].set_title("DAB OD (absolute)")
        ov = work[sl].copy()
        ov[g("mem_weak")] = (42, 120, 214); ov[g("mem_mod")] = (235, 104, 52); ov[g("mem_strong")] = (227, 73, 72)
        ax_row[2].imshow(ov); ax_row[2].set_title("membrane DAB\n(blue weak / orange mod / red strong)", fontsize=8)
        ov2 = work[sl].copy(); ov2[g("nonmem")] = (237, 161, 0); ov2[g("nuc")] = (98, 80, 214)
        ax_row[3].imshow(ov2); ax_row[3].set_title("non-membranous DAB (amber)\n/ H-dominant nuclei (violet)", fontsize=8)
        for a in ax_row: a.axis("off")
    fig.suptitle("Morphological membrane prior: an unverified hypothesis for expert review", y=1.0)
    savefig(fig, "fig03_membrane_prior_QA")
if not (FIG / "fig03_membrane_prior_QA.png").exists(): qa_membrane_figure()

## 5. Quality control: flag rather than drop, and test whether QC is itself confounded by class
The earlier pipeline dropped 49% of the training cores with a global Laplacian-variance threshold. That statistic mostly measures the tissue
amount and the DAB contrast, so the threshold dropped 52% of IHC-0 cores but only 21% of IHC-3 cores.

This notebook handles QC differently:
* Focus is measured **per tissue tile** and expressed as a robust z-score (median and MAD).
* Flagged cores are **kept**. A sensitivity analysis later re-scores the test set without them.
* The only hard exclusion is a core with fewer than `MIN_TILES_KEEP` tissue tiles.

In [ ]:
QC = pd.DataFrame([json.loads(np.load(FEAT_DIR / f"{i}.npz")["qc"].item()) for i in ALL.image_id if (FEAT_DIR / f"{i}.npz").exists()])
ALLQ = ALL.merge(QC, on="image_id", how="left")
def robust_z(x, ref):
    med = np.nanmedian(ref); mad = 1.4826 * np.nanmedian(np.abs(ref - med)) + 1e-9
    return (x - med) / mad
tr_mask = ALLQ.split == "train"
lf = np.log(ALLQ.focus_med.clip(lower=1e-6))
ALLQ["focus_z"] = robust_z(lf, lf[tr_mask])
ALLQ["stain_z"] = robust_z(ALLQ.dab_hue_cos, ALLQ.dab_hue_cos[tr_mask])
# Train: cores with too little tissue are excluded from model fitting. Test: NO core is dropped (intention-to-diagnose);
# a test core with < MIN_TILES_KEEP tiles is kept, flagged low-tissue, and routed to "refer". Only empty cores (0 tiles) are dropped.
ALLQ["exclude"] = np.where(ALLQ.split == "train", ALLQ.n_tiles.fillna(0) < CFG.MIN_TILES_KEEP, ALLQ.n_tiles.fillna(0) < 1)
ALLQ["few_tiles"] = ALLQ.n_tiles.fillna(0) < CFG.MIN_TILES_KEEP
ALLQ["flag_blur"] = ALLQ.focus_z < -3.5
ALLQ["flag_dark"] = ALLQ.dark_frac > 0.15
ALLQ["flag_stain"] = ALLQ.stain_z < -4
ALLQ["flag_lowtissue"] = (ALLQ.tissue_mm2 < 0.10) | ALLQ.few_tiles
ALLQ["qc_flag"] = ALLQ[["flag_blur", "flag_dark", "flag_stain", "flag_lowtissue"]].any(axis=1)
qc_tab = ALLQ.groupby(["split", "group"])[["exclude", "flag_blur", "flag_dark", "flag_stain", "flag_lowtissue", "qc_flag"]].mean().round(3)
qc_tab.to_csv(TAB / "qc_flags_by_group.csv"); print(qc_tab)
trq = ALLQ[tr_mask & ~ALLQ.exclude]
ct = pd.crosstab(trq.group, trq.qc_flag)
chi_p = stats.chi2_contingency(ct)[1] if ct.shape == (ct.shape[0], 2) and ct.values.min() >= 0 and ct.shape[1] == 2 else np.nan
rho_old = stats.spearmanr(trq.ihc_score, trq.global_lapvar, nan_policy="omit")
rho_new = stats.spearmanr(trq.ihc_score, trq.focus_z, nan_policy="omit")
old_thr = np.nanmedian(trq.global_lapvar)
old_fail = trq.assign(f=trq.global_lapvar < old_thr).groupby("group").f.mean().reindex(GROUP_ORDER)
qc_summary = dict(chi2_p_new_flags_vs_group=chi_p, spearman_old_globalLapVar_vs_score=(rho_old.correlation, rho_old.pvalue),
                  spearman_new_tileFocus_vs_score=(rho_new.correlation, rho_new.pvalue),
                  old_style_fail_rate_by_group_at_median_threshold=old_fail.round(3).to_dict(),
                  n_excluded=int(ALLQ.exclude.sum()), n_flagged=int(ALLQ.qc_flag.sum()))
# edge artefact: membrane-shaped DAB density at the core rim vs interior (paired, per core)
ea = ALLQ[~ALLQ.exclude & ALLQ.mem_weak_rim.notna()]
w = stats.wilcoxon(ea.mem_weak_rim, ea.mem_weak_interior) if len(ea) > 10 else None
qc_summary["edge_artefact_rim_vs_interior_median"] = (float(ea.mem_weak_rim.median()), float(ea.mem_weak_interior.median()))
qc_summary["edge_artefact_wilcoxon_p"] = float(w.pvalue) if w is not None else np.nan
# temporal drift train -> test on IHC-0 cores (background staining & illumination)
drift = {}
for c in ["I0", "dab_hue_cos", "dab_mean_core", "focus_z"]:
    a = ALLQ[(ALLQ.split == "train") & (ALLQ.ihc_score == 0)][c].dropna(); b = ALLQ[(ALLQ.split == "test") & (ALLQ.ihc_score == 0)][c].dropna()
    if len(a) > 3 and len(b) > 3:
        drift[c] = dict(train_median=float(a.median()), test_median=float(b.median()), mannwhitney_p=float(stats.mannwhitneyu(a, b).pvalue))
qc_summary["train_test_drift_IHC0"] = drift
save_json(qc_summary, TAB / "qc_summary.json"); print(json.dumps(qc_summary, indent=1, default=str))

fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.3))
for ax, col, title in [(axes[0], "global_lapvar", "Old criterion: global Laplacian variance"),
                       (axes[1], "focus_z", "New: tile-level focus (robust z)")]:
    data = [trq[trq.group == g][col].dropna().values for g in GROUP_ORDER]
    bp = ax.boxplot(data, showfliers=False, patch_artist=True, widths=0.55); ax.set_xticks(range(1, len(GROUP_ORDER) + 1)); ax.set_xticklabels(GROUP_ORDER)
    for patch, g in zip(bp["boxes"], GROUP_ORDER): patch.set_facecolor(GROUP_COL[g]); patch.set_alpha(0.55)
    ax.set_title(title); ax.set_xlabel("IHC group (2-/2+ = FISH)")
axes[0].axhline(old_thr, color="#52514e", ls="--", lw=1)
axes[1].axhline(-3.5, color="#52514e", ls="--", lw=1)
axes[2].scatter(ea.mem_weak_interior, ea.mem_weak_rim, s=6, alpha=0.5, color=PAL[0])
mx = float(np.nanmax([ea.mem_weak_interior.max(), ea.mem_weak_rim.max(), 1e-3]))
axes[2].plot([0, mx], [0, mx], color="#52514e", lw=1, ls="--"); axes[2].set_xlabel("membranous DAB fraction: interior")
axes[2].set_ylabel("rim"); axes[2].set_title("Core-edge staining (edge artefact check)")
savefig(fig, "fig04_qc")

In [ ]:
# ---------------- near-duplicate detection (thumbnail dHash) ----------------
hq = ALLQ[ALLQ.dhash.notna()].reset_index(drop=True)
bits = np.array([[int(c, 16) >> s & 1 for c in h for s in (3, 2, 1, 0)] for h in hq.dhash], np.float32)
ham = bits.sum(1)[:, None] + bits.sum(1)[None, :] - 2 * bits @ bits.T
iu = np.triu_indices(len(hq), 1)
close = [(hq.image_id[i], hq.image_id[j], int(ham[i, j])) for i, j in zip(*iu) if ham[i, j] <= 12]
cross = [c for c in close if c[0].split("__")[0] != c[1].split("__")[0]]
print(f"near-duplicate thumbnail pairs: {len(close)} (across train/test: {len(cross)})")
pd.DataFrame(close, columns=["a", "b", "hamming"]).to_csv(TAB / "near_duplicates.csv", index=False)

## 6. Load the bags, normalise the features, and build proxy patient groups
**Leakage.** Up to about 1,320 of the training cores come from 165 tumours, with 8 cores each, and there are no patient IDs. Random cross-validation
therefore mixes cores of the same tumour across folds. As a mitigation, this section builds **proxy groups**: average-linkage clusters of
core-level morphology and stain descriptors, sized to the expected number of patients. `StratifiedGroupKFold` then keeps each proxy group inside
one fold.

This is imperfect. Section 8 measures the effect by comparing grouped and random CV. The **official temporally independent test set is the
primary evaluation.**

In [ ]:
KEEP = ALLQ[~ALLQ.exclude & ALLQ.n_tiles.notna()].reset_index(drop=True)
TR = KEEP[KEEP.split == "train"].reset_index(drop=True)
TE = KEEP[KEEP.split == "test"].reset_index(drop=True)
print(f"usable cores: train {len(TR)} | test {len(TE)}  (excluded: {int(ALLQ.exclude.sum())})")

def transform_hand(h):
    h = h.copy().astype(np.float32)
    fi = HAND_NAMES.index("focus")
    h[:, fi] = np.log1p(np.clip(h[:, fi], 0, None))
    o = [i for i in range(h.shape[1]) if i != fi]
    h[:, o] = np.sqrt(np.clip(h[:, o], 0, None))
    return h

BAGS = {}
for iid in tqdm(KEEP.image_id, desc="load bags"):
    z = np.load(FEAT_DIR / f"{iid}.npz")
    BAGS[iid] = dict(emb=z["emb"], hand_raw=z["hand"], hand=transform_hand(z["hand"]), coords=z["coords"])
    if "emb2" in z.files: BAGS[iid]["emb2"] = z["emb2"]
HAS_EMB2 = all("emb2" in b for b in BAGS.values())
_ht = np.concatenate([BAGS[i]["hand"] for i in TR.image_id])
HAND_MU, HAND_SD = _ht.mean(0), _ht.std(0) + 1e-6
for b in BAGS.values():
    b["hand_z"] = ((b["hand"] - HAND_MU) / HAND_SD).astype(np.float32)
EMB_DIM = next(iter(BAGS.values()))["emb"].shape[1]

# ---------- proxy patient groups ----------
desc_e = np.stack([BAGS[i]["emb"].astype(np.float32).mean(0) for i in TR.image_id])
desc_e /= np.linalg.norm(desc_e, axis=1, keepdims=True) + 1e-9
desc_h = np.stack([BAGS[i]["hand_z"].mean(0) for i in TR.image_id])
desc_h = (desc_h - desc_h.mean(0)) / (desc_h.std(0) + 1e-6)
desc = np.concatenate([desc_e, 0.5 * desc_h / np.sqrt(desc_h.shape[1])], 1)
n_groups = int(np.clip(round(len(TR) * CFG.EXPECTED_PATIENTS / CFG.EXPECTED_CORES), 2, len(TR) - 1))
try:
    agg = AgglomerativeClustering(n_clusters=n_groups, metric="cosine", linkage="average")
except TypeError:
    agg = AgglomerativeClustering(n_clusters=n_groups, affinity="cosine", linkage="average")
TR["proxy_group"] = agg.fit_predict(desc)
sizes = TR.proxy_group.value_counts()
def within_agree(g, y):
    d = pd.DataFrame({"g": g, "y": y}); tot = 0; agree = 0
    for _, s in d.groupby("g"):
        n = len(s)
        if n > 1:
            vc = s.y.value_counts(); agree += (vc * (vc - 1)).sum(); tot += n * (n - 1)
    return agree / max(tot, 1)
obs = within_agree(TR.proxy_group, TR.ihc_score)
perm = np.mean([within_agree(np.random.default_rng(k).permutation(TR.proxy_group.values), TR.ihc_score) for k in range(50)])
group_info = dict(n_groups=n_groups, size_median=float(sizes.median()), size_max=int(sizes.max()),
                  singleton_frac=float((sizes == 1).mean()), within_group_score_agreement=obs, permutation_expectation=perm)
save_json(group_info, TAB / "proxy_groups.json"); print(group_info)

ylab = TR.group.map({g: i for i, g in enumerate(GROUP_ORDER)}).values
FOLDS_PATH = CACHE / "folds.csv"
_prev_folds = pd.read_csv(FOLDS_PATH) if FOLDS_PATH.exists() else None
if _prev_folds is not None and set(_prev_folds.image_id) == set(TR.image_id):
    # restart: reuse the exact saved split so every cached model/result stays consistent with it
    TR = TR.drop(columns=[c for c in ("proxy_group", "fold", "fold_random") if c in TR]).merge(_prev_folds, on="image_id", how="left")
    print("folds: loaded saved split", FOLDS_PATH)
else:
    sgkf = StratifiedGroupKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
    TR["fold"] = -1
    for k, (_, va) in enumerate(sgkf.split(TR, ylab, TR.proxy_group)):
        TR.loc[va, "fold"] = k
    skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
    TR["fold_random"] = -1
    for k, (_, va) in enumerate(skf.split(TR, ylab)):
        TR.loc[va, "fold_random"] = k
    atomic_csv(TR[["image_id", "proxy_group", "fold", "fold_random"]], FOLDS_PATH, index=False)
print(pd.crosstab(TR.fold, TR.group).reindex(columns=GROUP_ORDER, fill_value=0))

## 7. Unsupervised tissue-type clusters, and a 10-minute expert naming task
HER2 labels contain no tumour/stroma information, so tumour localisation can't be *learned* from them. Instead, tile embeddings are clustered and
a gallery is exported. A pathologist fills in `cluster_naming_template.csv` (tumour epithelium / stroma / necrosis / mucin / lymphoid / artefact)
and points `CFG.CLUSTER_NAMING_CSV` to it. The tumour-candidate maps and the tumour-restricted cell features then switch on automatically.
Until that happens, every result is computed on **all tissue** and labelled as such.

In [ ]:
# Single-threaded BLAS/OpenMP here: after PyTorch DataLoader workers have run, scikit-learn's PCA/k-means can
# dead-lock on Kaggle when two OpenMP runtimes are loaded. Single-threaded, this whole step takes seconds.
from threadpoolctl import threadpool_limits
t0 = time.time(); rng = np.random.default_rng(CFG.SEED)
_ids = list(TR.image_id); _n = np.array([len(BAGS[i]["emb"]) for i in _ids]); _tot = int(_n.sum())
_pick = np.sort(rng.choice(_tot, min(30000, _tot), replace=False)); _off = np.concatenate([[0], np.cumsum(_n)])
sub = np.stack([BAGS[_ids[b]]["emb"][j - _off[b]] for j, b in zip(_pick, np.searchsorted(_off, _pick, side="right") - 1)]).astype(np.float32)
with threadpool_limits(limits=1):
    pca = PCA(n_components=min(50, sub.shape[1], len(sub) - 1), svd_solver="randomized", random_state=CFG.SEED).fit(sub)
    print(f"[tissue clusters] PCA on {len(sub)} tiles: {time.time() - t0:.0f}s", flush=True)
    km = MiniBatchKMeans(n_clusters=CFG.N_TISSUE_CLUSTERS, random_state=CFG.SEED, batch_size=4096, n_init=3).fit(pca.transform(sub))
    print(f"[tissue clusters] k-means: {time.time() - t0:.0f}s", flush=True)
    for iid, b in BAGS.items():
        b["tc"] = km.predict(pca.transform(b["emb"].astype(np.float32))) if len(b["emb"]) else np.zeros(0, int)
print(f"[tissue clusters] all {len(BAGS)} cores assigned: {time.time() - t0:.0f}s", flush=True)
comp = pd.DataFrame([np.bincount(BAGS[i]["tc"], minlength=CFG.N_TISSUE_CLUSTERS) / max(len(BAGS[i]["tc"]), 1) for i in TR.image_id])
comp["group"] = TR.group.values
comp_g = comp.groupby("group").mean().reindex(GROUP_ORDER)
mem_by_c = np.zeros(CFG.N_TISSUE_CLUSTERS); n_by_c = np.zeros(CFG.N_TISSUE_CLUSTERS)
for i in TR.image_id:
    b = BAGS[i]; np.add.at(mem_by_c, b["tc"], b["hand_raw"][:, HAND_NAMES.index("mem_weak")]); np.add.at(n_by_c, b["tc"], 1)
tmpl = pd.DataFrame({"cluster": range(CFG.N_TISSUE_CLUSTERS), "n_tiles_train": n_by_c.astype(int),
                     "mean_membranous_DAB_frac": (mem_by_c / np.maximum(n_by_c, 1)).round(4),
                     **{f"share_in_IHC{g}": comp_g.loc[g].values.round(3) if g in comp_g.index and comp_g.loc[g].notna().all() else np.nan for g in GROUP_ORDER},
                     "tissue_type": "", "notes": ""})
tmpl.to_csv(TAB / "cluster_naming_template.csv", index=False)

def gallery(n_per=8):
    picks = defaultdict(list)
    ids = list(TR.image_id); rng.shuffle(ids)
    for iid in ids:
        b = BAGS[iid]
        for c in range(CFG.N_TISSUE_CLUSTERS):
            if len(picks[c]) < n_per:
                w = np.where(b["tc"] == c)[0]
                if len(w): picks[c].append((iid, b["coords"][rng.choice(w)]))
        if all(len(picks[c]) >= n_per for c in range(CFG.N_TISSUE_CLUSTERS)): break
    need = defaultdict(list)
    for c, lst in picks.items():
        for j, (iid, yx) in enumerate(lst): need[iid].append((c, j, yx))
    T = CFG.TILE; canvas = np.full((CFG.N_TISSUE_CLUSTERS * T, n_per * T, 3), 255, np.uint8)
    path_of = dict(zip(KEEP.image_id, KEEP.path))
    for iid, lst in tqdm(need.items(), desc="cluster gallery"):
        work = read_rgb(path_of[iid], CFG.WORK_DS)
        for c, j, (y, x) in lst: canvas[c * T:(c + 1) * T, j * T:(j + 1) * T] = work[y:y + T, x:x + T]
    fig, ax = plt.subplots(figsize=(n_per * 1.1, CFG.N_TISSUE_CLUSTERS * 1.1))
    ax.imshow(canvas); ax.set_yticks([c * T + T / 2 for c in range(CFG.N_TISSUE_CLUSTERS)])
    ax.set_yticklabels([f"cluster {c}" for c in range(CFG.N_TISSUE_CLUSTERS)]); ax.set_xticks([]); ax.grid(False)
    ax.set_title("Tile clusters for expert naming (fill tables/cluster_naming_template.csv)")
    savefig(fig, "fig05_cluster_gallery")
if not (FIG / "fig05_cluster_gallery.png").exists(): gallery()

TUMOUR_CLUSTERS = None
if CFG.CLUSTER_NAMING_CSV and Path(CFG.CLUSTER_NAMING_CSV).exists():
    nm = pd.read_csv(CFG.CLUSTER_NAMING_CSV)
    TUMOUR_CLUSTERS = set(nm[nm.tissue_type.astype(str).str.lower().str.contains("tum")].cluster.astype(int))
    print("expert-named tumour clusters:", TUMOUR_CLUSTERS)
else:
    print("No cluster naming supplied: tumour localisation stays UNVERIFIED; features use all tissue.")

## 8. Multiple-instance learning: baselines, the proposed SANC-MIL, and ablations
**SANC-MIL** stands for *Stain-Anchored, Negative-Constrained, ordinal* MIL. It combines the following:
* **Additive instance-to-bag decomposition.** The bag logit is $z=\sum_t a_t s_t$, where $s_t$ is a per-tile *positivity* logit and $a_t$ is
  gated attention. Each tile's contribution $a_t s_t$ is therefore exact. Attention alone only says "important", which includes important
  *negative* evidence. That is why attention maps are not positivity maps.
* **Negative-bag constraint.** In IHC 0/1+ cores, a pathologist saw no 3+-type staining. Under the MIL assumption, every tile there is a known
  negative. This is the one piece of instance-level truth the labels contain. It is not applied to 2+/FISH-negative cores, which are equivocal.
* **Top-k positive constraint.** HER2-positive cores must contain at least a small positive cluster ($k=\lceil 5\%\,n\rceil$). This mirrors the
  biopsy rule that a cohesive cluster suffices. No proportion rule is imposed.
* **Stain anchoring.** The 14 calibrated stain/morphology features are fused with the deep embedding.
* **Ordinal head.** CORAL on IHC 0–3, trained jointly with the FISH-resolved status.

**Baselines:** the stain-intensity rule of Pisula *et al.* (maximum tile DAB, then logistic regression), a calibrated stain-feature logistic
regression, mean pooling, max pooling, ABMIL (Ilse 2018, the BJC 2023 method), CLAM-SB and TransMIL-lite. All of them use the same tiles,
features, folds and class-balanced sampling.

In [ ]:
class BagStore:
    """All tiles of all bags resident on the GPU; batches are built by index gather (fast padded batching)."""
    def __init__(self, ids, key="emb"):
        self.ids = list(ids); self.pos = {iid: k for k, iid in enumerate(self.ids)}
        embs = [BAGS[i][key] for i in self.ids]; hands = [BAGS[i]["hand_z"] for i in self.ids]
        self.n = np.array([len(e) for e in embs]); self.off = np.concatenate([[0], np.cumsum(self.n)[:-1]])
        self.E = torch.from_numpy(np.concatenate(embs)).to(DEVICE)
        self.Hd = torch.from_numpy(np.concatenate(hands)).to(DEVICE).half()
    def batch(self, bag_pos, max_tiles=None, drop=0.0, rng=None):
        sel = []
        for b in bag_pos:
            idx = np.arange(self.n[b])
            if max_tiles and len(idx) > max_tiles: idx = np.sort(rng.choice(idx, max_tiles, replace=False))
            if drop > 0 and len(idx) > 4:
                k = rng.random(len(idx)) >= drop
                idx = idx[k] if k.sum() > 0 else idx
            sel.append(idx)
        L = max(len(s) for s in sel)
        I = np.full((len(sel), L), -1, np.int64)
        for r, (b, s) in enumerate(zip(bag_pos, sel)): I[r, :len(s)] = self.off[b] + s
        It = torch.from_numpy(I).to(DEVICE); mask = It >= 0; Ic = It.clamp(min=0)
        return self.E[Ic].float(), self.Hd[Ic].float(), mask, sel

_STORES = {}
def use_embedding(key="emb"):
    """Switch the tile-embedding store (ImageNet ResNet50 'emb' or pathology foundation model 'emb2')."""
    global STORE, EMB_DIM
    if key not in _STORES: _STORES[key] = BagStore(list(TR.image_id) + list(TE.image_id), key)
    STORE = _STORES[key]; EMB_DIM = STORE.E.shape[1]
use_embedding("emb")
TRP = np.array([STORE.pos[i] for i in TR.image_id]); TEP = np.array([STORE.pos[i] for i in TE.image_id])

def masked_softmax(lg, mask):
    return torch.softmax(lg.masked_fill(~mask, -1e4), dim=1)

class GatedAttn(nn.Module):
    def __init__(self, d, da=128):
        super().__init__(); self.V = nn.Linear(d, da); self.U = nn.Linear(d, da); self.w = nn.Linear(da, 1)
    def forward(self, h): return self.w(torch.tanh(self.V(h)) * torch.sigmoid(self.U(h))).squeeze(-1)

class MILModel(nn.Module):
    def __init__(self, arch, use_emb=True, use_hand=False, pool="additive", ordinal=False, d=256, p=0.25):
        super().__init__()
        self.arch, self.use_emb, self.use_hand, self.pool, self.ordinal = arch, use_emb, use_hand, pool, ordinal
        din = (EMB_DIM if use_emb else 0) + (len(HAND_NAMES) if use_hand else 0)
        self.hand_proj = nn.Sequential(nn.Linear(len(HAND_NAMES), 64), nn.ReLU()) if use_hand else None
        din = (EMB_DIM if use_emb else 0) + (64 if use_hand else 0)
        self.proj = nn.Sequential(nn.Linear(din, d), nn.ReLU(), nn.Dropout(p))
        self.cls = nn.Linear(d, 1)
        if arch in ("abmil", "clam", "sanc"): self.attn = GatedAttn(d)
        if arch in ("max", "sanc"): self.inst = nn.Sequential(nn.Dropout(p), nn.Linear(d, 1))
        if arch == "clam": self.inst_clf = nn.ModuleList([nn.Linear(d, 2) for _ in range(2)])
        if arch == "transmil":
            self.cls_tok = nn.Parameter(torch.zeros(1, 1, d))
            self.tf = nn.TransformerEncoder(nn.TransformerEncoderLayer(d, 4, 2 * d, 0.1, batch_first=True), 2)
        if ordinal:
            self.ord_fc = nn.Linear(d, 1, bias=False); self.ord_b = nn.Parameter(torch.tensor([0.0, -1.0, -2.0]))

    def forward(self, emb, hand, mask):
        parts = []
        if self.use_emb: parts.append(emb)
        if self.use_hand: parts.append(self.hand_proj(hand))
        h = self.proj(torch.cat(parts, -1)); m = mask.float()
        out = {"inst": None, "attn": None}
        if self.arch == "mean":
            z = (h * m[..., None]).sum(1) / m.sum(1, keepdim=True).clamp(min=1)
            out["logit"] = self.cls(z).squeeze(-1); out["inst"] = self.cls(h).squeeze(-1)
        elif self.arch == "max":
            s = self.inst(h).squeeze(-1); out["inst"] = s
            out["logit"] = s.masked_fill(~mask, -1e4).max(1).values
            z = (h * m[..., None]).sum(1) / m.sum(1, keepdim=True).clamp(min=1)
        elif self.arch in ("abmil", "clam"):
            a = masked_softmax(self.attn(h), mask); out["attn"] = a
            z = (a[..., None] * h).sum(1); out["logit"] = self.cls(z).squeeze(-1)
            if self.arch == "clam":
                l2 = self.inst_clf[1](h); out["inst"] = l2[..., 1] - l2[..., 0]; out["h"] = h
        elif self.arch == "transmil":
            t = torch.cat([self.cls_tok.expand(h.shape[0], -1, -1), h], 1)
            km = torch.cat([torch.zeros(h.shape[0], 1, dtype=torch.bool, device=h.device), ~mask], 1)
            z = self.tf(t, src_key_padding_mask=km)[:, 0]; out["logit"] = self.cls(z).squeeze(-1)
        elif self.arch == "sanc":
            a = masked_softmax(self.attn(h), mask); s = self.inst(h).squeeze(-1)
            out["attn"], out["inst"] = a, s
            z = (a[..., None] * h).sum(1)
            out["logit"] = (a * s).sum(1) if self.pool == "additive" else self.cls(z).squeeze(-1)
            out["contrib"] = a * s
        if self.ordinal:
            out["ord"] = self.ord_fc(z) + self.ord_b
        return out

VARIANTS = {
    "MeanPool":                     dict(arch="mean"),
    "MaxPool":                      dict(arch="max"),
    "ABMIL (Ilse; BJC-2023)":       dict(arch="abmil"),
    "CLAM-SB":                      dict(arch="clam"),
    "TransMIL-lite":                dict(arch="transmil"),
    "SANC-MIL (proposed)":          dict(arch="sanc", use_hand=True, neg=True, pos=True, ordinal=True, pool="additive"),
    "SANC − stain features":        dict(arch="sanc", use_hand=False, neg=True, pos=True, ordinal=True, pool="additive"),
    "SANC − negative constraint":   dict(arch="sanc", use_hand=True, neg=False, pos=True, ordinal=True, pool="additive"),
    "SANC − top-k constraint":      dict(arch="sanc", use_hand=True, neg=True, pos=False, ordinal=True, pool="additive"),
    "SANC − ordinal head":          dict(arch="sanc", use_hand=True, neg=True, pos=True, ordinal=False, pool="additive"),
    "SANC − additive (attn pool)":  dict(arch="sanc", use_hand=True, neg=True, pos=True, ordinal=True, pool="attention"),
    "Stain-only MIL (no deep emb)": dict(arch="sanc", use_emb=False, use_hand=True, neg=True, pos=True, ordinal=True, pool="additive"),
}
PROPOSED = "SANC-MIL (proposed)"; MAIN_BASELINE = "ABMIL (Ilse; BJC-2023)"

def build(v):
    return MILModel(v["arch"], use_emb=v.get("use_emb", True), use_hand=v.get("use_hand", False),
                    pool=v.get("pool", "additive"), ordinal=v.get("ordinal", False)).to(DEVICE)

Y_STATUS = torch.tensor(np.concatenate([TR.her2_status.values, TE.her2_status.values]), dtype=torch.float32, device=DEVICE)
Y_SCORE = torch.tensor(np.concatenate([TR.ihc_score.values, TE.ihc_score.values]), dtype=torch.long, device=DEVICE)

def mil_loss(out, bpos, v, mask):
    ys, ysc = Y_STATUS[bpos], Y_SCORE[bpos]
    loss = F.binary_cross_entropy_with_logits(out["logit"], ys)
    if v.get("ordinal"):
        lv = torch.stack([(ysc > k).float() for k in range(3)], 1)
        loss = loss + CFG.W_ORD * F.binary_cross_entropy_with_logits(out["ord"], lv)
    s = out["inst"]
    if v.get("neg") and s is not None:
        nb = (ysc <= 1) & (ys == 0)
        if nb.any():
            mm = mask[nb]
            loss = loss + CFG.W_NEG * F.binary_cross_entropy_with_logits(s[nb][mm], torch.zeros_like(s[nb][mm]))
    if v.get("pos") and s is not None:
        pb = ys == 1
        if pb.any():
            tk = []
            for r in torch.nonzero(pb).flatten():
                sv = s[r][mask[r]]; k = max(1, int(math.ceil(CFG.TOPK_FRAC * sv.numel())))
                tk.append(sv.topk(k).values.mean())
            tk = torch.stack(tk); loss = loss + CFG.W_POS * F.binary_cross_entropy_with_logits(tk, torch.ones_like(tk))
    if v["arch"] == "clam":
        a, h = out["attn"], out["h"]; il = []
        for r in range(h.shape[0]):
            n = int(mask[r].sum()); k = max(1, min(8, n // 4))
            if n < 2: continue
            ar = a[r][:n]; top = ar.topk(k).indices; bot = (-ar).topk(k).indices
            lg = model_clam_inst(h[r], int(ys[r].item()), top, bot)
            il.append(lg)
        if il: loss = loss + CFG.W_CLAM * torch.stack(il).mean()
    return loss

_CLAM_REF = {}
def model_clam_inst(hr, y, top, bot):
    m = _CLAM_REF["m"]; clf = m.inst_clf[y]
    lg = clf(torch.cat([hr[top], hr[bot]]))
    tgt = torch.cat([torch.ones(len(top)), torch.zeros(len(bot))]).long().to(hr.device)
    return F.cross_entropy(lg, tgt)

def train_mil(v, train_pos, seed, epochs=None):
    seed_everything(seed); rng = np.random.default_rng(seed)
    m = build(v); _CLAM_REF["m"] = m
    opt = torch.optim.AdamW(m.parameters(), lr=CFG.MIL_LR, weight_decay=CFG.MIL_WD)
    epochs = epochs or CFG.MIL_EPOCHS
    grp = np.concatenate([TR.group.values, TE.group.values])[train_pos]
    freq = pd.Series(grp).value_counts(); w = np.array([1.0 / freq[g] for g in grp]); w /= w.sum()
    steps = int(math.ceil(len(train_pos) / CFG.MIL_BS))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG.MIL_LR, total_steps=epochs * steps, pct_start=0.1)
    m.train(); m.loss_hist = []
    for ep in range(epochs):
        draw = rng.choice(train_pos, size=steps * CFG.MIL_BS, replace=True, p=w); ep_loss = []
        for s in range(steps):
            bp = draw[s * CFG.MIL_BS:(s + 1) * CFG.MIL_BS]
            e, hd, mask, _ = STORE.batch(bp, CFG.MIL_MAX_TILES_TRAIN, CFG.MIL_TILE_DROP, rng)
            out = m(e, hd, mask); loss = mil_loss(out, torch.from_numpy(bp).to(DEVICE), v, mask)
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step(); sched.step(); ep_loss.append(loss.item())
        m.loss_hist.append(float(np.mean(ep_loss)))
    m.eval(); return m

def _enable_dropout(m):
    for mod in m.modules():
        if isinstance(mod, nn.Dropout): mod.train()

@torch.no_grad()
def predict_mil(m, pos, mc=0, bs=8):
    m.eval(); res = []
    for b in range(0, len(pos), bs):
        bp = pos[b:b + bs]
        e, hd, mask, sel = STORE.batch(bp)
        out = m(e, hd, mask)
        mcl = None
        if mc:
            _enable_dropout(m); mcl = torch.stack([m(e, hd, mask)["logit"] for _ in range(mc)], 1).cpu().numpy(); m.eval()
        for r in range(len(bp)):
            n = int(mask[r].sum())
            d = dict(logit=float(out["logit"][r]), inst=None if out["inst"] is None else out["inst"][r, :n].cpu().numpy(),
                     attn=None if out["attn"] is None else out["attn"][r, :n].cpu().numpy())
            if "ord" in out: d["ord"] = out["ord"][r].cpu().numpy()
            if mcl is not None: d["mc"] = mcl[r]
            res.append(d)
    return res

def slug(name): return re.sub("[^A-Za-z0-9]+", "_", name).strip("_")
def model_path(name, k): return OUT / "models" / f"{slug(name)}_fold{k}.pt"
def fold_models(name):
    """Fold models of `name` (loaded if saved, otherwise re-trained deterministically and saved)."""
    v = VARIANTS[name]; ms = []
    for k in range(CFG.N_FOLDS):
        p = model_path(name, k)
        if p.exists():
            m = build(v); m.load_state_dict(torch.load(p, map_location=DEVICE))
        else:
            m = train_mil(v, TRP[TR.fold.values != k], seed=k); atomic_torch_save(m.state_dict(), p)
        ms.append(m.eval())
    return ms

def run_cv(name, v, fold_col="fold", seed=0, mc_test=0, emb="emb"):
    tag = "" if emb == "emb" else f"_{emb}"
    path = CACHE / "mil" / f"{re.sub('[^A-Za-z0-9]+', '_', name)}_{fold_col}_s{seed}{tag}.pkl"
    cached = load_pickle(path) if path.exists() else None
    if cached is not None: return cached
    use_embedding(emb)
    oof = [None] * len(TR); test_runs = []; losses = []
    for k in range(CFG.N_FOLDS):
        tr_idx = np.where(TR[fold_col].values != k)[0]; va_idx = np.where(TR[fold_col].values == k)[0]
        m = train_mil(v, TRP[tr_idx], seed=seed * 100 + k)
        for j, d in zip(va_idx, predict_mil(m, TRP[va_idx], mc=mc_test)): oof[j] = d
        test_runs.append(predict_mil(m, TEP, mc=mc_test)); losses.append(getattr(m, "loss_hist", []))
        if name in (PROPOSED, MAIN_BASELINE) and fold_col == "fold" and seed == 0 and emb == "emb":
            atomic_torch_save(m.state_dict(), model_path(name, k))
        del m; torch.cuda.empty_cache()
    use_embedding("emb")
    res = dict(oof=oof, test=test_runs, loss=losses); atomic_pickle(res, path); return res

MIL_RES = {}
t0 = time.time()
for name, v in VARIANTS.items():
    mc = CFG.MC_SAMPLES if name in (PROPOSED, MAIN_BASELINE) else 0
    MIL_RES[name] = run_cv(name, v, mc_test=mc)
    print(f"{name:32s} done  ({(time.time() - t0) / 60:.1f} min)")
LEAK_RES = {n: run_cv(n, VARIANTS[n], fold_col="fold_random") for n in (PROPOSED, MAIN_BASELINE)}
# repeated CV seeds for the variants the paper's claims rest on (seed 0 = the main run above)
SEED_VARIANTS = [PROPOSED, "SANC − stain features", "SANC − negative constraint", MAIN_BASELINE, "MaxPool"]
SEED_RES = {n: [MIL_RES[n]] + [run_cv(n, VARIANTS[n], seed=sd) for sd in range(1, CFG.N_SEEDS)] for n in SEED_VARIANTS}
# encoder ablation: same MIL variants on pathology-foundation-model tile embeddings (extracted in pass 1)
ENC_VARIANTS = [PROPOSED, "SANC − stain features", MAIN_BASELINE, "MaxPool"]
ENC_RES = {}
if HAS_EMB2:
    for n in ENC_VARIANTS:
        ENC_RES[f"{n} [{CFG.ENCODER2.split('/')[-1]}]"] = run_cv(n, VARIANTS[n], emb="emb2")
    print("encoder ablation done:", list(ENC_RES))
section_time("MIL training (variants, leakage, seeds, encoder ablation)")

### 8b. Grad-CAM baseline: whole-core CNN (the approach of the original draft and of Pisula et al.'s ResNet)
A ResNet-18 is trained on the whole-core thumbnail (≈1.8 µm/px) with the same folds and class-balanced sampling. Its
Grad-CAM map is pooled onto the same 224-px tiles, so it can be scored with the **same** localisation metrics.

Two read-outs are reported:
1. the per-image min–max-normalised CAM, as in the original draft
2. CAM × P(HER2+)

Per-image normalisation guarantees "hot" tiles even in IHC-0 cores. The tile false-positive rate in IHC-0 cores
quantifies that flaw.

In [ ]:
CAM_SIZE = 672 if RUN_MODE != "debug" else 128
def _load_thumb(iid):
    im = cv2.cvtColor(cv2.imread(str(THUMB_DIR / f"{iid}.jpg")), cv2.COLOR_BGR2RGB)
    return cv2.resize(im, (CAM_SIZE, CAM_SIZE), interpolation=cv2.INTER_AREA), im.shape[:2]
def _cam_logit(v): v = np.clip(v, 1e-4, 1 - 1e-4); return np.log(v / (1 - v))

def run_gradcam_baseline():
    path = CACHE / "mil" / "gradcam_thumbnail_cnn_v2.pkl"
    cached = load_pickle(path) if path.exists() else None
    if cached is not None: return cached
    ids = list(TR.image_id) + list(TE.image_id); shapes = {}
    Xn = np.empty((len(ids), CAM_SIZE, CAM_SIZE, 3), np.uint8)       # filled in place: 1,905 x 672^2 x 3 = 2.6 GB, no duplicate copy
    for k, iid in enumerate(tqdm(ids, desc="load thumbnails")): Xn[k], shapes[iid] = _load_thumb(iid)
    X = torch.from_numpy(Xn).permute(0, 3, 1, 2)                     # uint8 view on CPU; batches go to the GPU one at a time
    ypos = np.concatenate([TR.her2_status.values, TE.her2_status.values]).astype(np.float32)
    grp = np.concatenate([TR.group.values, TE.group.values]); ntr = len(TR)
    mean = torch.tensor(IMNET_MEAN, device=DEVICE).view(1, 3, 1, 1); std = torch.tensor(IMNET_STD, device=DEVICE).view(1, 3, 1, 1)
    def mk():
        try: return timm.create_model(CFG.CAM_ARCH, pretrained=True, num_classes=1).to(DEVICE)
        except Exception: return timm.create_model(CFG.CAM_ARCH, pretrained=False, num_classes=1).to(DEVICE)
    def tile_scores(cam, iid):
        h0, w0 = shapes[iid]; c = cv2.resize(cam, (w0, h0), interpolation=cv2.INTER_LINEAR)
        T, R = CFG.TILE, CFG.THUMB_REL
        out = []
        for y, x in BAGS[iid]["coords"]:      # clamp to >=1 px inside the thumbnail, so an edge tile can never give an empty-slice NaN
            y0 = min(y // R, h0 - 1); x0 = min(x // R, w0 - 1)
            out.append(c[y0:max(y0 + 1, (y + T) // R), x0:max(x0 + 1, (x + T) // R)].mean())
        return np.array(out, np.float32)
    def gradcam(m, idx):
        feats = {}
        h = m.layer4.register_forward_hook(lambda mod, i, o: feats.__setitem__("a", o))
        out = []
        m.eval()
        for b in range(0, len(idx), 8):
            bi = idx[b:b + 8]; x = ((X[bi].to(DEVICE).float() / 255) - mean) / std
            x.requires_grad_(False); lg = m(x).squeeze(1); a = feats["a"]
            g = torch.autograd.grad(lg.sum(), a)[0]
            cam = F.relu((g.mean((2, 3), keepdim=True) * a).sum(1)).detach().cpu().numpy()
            for j, k in enumerate(bi):
                cn = np.nan_to_num(cam[j], nan=0.0, posinf=0.0, neginf=0.0); cn = (cn - cn.min()) / (cn.max() - cn.min() + 1e-8)
                ts = tile_scores(cn, ids[k]); out.append((float(lg[j].detach()), ts))
        h.remove(); return out
    oof = [None] * ntr; test_norm, test_prob = [], []
    for k in range(CFG.N_FOLDS):
        tr_idx = np.where(TR.fold.values != k)[0]; va_idx = np.where(TR.fold.values == k)[0]
        fold_path = CACHE / "mil" / f"gradcam_v2_fold{k}.pkl"; fk = load_pickle(fold_path) if fold_path.exists() else None
        if fk is not None:                       # resume: this fold finished in an earlier run
            for j, d in zip(va_idx, fk["oof"]): oof[j] = d
            test_norm.append(fk["test_norm"]); test_prob.append(fk["test_prob"])
            print(f"  Grad-CAM CNN fold {k}: done earlier, loaded from cache"); continue
        seed_everything(500 + k); m = mk(); opt = torch.optim.AdamW(m.parameters(), lr=3e-4, weight_decay=1e-4)
        freq = pd.Series(grp[tr_idx]).value_counts(); w = np.array([1 / freq[g] for g in grp[tr_idx]]); w /= w.sum()
        steps = int(math.ceil(len(tr_idx) / CFG.CAM_BS))
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-4, total_steps=max(20, CFG.CAM_EPOCHS * steps), pct_start=0.1)
        rng = np.random.default_rng(k)
        # fp16 autocast needs loss scaling, otherwise small gradients underflow to zero and the baseline is under-trained
        cam_scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda") if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler") \
            else torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")
        for ep in range(CFG.CAM_EPOCHS):
            m.train(); draw = rng.choice(tr_idx, steps * CFG.CAM_BS, replace=True, p=w)
            for st in range(steps):
                bi = draw[st * CFG.CAM_BS:(st + 1) * CFG.CAM_BS]; x = X[bi].to(DEVICE).float() / 255
                if random.random() < 0.5: x = x.flip(-1)
                if random.random() < 0.5: x = x.flip(-2)
                x = torch.rot90(x, random.randint(0, 3), (-2, -1))
                x = (x * (1 + (torch.rand(len(bi), 3, 1, 1, device=DEVICE) - 0.5) * 0.1)).clamp(0, 1)
                with autocast(): lg = m((x - mean) / std).squeeze(1)
                loss = F.binary_cross_entropy_with_logits(lg.float(), torch.from_numpy(ypos[bi]).to(DEVICE))
                opt.zero_grad(set_to_none=True); cam_scaler.scale(loss).backward(); cam_scaler.step(opt); cam_scaler.update()
                if sched.last_epoch < sched.total_steps - 1: sched.step()
        for j, (lg, ts) in zip(va_idx, gradcam(m, list(va_idx))): oof[j] = dict(logit=lg, inst=None, attn=None)
        te = gradcam(m, list(range(ntr, len(ids))))
        test_norm.append([dict(logit=lg, inst=_cam_logit(ts), attn=None) for lg, ts in te])
        test_prob.append([dict(logit=lg, inst=_cam_logit(ts / (1 + math.exp(-lg))), attn=None) for lg, ts in te])
        atomic_pickle(dict(oof=[oof[j] for j in va_idx], test_norm=test_norm[-1], test_prob=test_prob[-1]), fold_path)
        print(f"  Grad-CAM CNN fold {k} done"); del m; torch.cuda.empty_cache()
    res = {"Grad-CAM CNN (per-image normalised CAM)": dict(oof=oof, test=test_norm),
           "Grad-CAM CNN (CAM × P(HER2+))": dict(oof=oof, test=test_prob)}
    atomic_pickle(res, path); return res

CAM_RES = run_gradcam_baseline() if CFG.RUN_GRADCAM else {}
section_time("Grad-CAM baseline")

In [ ]:
# ---------------- non-MIL baselines (stain intensity rules) ----------------
def core_stain_descriptor(iid):
    h = BAGS[iid]["hand_raw"]; g = lambda n: h[:, HAND_NAMES.index(n)]
    top = lambda x: np.sort(x)[-max(1, int(math.ceil(0.05 * len(x)))):].mean()
    return [g("dab_mean").max(), top(g("mem_weak")), top(g("mem_mod")), top(g("mem_strong")), g("mem_weak").mean(),
            g("mem_mod").mean(), g("mem_strong").mean(), g("nonmem_dab").mean(), g("nuclear_dab").mean()]
XS_tr = np.array([core_stain_descriptor(i) for i in TR.image_id]); XS_te = np.array([core_stain_descriptor(i) for i in TE.image_id])

def sk_cv(Xtr, Xte, y, fold_col="fold"):
    oof = np.zeros(len(Xtr)); te = []
    for k in range(CFG.N_FOLDS):
        tr = TR[fold_col].values != k
        mdl = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000, C=1.0)).fit(Xtr[tr], y[tr])
        oof[~tr] = mdl.decision_function(Xtr[~tr]); te.append(mdl.decision_function(Xte))
    return oof, np.array(te)
ystat_tr, ystat_te = TR.her2_status.values, TE.her2_status.values
SK_RES = {"Stain intensity rule (Pisula 2023)": sk_cv(XS_tr[:, :1], XS_te[:, :1], ystat_tr),
          "Calibrated stain features + LR": sk_cv(XS_tr, XS_te, ystat_tr)}

## 9. Classification evaluation (independent of localisation)
Logits are averaged over the 5 fold models. The temperature and the decision threshold (maximum balanced accuracy) are fitted **on the OOF
predictions only**, then frozen for the test set. CIs come from a stratified bootstrap with `N_BOOT` resamples. The paired bootstrap ΔAUC compares
the proposed model with each baseline.

With only about 16 positive test cores, CIs are wide. Small differences are **not** evidence of superiority.

In [ ]:
def sig(x): return 1 / (1 + np.exp(-np.asarray(x, float)))
def fit_temperature(logit, y):
    lt = torch.tensor(logit, dtype=torch.float64); yt = torch.tensor(y, dtype=torch.float64)
    lT = torch.zeros(1, dtype=torch.float64, requires_grad=True); opt = torch.optim.LBFGS([lT], lr=0.1, max_iter=200)
    def cl():
        opt.zero_grad(); l = F.binary_cross_entropy_with_logits(lt / lT.exp(), yt); l.backward(); return l
    opt.step(cl); return float(lT.exp())
def best_threshold(y, p):
    grid = np.unique(np.quantile(p, np.linspace(0.01, 0.99, 199)))
    return float(grid[np.argmax([balanced_accuracy_score(y, p >= t) for t in grid])])
def ece(y, p, n=10):
    b = np.minimum((p * n).astype(int), n - 1); e = 0
    for k in range(n):
        s = b == k
        if s.any(): e += s.mean() * abs(p[s].mean() - y[s].mean())
    return float(e)
def metrics(y, p, thr):
    yh = (p >= thr).astype(int); tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0, 1]).ravel()
    return dict(AUROC=roc_auc_score(y, p) if len(set(y)) > 1 else np.nan, AUPRC=average_precision_score(y, p),
                BalAcc=balanced_accuracy_score(y, yh), Sens=tp / max(tp + fn, 1), Spec=tn / max(tn + fp, 1),
                F1=f1_score(y, yh, zero_division=0), MCC=matthews_corrcoef(y, yh), Brier=brier_score_loss(y, p), ECE=ece(y, p))
def boot_ci(y, p, thr, n=None, seed=0):
    n = n or CFG.N_BOOT; r = np.random.default_rng(seed); pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]; out = []
    for _ in range(n):
        idx = np.concatenate([r.choice(pos, len(pos)), r.choice(neg, len(neg))])
        out.append(metrics(y[idx], p[idx], thr))
    d = pd.DataFrame(out); return d.quantile(0.025), d.quantile(0.975)

def summarize(name, oof_logit, test_logit_folds, y_tr=ystat_tr, y_te=ystat_te):
    T = fit_temperature(oof_logit, y_tr); p_oof = sig(oof_logit / T)
    thr = best_threshold(y_tr, p_oof)
    p_te = sig(np.mean(test_logit_folds, 0) / T)
    m_te = metrics(y_te, p_te, thr); lo, hi = boot_ci(y_te, p_te, thr)
    m_oof = metrics(y_tr, p_oof, thr)
    row = {"method": name, "temp": T, "thr": thr, "OOF_AUROC": m_oof["AUROC"], "OOF_BalAcc": m_oof["BalAcc"]}
    for k in ["AUROC", "AUPRC", "BalAcc", "Sens", "Spec", "F1", "MCC", "Brier", "ECE"]:
        row[k] = m_te[k]; row[k + "_lo"] = lo[k]; row[k + "_hi"] = hi[k]
    return row, p_oof, p_te

ROWS, P_TE, P_OOF = [], {}, {}
for name, (oof, te) in SK_RES.items():
    r, po, pt = summarize(name, oof, te); ROWS.append(r); P_TE[name] = pt; P_OOF[name] = po
_cls_sources = {**MIL_RES, **ENC_RES}
if CAM_RES: _cls_sources["Grad-CAM CNN (thumbnail)"] = next(iter(CAM_RES.values()))
for name, res in _cls_sources.items():
    oof = np.array([d["logit"] for d in res["oof"]]); te = np.array([[d["logit"] for d in run] for run in res["test"]])
    r, po, pt = summarize(name, oof, te); ROWS.append(r); P_TE[name] = pt; P_OOF[name] = po
CLS = pd.DataFrame(ROWS)
fmt = lambda r, k: f"{r[k]:.3f} [{r[k + '_lo']:.3f}–{r[k + '_hi']:.3f}]"
CLS_PRETTY = pd.DataFrame({"method": CLS.method, "OOF AUROC (grouped CV)": CLS.OOF_AUROC.round(3),
                           **{f"Test {k}": [fmt(r, k) for _, r in CLS.iterrows()] for k in ["AUROC", "AUPRC", "BalAcc", "Sens", "Spec", "MCC"]},
                           "Test ECE": CLS.ECE.round(3)})
CLS.to_csv(TAB / "classification_full.csv", index=False); CLS_PRETTY.to_csv(TAB / "classification_table.csv", index=False)
print(CLS_PRETTY.to_string(index=False))

def paired_delta_auc(pa, pb, y, n=None, seed=1):
    n = n or CFG.N_BOOT; r = np.random.default_rng(seed); pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]; d = []
    for _ in range(n):
        idx = np.concatenate([r.choice(pos, len(pos)), r.choice(neg, len(neg))])
        d.append(roc_auc_score(y[idx], pa[idx]) - roc_auc_score(y[idx], pb[idx]))
    d = np.array(d); return float(np.mean(d)), float(np.quantile(d, 0.025)), float(np.quantile(d, 0.975)), float(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())))
DELTA = pd.DataFrame([dict(comparison=f"{PROPOSED} − {b}", **dict(zip(["dAUC", "lo", "hi", "p_boot"], paired_delta_auc(P_TE[PROPOSED], P_TE[b], ystat_te))))
                      for b in P_TE if b != PROPOSED])
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for r, i in enumerate(o): run = max(run, min(1.0, (len(p) - r) * p[i])); adj[i] = run
    return adj
DELTA["p_holm"] = holm(DELTA.p_boot.values)     # family-wise correction over all comparisons with the proposed model
DELTA.to_csv(TAB / "paired_delta_auc.csv", index=False); print(DELTA.round(3).to_string(index=False))

# per-IHC-group error profile (mirrors Table 2 of Pisula et al.)
def group_profile(name):
    r = CLS[CLS.method == name].iloc[0]; yh = (P_TE[name] >= r.thr).astype(int)
    d = TE.assign(pred=yh)
    return d.groupby("group").apply(lambda s: pd.Series({"n": len(s), "pred_pos_rate": s.pred.mean(),
                                                          "error_rate": (s.pred != s.her2_status).mean()})).reindex(GROUP_ORDER)
GP = pd.concat({n: group_profile(n) for n in [PROPOSED, MAIN_BASELINE, "Stain intensity rule (Pisula 2023)"]}, axis=1)
GP.to_csv(TAB / "test_error_by_ihc_group.csv"); print(GP.round(3))

# leakage sensitivity: grouped vs random CV (OOF AUROC). A large gap = within-patient correlation inflating random CV.
LEAK = []
for n, res in LEAK_RES.items():
    oof_r = np.array([d["logit"] for d in res["oof"]]); oof_g = np.array([d["logit"] for d in MIL_RES[n]["oof"]])
    LEAK.append(dict(method=n, OOF_AUROC_random_CV=roc_auc_score(ystat_tr, oof_r), OOF_AUROC_proxy_grouped_CV=roc_auc_score(ystat_tr, oof_g)))
LEAK = pd.DataFrame(LEAK); LEAK.to_csv(TAB / "leakage_sensitivity.csv", index=False); print(LEAK.round(4))

# QC sensitivity: test metrics without QC-flagged cores
flag_te = TE.qc_flag.values.astype(bool)
QCS = pd.DataFrame([dict(method=n, AUROC_all=roc_auc_score(ystat_te, P_TE[n]),
                         AUROC_unflagged=roc_auc_score(ystat_te[~flag_te], P_TE[n][~flag_te]) if len(set(ystat_te[~flag_te])) > 1 else np.nan,
                         n_flagged=int(flag_te.sum())) for n in [PROPOSED, MAIN_BASELINE]])
QCS.to_csv(TAB / "qc_sensitivity.csv", index=False)

# ordinal IHC score from SANC (CORAL -> class probabilities)
def coral_probs(ordl):
    pk = sig(ordl); p = np.stack([1 - pk[:, 0], pk[:, 0] - pk[:, 1], pk[:, 1] - pk[:, 2], pk[:, 2]], 1)
    return np.clip(p, 0, 1) / np.clip(p, 0, 1).sum(1, keepdims=True)
ORD_TE = coral_probs(np.mean([[d["ord"] for d in run] for run in MIL_RES[PROPOSED]["test"]], 0))
ORD_OOF = coral_probs(np.array([d["ord"] for d in MIL_RES[PROPOSED]["oof"]]))
score_pred = ORD_TE.argmax(1)
ord_metrics = dict(test_QWK=cohen_kappa_score(TE.ihc_score, score_pred, weights="quadratic"),
                   test_BalAcc_present_classes=balanced_accuracy_score(TE.ihc_score, score_pred),
                   OOF_QWK=cohen_kappa_score(TR.ihc_score, ORD_OOF.argmax(1), weights="quadratic"),
                   classes_present_in_test=sorted(TE.ihc_score.unique().tolist()))
save_json(ord_metrics, TAB / "ordinal_score_metrics.json"); print(ord_metrics)

# figures: ROC + PR + reliability + score confusion
fig, ax = plt.subplots(1, 4, figsize=(15, 3.6))
show = ["Stain intensity rule (Pisula 2023)", "Calibrated stain features + LR", MAIN_BASELINE, "CLAM-SB", PROPOSED]
for c, n in zip([PAL[3], PAL[2], PAL[0], PAL[6], PAL[1]], show):
    f, t, _ = roc_curve(ystat_te, P_TE[n]); r = CLS[CLS.method == n].iloc[0]
    ax[0].plot(f, t, color=c, label=f"{n.split(' (')[0]}  {r.AUROC:.3f} [{r.AUROC_lo:.2f}–{r.AUROC_hi:.2f}]")
    pr, rc, _ = precision_recall_curve(ystat_te, P_TE[n]); ax[1].plot(rc, pr, color=c)
ax[0].plot([0, 1], [0, 1], color="#c3c2b7", lw=1, ls="--"); ax[0].set_xlabel("1 − specificity"); ax[0].set_ylabel("sensitivity")
ax[0].legend(fontsize=6.5, loc="lower right"); ax[0].set_title(f"Test ROC (n={len(TE)}, {int(ystat_te.sum())} HER2+)")
ax[1].set_xlabel("recall"); ax[1].set_ylabel("precision"); ax[1].set_title("Test precision–recall")
for c, n in zip([PAL[0], PAL[1]], [MAIN_BASELINE, PROPOSED]):
    p = P_OOF[n]; b = np.minimum((p * 10).astype(int), 9)
    xs = [p[b == k].mean() for k in range(10) if (b == k).sum() > 5]; ys = [ystat_tr[b == k].mean() for k in range(10) if (b == k).sum() > 5]
    ax[2].plot(xs, ys, "o-", color=c, ms=4, label=n.split(" (")[0])
ax[2].plot([0, 1], [0, 1], color="#c3c2b7", lw=1, ls="--"); ax[2].set_xlabel("predicted P(HER2+)"); ax[2].set_ylabel("observed")
ax[2].set_title("Reliability (OOF, temperature-scaled)"); ax[2].legend(fontsize=7)
cm = confusion_matrix(TE.ihc_score, score_pred, labels=[0, 1, 2, 3])
ax[3].imshow(cm, cmap="Blues"); ax[3].grid(False)
for i in range(4):
    for j in range(4): ax[3].text(j, i, cm[i, j], ha="center", va="center", fontsize=8, color="#0b0b0b")
ax[3].set_xticks(range(4)); ax[3].set_yticks(range(4)); ax[3].set_xlabel("predicted IHC"); ax[3].set_ylabel("true IHC")
ax[3].set_title(f"SANC ordinal head (QWK {ord_metrics['test_QWK']:.2f})")
savefig(fig, "fig06_classification")

## 10. Uncertainty, abstention, and guideline-faithful handling of equivocal cases
* **Uncertainty.** The deep ensemble (5 folds) is combined with MC-dropout. The outputs are predictive entropy (total uncertainty) and mutual
  information (epistemic uncertainty).
* **Abstention.** The entropy threshold is chosen on OOF for `ABSTAIN_COVERAGE`, then applied to the test set. Risk–coverage curves show the trade-off.
* **Equivocal routing.** If the ordinal head gives P(IHC 2) ≥ 0.3 or predicts 2, the core is flagged "**ISH required**", whatever the status
  probability. The model's status output for those cores is reported as research information only.

In [ ]:
def mc_stats(res, T):
    runs = np.stack([np.stack([d["mc"] for d in run]) for run in res["test"]], 1)  # (n, folds, mc)
    p = sig(runs / T).reshape(len(TE), -1); pm = p.mean(1)
    H = lambda q: -(q * np.log(q + 1e-9) + (1 - q) * np.log(1 - q + 1e-9))
    return pm, H(pm), H(pm) - H(p).mean(1)
def oof_entropy(res, T):
    p = sig(np.array([d["mc"] for d in res["oof"]]) / T); pm = p.mean(1)
    return -(pm * np.log(pm + 1e-9) + (1 - pm) * np.log(1 - pm + 1e-9))
UNC = {}
for n in [PROPOSED, MAIN_BASELINE]:
    T = CLS[CLS.method == n].temp.iloc[0]; thr = CLS[CLS.method == n].thr.iloc[0]
    pm, ent, mi = mc_stats(MIL_RES[n], T)
    tau = np.quantile(oof_entropy(MIL_RES[n], T), CFG.ABSTAIN_COVERAGE)
    keep = ent <= tau
    cov = np.linspace(0.3, 1, 15); risk = []
    order = np.argsort(ent)
    for c in cov:
        s = order[:max(2, int(c * len(order)))]; yh = (pm[s] >= thr).astype(int)
        risk.append(1 - balanced_accuracy_score(ystat_te[s], yh) if len(set(ystat_te[s])) > 1 else np.nan)
    UNC[n] = dict(pm=pm, ent=ent, mi=mi, tau=tau, keep=keep, cov=cov, risk=np.array(risk), thr=thr)
pu = UNC[PROPOSED]
# ISH routing: threshold on P(IHC 2) chosen on OUT-OF-FOLD predictions to reach the target sensitivity for IHC 2 cores
# (v1 used a fixed 0.3 cut-off and caught only 40% of test IHC 2 cores).
_oof2 = ORD_OOF[TR.ihc_score.values == 2, 2]
ISH_TAU = float(np.quantile(_oof2, 1 - CFG.ISH_ROUTE_TARGET_SENS)) if len(_oof2) else 0.5
route_oof = ORD_OOF[:, 2] >= ISH_TAU
route_ish = ORD_TE[:, 2] >= ISH_TAU
ab = dict(abstain_rate_test=float(1 - pu["keep"].mean()),
          balacc_all=float(balanced_accuracy_score(ystat_te, pu["pm"] >= pu["thr"])),
          balacc_retained=float(balanced_accuracy_score(ystat_te[pu["keep"]], pu["pm"][pu["keep"]] >= pu["thr"])) if len(set(ystat_te[pu["keep"]])) > 1 else np.nan,
          ISH_routing_sensitivity_for_true_IHC2=float(route_ish[TE.ihc_score.values == 2].mean()) if (TE.ihc_score == 2).any() else np.nan,
          ISH_routing_rate_in_non_IHC2=float(route_ish[TE.ihc_score.values != 2].mean()),
          ISH_threshold_P_IHC2=ISH_TAU, ISH_routing_OOF_sensitivity=float(route_oof[TR.ihc_score.values == 2].mean()),
          ISH_routing_OOF_rate_non_IHC2=float(route_oof[TR.ihc_score.values != 2].mean()),
          ISH_routing_test_rate_IHC0=float(route_ish[TE.ihc_score.values == 0].mean()),
          ISH_routing_test_rate_IHC3=float(route_ish[TE.ihc_score.values == 3].mean()) if (TE.ihc_score == 3).any() else np.nan,
          AURC_proposed=float(np.nanmean(UNC[PROPOSED]["risk"])), AURC_abmil=float(np.nanmean(UNC[MAIN_BASELINE]["risk"])))
save_json(ab, TAB / "abstention.json"); print(ab)
section_time("classification evaluation + uncertainty")
TE_PRED = TE[["image_id", "filename", "ihc_score", "her2_status", "group", "qc_flag"]].assign(
    p_her2=pu["pm"], entropy=pu["ent"], mutual_info=pu["mi"], abstain=~pu["keep"],
    **{f"p_ihc{k}": ORD_TE[:, k] for k in range(4)}, route_ISH=route_ish,
    decision=np.where(~pu["keep"] | TE.qc_flag.values, "ABSTAIN (refer)", np.where(route_ish, "EQUIVOCAL: ISH required",
                      np.where(pu["pm"] >= pu["thr"], "HER2-positive (model)", "HER2-negative (model)"))))
TE_PRED.to_csv(TAB / "test_predictions.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.3))
for c, n in zip([PAL[0], PAL[1]], [MAIN_BASELINE, PROPOSED]):
    ax[0].plot(UNC[n]["cov"], UNC[n]["risk"], "o-", ms=3, color=c, label=n.split(" (")[0])
ax[0].set_xlabel("coverage"); ax[0].set_ylabel("balanced error"); ax[0].set_title("Risk–coverage (test)"); ax[0].legend(fontsize=7)
for g in GROUP_ORDER:
    s = TE.group.values == g
    if s.any(): ax[1].scatter(pu["pm"][s], pu["ent"][s], s=14, color=GROUP_COL[g], label=f"IHC {g} (n={s.sum()})", alpha=0.8)
ax[1].axhline(pu["tau"], color="#52514e", ls="--", lw=1); ax[1].set_xlabel("P(HER2+)"); ax[1].set_ylabel("predictive entropy")
ax[1].set_title("Uncertainty by IHC group (test)"); ax[1].legend(fontsize=6.5)
savefig(fig, "fig07_uncertainty")

## 11. Localisation without reference masks: one grounded measure and several proxies
* **Grounded:** the instance false-positive rate in **test IHC-0 cores**. A pathologist found no 3+-type staining there, so any tile with high
  positivity is an error. Attention-only models (ABMIL, TransMIL) give no positivity score, so this does not apply to them. That gap is itself a
  finding.
* **Proxy: stability.** The mean pairwise Spearman correlation of tile scores across the 5 fold models, and the Jaccard overlap of the top-10% tiles.
* **Proxy: faithfulness to the model.** Remove the top-scored tiles and measure the drop in P(HER2+), compared with removing random tiles.
  This shows the map reflects the model's reasoning. It does **not** show the map is biologically correct.
* **Proxy: stain concordance.** The Spearman correlation between tile positivity and membranous DAB at moderate or higher, within positive cores.
  It is shown for SANC *without* the stain features, to avoid circularity.

None of these establishes spatial accuracy.

In [ ]:
def per_fold_tile_scores(res, kind="inst"):
    """res: a CV result dict (or a variant name in MIL_RES). Returns list over test bags of array (folds, n_tiles)."""
    if isinstance(res, str): res = MIL_RES[res]
    runs = res["test"]
    if runs[0][0][kind] is None: return None
    return [np.stack([runs[k][b][kind] for k in range(len(runs))]) for b in range(len(TE))]

neg0 = np.where(TE.ihc_score.values == 0)[0]; posb = np.where(TE.her2_status.values == 1)[0]
ihc3 = np.where(TE.ihc_score.values == 3)[0]
rng = np.random.default_rng(CFG.SEED)

def fold_stability(s_, min_tiles=5):
    """Mean pairwise Spearman rho and top-10% Jaccard between the fold models' tile scores of one bag.
    Written without scipy.stats.spearmanr on a matrix: some SciPy versions return a single NaN for the whole matrix when any
    input is constant or non-finite, which crashed the v2.1 run on Kaggle. Non-finite tiles are dropped; constant folds -> NaN."""
    s_ = np.asarray(s_, np.float64)
    if s_.ndim != 2 or s_.shape[0] < 2: return np.nan, np.nan
    s_ = s_[:, np.isfinite(s_).all(0)]
    if s_.shape[1] < min_tiles: return np.nan, np.nan
    R = stats.rankdata(s_, axis=1); R = R - R.mean(1, keepdims=True); nr = np.sqrt((R ** 2).sum(1))
    iu = np.triu_indices(len(R), 1); num = (R @ R.T)[iu]; den = (nr[:, None] * nr[None, :])[iu]
    rho = np.nanmean(np.where(den > 0, num / np.where(den > 0, den, 1), np.nan)) if (den > 0).any() else np.nan
    k = max(1, int(0.1 * s_.shape[1])); tops = [set(np.argsort(-x, kind="stable")[:k]) for x in s_]
    jac = np.mean([len(a & b) / len(a | b) for i, a in enumerate(tops) for b in tops[i + 1:]])
    return float(rho), float(jac)

def _fin(x): x = np.asarray(x, np.float64); return x[np.isfinite(x)]

def loc_metrics(res, name):
    """Grounded check (IHC-0 tile false positives), grounded-direction check (IHC 3+ detection), and proxies."""
    inst = per_fold_tile_scores(res, "inst"); attn = per_fold_tile_scores(res, "attn")
    sc = inst if inst is not None else attn
    row = {"method": name, "score_type": "positivity" if inst is not None else ("attention" if attn is not None else "none")}
    if sc is None: return row
    row["nonfinite_tile_scores"] = int(sum((~np.isfinite(s_)).sum() for s_ in sc))   # should be 0; reported, never hidden
    rhos, jac = [], []
    for s_ in sc:
        r_, j_ = fold_stability(s_)
        rhos.append(r_); jac.append(j_)
    row["fold_stability_spearman"] = float(np.nanmean(rhos)) if np.isfinite(rhos).any() else np.nan
    row["top10pct_jaccard"] = float(np.nanmean(jac)) if np.isfinite(jac).any() else np.nan
    if inst is not None:
        P = [sig(x.mean(0)) for x in inst]
        row["IHC0_tile_FP_rate@0.5"] = float(np.mean(np.concatenate([P[b] > 0.5 for b in neg0]))) if len(neg0) else np.nan
        if len(neg0):   # core-level bootstrap CI of the pooled tile FP rate (cores, not tiles, are the independent units)
            fp_n = np.array([np.sum(P[b] > 0.5) for b in neg0]); tl_n = np.array([len(P[b]) for b in neg0]); rb = np.random.default_rng(0)
            bs = [fp_n[i].sum() / max(tl_n[i].sum(), 1) for i in (rb.integers(0, len(neg0), len(neg0)) for _ in range(1000))]
            row["IHC0_tile_FP_rate@0.5_lo"], row["IHC0_tile_FP_rate@0.5_hi"] = float(np.quantile(bs, 0.025)), float(np.quantile(bs, 0.975))
        oof_neg = [j for j in np.where((TR.ihc_score.values == 0))[0] if res["oof"][j] is not None and res["oof"][j].get("inst") is not None]
        if oof_neg:   # the same grounded check on ~1,300 out-of-fold IHC-0 TRAINING cores (scored by the fold model that never saw them)
            row["OOF_IHC0_tile_FP_rate@0.5"] = float(np.mean(np.concatenate([sig(res["oof"][j]["inst"]) > 0.5 for j in oof_neg])))
            row["OOF_IHC0_cores_with_any_tile>0.9"] = float(np.mean([np.any(sig(res["oof"][j]["inst"]) > 0.9) for j in oof_neg]))
        row["IHC0_tile_FP_rate@0.9"] = float(np.mean(np.concatenate([P[b] > 0.9 for b in neg0]))) if len(neg0) else np.nan
        row["IHC0_cores_with_any_tile>0.9"] = float(np.mean([np.any(P[b] > 0.9) for b in neg0])) if len(neg0) else np.nan
        row["IHC3_cores_detected(any tile>0.5)"] = float(np.mean([np.any(P[b] > 0.5) for b in ihc3])) if len(ihc3) else np.nan
        row["IHC3_median_pos_tile_frac"] = float(np.median([np.mean(P[b] > 0.5) for b in ihc3])) if len(ihc3) else np.nan
        if len(ihc3) and len(neg0):
            t3 = _fin(np.concatenate([P[b] for b in ihc3])); t0 = _fin(np.concatenate([P[b] for b in neg0]))
            row["tile_AUROC_IHC3_vs_IHC0"] = float(roc_auc_score(np.r_[np.ones(len(t3)), np.zeros(len(t0))], np.r_[t3, t0])) if len(t3) and len(t0) else np.nan
        conc = []
        for b in posb:
            mm = BAGS[TE.image_id[b]]["hand_raw"][:, HAND_NAMES.index("mem_mod")]
            v = inst[b].mean(0); ok = np.isfinite(v) & np.isfinite(mm)
            if ok.sum() > 5 and mm[ok].std() > 0 and v[ok].std() > 0: conc.append(stats.spearmanr(v[ok], mm[ok]).correlation)
        row["stain_concordance_rho(pos cores)"] = float(np.nanmean(conc)) if conc else np.nan
    return row

LOC_ROWS = []
for n, r in {**MIL_RES, **CAM_RES, **ENC_RES}.items():
    try: LOC_ROWS.append(loc_metrics(r, n))
    except Exception as e:      # one failing method must not stop the run; the failure is recorded in the table
        print(f"!! localisation metrics failed for {n}: {type(e).__name__}: {e}"); LOC_ROWS.append({"method": n, "score_type": f"ERROR: {type(e).__name__}"})
LOC = pd.DataFrame(LOC_ROWS)
_bad = LOC[LOC.get("nonfinite_tile_scores", pd.Series(0, index=LOC.index)).fillna(0) > 0]
if len(_bad): print("!! non-finite tile scores (excluded from rank metrics):\n", _bad[["method", "nonfinite_tile_scores"]].to_string(index=False))

# deletion faithfulness (model-centric) for the proposed model and ABMIL
@torch.no_grad()
def deletion_curve(name, frac_list=(0.0, 0.05, 0.1, 0.2, 0.3)):
    T = CLS[CLS.method == name].temp.iloc[0]
    models = fold_models(name)
    sc = per_fold_tile_scores(MIL_RES[name], "inst")
    sc = sc if sc is not None else per_fold_tile_scores(MIL_RES[name], "attn")
    top_drop, rnd_drop = [], []
    for b in posb:
        s = sc[b].mean(0); n = len(s); bp = TEP[b]
        e_full, h_full, _, _ = STORE.batch([bp])
        def prob(keep_idx):
            e, h = e_full[:, keep_idx], h_full[:, keep_idx]; mk = torch.ones(1, len(keep_idx), dtype=torch.bool, device=DEVICE)
            return float(np.mean([sig(m(e, h, mk)["logit"].item() / T) for m in models]))
        base = prob(np.arange(n)); t_row, r_row = [], []
        for f in frac_list[1:]:
            k = max(1, int(f * n)); order = np.argsort(-s)
            t_row.append(base - prob(np.sort(order[k:]))) if n - k > 0 else t_row.append(np.nan)
            r_row.append(np.mean([base - prob(np.sort(rng.choice(n, n - k, replace=False))) for _ in range(5)]) if n - k > 0 else np.nan)
        top_drop.append(t_row); rnd_drop.append(r_row)
    for m in models: del m
    return np.array(frac_list[1:]), np.nanmean(top_drop, 0), np.nanmean(rnd_drop, 0)
DEL_PATH = CACHE / "deletion_curves.pkl"
DEL = load_pickle(DEL_PATH) if DEL_PATH.exists() else None
if DEL is None:
    DEL = {name: deletion_curve(name) for name in [PROPOSED, MAIN_BASELINE] if len(posb)}
    atomic_pickle(DEL, DEL_PATH)
for name, (fr, td, rd) in DEL.items():
    LOC.loc[LOC.method == name, "deletion_drop@10%_top_minus_random"] = float(np.interp(0.1, fr, td) - np.interp(0.1, fr, rd))
LOC.to_csv(TAB / "localization_proxies.csv", index=False); print(LOC.round(3).to_string(index=False))

# ---------------- variability over repeated CV seeds (the claims must survive re-training) ----------------
SEED_ROWS = []
for n, runs in SEED_RES.items():
    for sd, res in enumerate(runs):
        te_logit = np.mean([[d["logit"] for d in run] for run in res["test"]], 0)
        r = loc_metrics(res, n); r["seed"] = sd; r["test_AUROC"] = roc_auc_score(ystat_te, te_logit)
        r["OOF_AUROC"] = roc_auc_score(ystat_tr, [d["logit"] for d in res["oof"]]); SEED_ROWS.append(r)
SEEDS = pd.DataFrame(SEED_ROWS)
_keys = ["test_AUROC", "OOF_AUROC", "IHC0_tile_FP_rate@0.5", "OOF_IHC0_tile_FP_rate@0.5", "IHC0_cores_with_any_tile>0.9", "IHC3_cores_detected(any tile>0.5)",
         "tile_AUROC_IHC3_vs_IHC0", "fold_stability_spearman"]
_keys = [k for k in _keys if k in SEEDS]
SEED_SUMMARY = SEEDS.groupby("method")[_keys].agg(["mean", "std"]).round(4)
SEEDS.to_csv(TAB / "seed_runs.csv", index=False); SEED_SUMMARY.to_csv(TAB / "seed_variability.csv")
print(f"Mean ± SD over {CFG.N_SEEDS} CV seeds:\n", SEED_SUMMARY.to_string())

# ---------------- H1: does the negative-bag constraint reduce false-positive evidence in IHC-0 cores? ----------------
# Paired per-core comparison (same cores, same folds): Wilcoxon signed-rank test + bootstrap 95% CI of the mean difference,
# on the independent test set AND on out-of-fold IHC-0 training cores, repeated for every CV seed.
def core_fp(res, split, thr=0.5):
    if split == "test":
        sc = per_fold_tile_scores(res, "inst")
        return None if sc is None else np.array([np.mean(sig(sc[b].mean(0)) > thr) for b in neg0])
    idx = np.where(TR.ihc_score.values == 0)[0]
    if res["oof"][idx[0]].get("inst") is None: return None
    return np.array([np.mean(sig(res["oof"][j]["inst"]) > thr) for j in idx])
def paired_test(a, b, n_boot=2000):
    d = a - b; rb = np.random.default_rng(1)
    bs = [d[rb.integers(0, len(d), len(d))].mean() for _ in range(n_boot)]
    try: p = stats.wilcoxon(a, b, zero_method="zsplit").pvalue
    except ValueError: p = 1.0
    return dict(n_cores=len(d), mean_A=a.mean(), mean_B=b.mean(), diff=d.mean(), lo=np.quantile(bs, 0.025), hi=np.quantile(bs, 0.975),
                wilcoxon_p=p, cores_A_better=int((d < 0).sum()), cores_B_better=int((d > 0).sum()))
H1_PAIRS = [(PROPOSED, "SANC − negative constraint"), ("SANC − stain features", "SANC − negative constraint"), (PROPOSED, "MaxPool")]
H1_ROWS = []
for A, B in H1_PAIRS:
    for sd in range(min(len(SEED_RES.get(A, [])), len(SEED_RES.get(B, [])))):
        for split in ("test", "oof"):
            a, b = core_fp(SEED_RES[A][sd], split), core_fp(SEED_RES[B][sd], split)
            if a is not None and b is not None: H1_ROWS.append(dict(A=A, B=B, seed=sd, split=split, **paired_test(a, b)))
if CAM_RES:
    a, b = core_fp(MIL_RES[PROPOSED], "test"), core_fp(CAM_RES["Grad-CAM CNN (per-image normalised CAM)"], "test")
    H1_ROWS.append(dict(A=PROPOSED, B="Grad-CAM CNN (per-image normalised CAM)", seed=0, split="test", **paired_test(a, b)))
H1 = pd.DataFrame(H1_ROWS); atomic_csv(H1, TAB / "h1_negative_constraint_tests.csv", index=False)
print("H1 paired per-core tests (FP fraction of tiles per IHC-0 core; A − B < 0 favours A):\n", H1.round(4).to_string(index=False))

fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
_show = [m for m in [MAIN_BASELINE, "MaxPool", "SANC − negative constraint", "SANC − stain features", PROPOSED] if m in SEEDS.method.unique()]
for j, (col, ttl) in enumerate([("test_AUROC", "Test AUROC (status)"), ("IHC0_tile_FP_rate@0.5", "IHC-0 tile FP rate, test"),
                                ("OOF_IHC0_tile_FP_rate@0.5", "IHC-0 tile FP rate, OOF train")]):
    if col not in SEEDS: ax[j].axis("off"); continue
    for i, m in enumerate(_show):
        v = SEEDS[SEEDS.method == m][col].dropna().values
        if len(v): ax[j].errorbar(v.mean(), i, xerr=v.std() if len(v) > 1 else 0, fmt="o", color=PAL[i], ms=6, capsize=3)
        ax[j].scatter(v, np.full(len(v), i), s=10, color=PAL[i], alpha=0.5)
    ax[j].set_yticks(range(len(_show))); ax[j].set_yticklabels(_show if j == 0 else [], fontsize=8); ax[j].set_title(ttl)
fig.suptitle(f"Effect of the negative-bag constraint (mean ± SD over {CFG.N_SEEDS} CV seeds; dots = individual seeds)", fontsize=10)
savefig(fig, "fig11_negative_constraint_seeds")
section_time("localisation analysis")

In [ ]:
# ---------------- heat-map figure: attention ≠ positivity; uncertainty shown explicitly ----------------
def tile_grid_map(iid, values, shape_lr):
    T, R = CFG.TILE, CFG.THUMB_REL; m = np.full(shape_lr, np.nan, np.float32)
    for (y, x), v in zip(BAGS[iid]["coords"], values):
        m[y // R:(y + T) // R, x // R:(x + T) // R] = v
    return m
def heatmap_figure(n_per=1):
    picks = []
    for g in ["0", "2-", "2+", "3"]:
        s = TE[TE.group == g]
        if len(s): picks += [s.index[i] for i in rng.choice(len(s), min(n_per, len(s)), replace=False)]
    inst_s = per_fold_tile_scores(PROPOSED, "inst"); att_a = per_fold_tile_scores(MAIN_BASELINE, "attn")
    cam_n = per_fold_tile_scores(CAM_RES["Grad-CAM CNN (per-image normalised CAM)"], "inst") if CAM_RES else None
    cols = ["core", "DAB OD", "membranous DAB ≥ moderate", "Grad-CAM (per-image norm.)", "ABMIL attention", "SANC P(positive tile)", "SANC uncertainty (fold SD)"]
    fig, axes = plt.subplots(len(picks), len(cols), figsize=(2.3 * len(cols), 2.35 * len(picks)))
    axes = np.atleast_2d(axes)
    for r, b in enumerate(picks):
        iid = TE.image_id[b]; mp = np.load(MAP_DIR / f"{iid}.npz"); th = cv2.cvtColor(cv2.imread(str(THUMB_DIR / f"{iid}.jpg")), cv2.COLOR_BGR2RGB)
        shp = mp["dab"].shape
        tmask = cv2.resize(mp["tissue"].astype(np.uint8), (shp[1], shp[0]), interpolation=cv2.INTER_NEAREST) > 0
        nan_out = lambda a: np.where(tmask, a.astype(np.float32), np.nan)
        pr = sig(inst_s[b]); att = att_a[b].mean(0); att = att / (att.max() + 1e-9)
        camv = tile_grid_map(iid, sig(cam_n[b]).mean(0), shp) if cam_n is not None else np.full(shp, np.nan, np.float32)
        panels = [(th, None), (nan_out(mp["dab"]), "magma"), (nan_out(mp["mem_mod"]), "magma"), (camv, "viridis"),
                  (tile_grid_map(iid, att, shp), "viridis"), (tile_grid_map(iid, pr.mean(0), shp), "RdBu_r"), (tile_grid_map(iid, pr.std(0), shp), "Greys")]
        for c, (img, cm) in enumerate(panels):
            ax = axes[r, c]; ax.axis("off")
            if cm is None: ax.imshow(img)
            else:
                ax.imshow(th, alpha=0.35)
                kw = dict(vmin=0, vmax=1) if c in (3, 4, 5) else (dict(vmin=0, vmax=0.35) if c == 6 else {})
                if c == 1: kw = dict(vmin=0, vmax=CALIB["t_strong"] * 1.5)
                if c == 2: kw = dict(vmin=0, vmax=max(0.05, float(np.nanpercentile(img, 99.5))))
                ax.imshow(cv2.resize(img, (th.shape[1], th.shape[0]), interpolation=cv2.INTER_NEAREST), cmap=cm, alpha=0.75, **kw)
            if r == 0: ax.set_title(cols[c], fontsize=8)
        axes[r, 0].text(5, 25, f"IHC {TE.group[b]}  P={pu['pm'][b]:.2f}", color="k", fontsize=7, bbox=dict(fc="w", ec="none", alpha=0.8))
    savefig(fig, "fig08_localization_maps")
heatmap_figure()

## 12. Pseudo-masks with explicit ignore regions and confidence (train cores only)
There are two **separate** targets, so tumour or evidence localisation is never conflated with membrane morphology.

**Membrane target**, driven by stain physics and morphology:

| Pixel condition | Label | Confidence |
|---|---|---|
| Membrane-shaped (line-like, locally prominent), non-nuclear, DAB ≥ `t_weak` | 1 | 0.5–1.0, rising with DAB OD |
| 1-px rim around each membrane (ambiguous width) | **ignore** | – |
| DAB < `t_bg`, or background | 0 | 1.0 |
| Non-membranous (cytoplasmic/blob) or nuclear DAB | 0 | 0.5 |
| Faint non-linear stain | 0 | 0.5 |
| Rim band, sub-threshold ridges, or ambiguous DAB | **ignore** (255) | – |

**Evidence target**, driven by the image labels via **out-of-fold** SANC tile positivity (each core is scored by the model that never saw it):

| Core group | Tiles | Label | Confidence |
|---|---|---|---|
| IHC 0/1+ | all | 0 | 1.0 (exact negative bag) |
| HER2+ | P ≥ 0.8 | 1 | P |
| HER2+ | P ≤ 0.2 | 0 | 0.5 |
| HER2+ | otherwise | ignore | – |
| 2+/FISH− | P ≤ 0.2 | 0 | 0.5 |
| 2+/FISH− | otherwise | ignore | – |

These are training signals, **not annotations**. They are never used as ground truth for accuracy.

In [ ]:
CROP_DIR = CACHE / "seg_crops"
CROP_INDEX = CACHE / "seg_crops_index.csv"
OOF_INST = {TR.image_id[j]: sig(d["inst"]) for j, d in enumerate(MIL_RES[PROPOSED]["oof"])}

@torch.no_grad()
def pseudo_labels(mp, iid, row):
    T = CFG.TILE; tis = mp["tissue"][0, 0]; rim = mp["rim"][0, 0]; dab = mp["DAB"][0, 0]
    H, W = tis.shape
    lab = torch.full((H, W), 255, dtype=torch.uint8, device=DEVICE); conf = torch.zeros((H, W), dtype=torch.uint8, device=DEVICE)
    def put(m, l, c):
        lab[m] = l; conf[m] = c
    put(~tis, 0, 255)
    put(tis & (dab < CALIB["t_bg"]), 0, 255)
    put(tis & ~mp["mem_shape"][0, 0] & (dab >= CALIB["t_bg"]) & (dab < CALIB["t_weak"]), 0, 128)
    put(tis & (dab >= CALIB["t_weak"]) & (mp["nonmem"][0, 0] | mp["nuc"][0, 0]), 0, 128)   # cytoplasmic / nuclear DAB
    pos = mp["mem_weak"][0, 0]
    ring = dilate(pos[None, None], 1)[0, 0] & ~pos                     # 1-px rim of each membrane: ambiguous width -> ignore
    lab[ring & tis] = 255; conf[ring & tis] = 0
    c = 0.5 + 0.5 * ((torch.log(dab.clamp(min=1e-6)) - math.log(CALIB["t_weak"])) / (math.log(CALIB["t_strong"]) - math.log(CALIB["t_weak"]))).clamp(0, 1)
    lab[pos] = 1; conf[pos] = (c[pos] * 255).to(torch.uint8)
    lab[rim] = 255; conf[rim] = 0
    # evidence target
    elab = torch.full((H, W), 255, dtype=torch.uint8, device=DEVICE); econf = torch.zeros((H, W), dtype=torch.uint8, device=DEVICE)
    elab[~tis] = 0; econf[~tis] = 76
    p = OOF_INST.get(iid); coords = BAGS[iid]["coords"]
    neg_bag = row.ihc_score <= 1 and row.her2_status == 0
    for (y, x), pv in zip(coords, p if p is not None else np.full(len(coords), 0.5)):
        sl = (slice(y, y + T), slice(x, x + T))
        if neg_bag: l, cc = 0, 255
        elif row.her2_status == 1 and pv >= CFG.EV_P_POS: l, cc = 1, int(255 * pv)
        elif pv <= CFG.EV_P_NEG: l, cc = 0, 128
        else: l, cc = 255, 0
        elab[sl] = l; econf[sl] = cc
    elab[~tis & (elab == 255)] = 0
    return torch.stack([lab, conf, elab, econf], -1)

def sample_centers(mp, lab, K, rng):
    S = CFG.SEG_CROP; tis = mp["tissue"][0, 0]; H, W = tis.shape
    def pick(mask, n):
        idx = torch.nonzero(mask)
        if len(idx) == 0 or n <= 0: return []
        sel = idx[torch.from_numpy(rng.choice(len(idx), min(n, len(idx)), replace=False)).to(idx.device)].cpu().numpy()
        return [tuple(v) for v in sel]
    k_mem, k_ev = K // 2, K // 4
    cs = pick(lab[..., 0] == 1, k_mem) + pick(lab[..., 2] == 1, k_ev)
    cs += pick(tis & ~mp["rim"][0, 0], K - len(cs))
    return [(int(np.clip(y - S // 2, 0, H - S)), int(np.clip(x - S // 2, 0, W - S))) for y, x in cs]

def _img_seed(iid): return int(hashlib.md5(iid.encode()).hexdigest()[:8], 16) + CFG.SEED   # per-core RNG: order-independent, resumable

if CROP_INDEX.exists():
    CROPS = pd.read_csv(CROP_INDEX)
else:
    plog = ProgressLog("seg_crops")                       # one line per finished core -> restart skips finished cores
    todo_tr = TR[[i not in plog for i in TR.image_id]].reset_index(drop=True)
    print(f"pseudo-label crops: {len(plog)} cores done earlier, {len(todo_tr)} to go")
    if len(todo_tr):
        dl = torch.utils.data.DataLoader(DecodeDS(todo_tr), batch_size=1, num_workers=CFG.NUM_WORKERS, collate_fn=lambda b: b[0])
        for i, work in tqdm(dl, total=len(todo_tr), desc="pseudo-label crops"):
            r = todo_tr.iloc[i]; iid = r.image_id; S = CFG.SEG_CROP; recs = []
            if work is not None and work.shape[0] >= S and work.shape[1] >= S:
                mz = np.load(MAP_DIR / f"{iid}.npz")
                mp = ENGINE.maps(work, mz["I0"], mz["tissue"], mz["rim"])
                lab = pseudo_labels(mp, iid, r)
                for k, (y, x) in enumerate(sample_centers(mp, lab, CFG.CROPS_PER_IMAGE, np.random.default_rng(_img_seed(iid)))):
                    cp = CROP_DIR / f"{iid}_{k}.jpg"; lp = CROP_DIR / f"{iid}_{k}.png"
                    cv2.imwrite(str(cp), cv2.cvtColor(work[y:y + S, x:x + S], cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 92])
                    cv2.imwrite(str(lp), lab[y:y + S, x:x + S].cpu().numpy())
                    recs.append(dict(image_id=iid, crop=str(cp), label=str(lp), y=int(y), x=int(x), fold=int(r.fold), group=r.group))
                del mp, lab
            plog.add(iid, dict(crops=recs))                   # logged only after all of this core's crops are on disk
    CROPS = pd.DataFrame([c for rec in plog.records() for c in rec["crops"]])
    atomic_csv(CROPS, CROP_INDEX, index=False)
print(f"{len(CROPS)} pseudo-labelled crops from {CROPS.image_id.nunique()} train cores")

def pseudo_label_figure(n=4):
    s = CROPS.groupby("group").head(1).head(n)
    fig, axes = plt.subplots(len(s), 4, figsize=(9.5, 2.4 * len(s))); axes = np.atleast_2d(axes)
    for ax, (_, r) in zip(axes, s.iterrows()):
        img = cv2.cvtColor(cv2.imread(r.crop), cv2.COLOR_BGR2RGB); lab = cv2.imread(r.label, cv2.IMREAD_UNCHANGED)
        def show_lab(a, l, c, title):
            v = np.zeros(l.shape + (3,), np.uint8) + 255
            v[l == 0] = (230, 230, 230); v[l == 1] = (227, 73, 72); v[l == 255] = (237, 161, 0)
            a.imshow(v); a.set_title(title, fontsize=8)
        ax[0].imshow(img); ax[0].set_title(f"{r.image_id[:20]} ({r.group})", fontsize=8)
        show_lab(ax[1], lab[..., 0], lab[..., 1], "membrane: 1 red / 0 grey / ignore amber")
        ax[2].imshow(lab[..., 1], cmap="Greys_r", vmin=0, vmax=255); ax[2].set_title("membrane confidence", fontsize=8)
        show_lab(ax[3], lab[..., 2], lab[..., 3], "evidence (OOF SANC)")
        for a in ax: a.axis("off")
    savefig(fig, "fig09_pseudo_labels")
pseudo_label_figure()

## 13. Segmentation student: MSA-U-Net-TR versus U-Net, and loss ablations
The architecture keeps the original idea, now implemented correctly:
* a full **decoder** that outputs at full work resolution, with a full-resolution stem so thin membranes are not lost
* attention-gated skip connections (Oktay 2018)
* a 2-layer transformer at the stride-32 bottleneck
* two heads: **membrane** and **evidence region**

Training uses confidence-weighted BCE plus soft Dice over **non-ignored** pixels only. Augmentation is GPU stain augmentation (HED jitter),
flips and rotations, and mild blur and contrast changes.

Ablations, all at the same epoch budget:
* a plain U-Net
* *naive* hard labels, where ignore is treated as 0 and confidence is ignored
* ignore regions without confidence weights

The held-out fold 0 crops measure **agreement with pseudo-labels**. That agreement is not accuracy.

In [ ]:
class SegCropDS(torch.utils.data.Dataset):
    def __init__(self, df): self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = cv2.cvtColor(cv2.imread(r.crop), cv2.COLOR_BGR2RGB); lab = cv2.imread(r.label, cv2.IMREAD_UNCHANGED)
        return torch.from_numpy(img), torch.from_numpy(lab)

M_STAIN = torch.tensor(CALIB["M"], dtype=torch.float32, device=DEVICE); M_STAIN_INV = torch.linalg.pinv(M_STAIN)
def hed_jitter(x, sa=0.05, sb=0.02, gen=None):
    """x: (B,3,H,W) float in [0,1]; Tellez-style stain augmentation in the calibrated stain space."""
    B = x.shape[0]
    od = -torch.log(x.clamp(min=1 / 255.0))
    c = torch.einsum("bchw,cd->bdhw", od, M_STAIN_INV)
    a = 1 + (torch.rand(B, 3, 1, 1, device=x.device, generator=gen) * 2 - 1) * sa
    b = (torch.rand(B, 3, 1, 1, device=x.device, generator=gen) * 2 - 1) * sb
    od2 = torch.einsum("bdhw,dc->bchw", (c * a + b).clamp(min=0), M_STAIN)
    return torch.exp(-od2).clamp(0, 1)

def gpu_augment(img, lab):
    if random.random() < 0.5: img, lab = img.flip(-1), lab.flip(-1)
    if random.random() < 0.5: img, lab = img.flip(-2), lab.flip(-2)
    k = random.randint(0, 3); img, lab = torch.rot90(img, k, (-2, -1)), torch.rot90(lab, k, (-2, -1))
    img = hed_jitter(img)
    if random.random() < 0.5:
        img = ((img - 0.5) * (1 + (random.random() - 0.5) * 0.2) + 0.5 + (random.random() - 0.5) * 0.06).clamp(0, 1)
    if random.random() < 0.2:
        g, _, _ = _gauss_1d(random.uniform(0.4, 1.0), img.device)
        img = torch.stack([sep_conv(img[:, c:c + 1], g, g)[:, 0] for c in range(3)], 1)
    return img, lab

def cbr(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))
class AttnGate(nn.Module):
    def __init__(self, cg, cx, ci):
        super().__init__(); self.wg = nn.Conv2d(cg, ci, 1); self.wx = nn.Conv2d(cx, ci, 1); self.psi = nn.Conv2d(ci, 1, 1)
    def forward(self, g, x): return x * torch.sigmoid(self.psi(F.relu(self.wg(g) + self.wx(x))))
class BottleneckTR(nn.Module):
    def __init__(self, c, d=256, layers=2):
        super().__init__(); self.inp = nn.Conv2d(c, d, 1); self.pos = nn.Conv2d(d, d, 3, padding=1, groups=d)
        self.tf = nn.TransformerEncoder(nn.TransformerEncoderLayer(d, 8, 2 * d, 0.1, batch_first=True, norm_first=True), layers)
        self.out = nn.Conv2d(d, c, 1)
    def forward(self, x):
        y = self.inp(x); y = y + self.pos(y); B, C, H, W = y.shape
        y = self.tf(y.flatten(2).transpose(1, 2)).transpose(1, 2).reshape(B, C, H, W)
        return x + self.out(y)
class SegNet(nn.Module):
    def __init__(self, enc_name, gates=True, tr=True):
        super().__init__()
        try: self.enc = timm.create_model(enc_name, pretrained=True, features_only=True)
        except Exception:
            print("!! segmentation encoder: pretrained weights unavailable, using random init"); self.enc = timm.create_model(enc_name, pretrained=False, features_only=True)
        ch = self.enc.feature_info.channels(); red = self.enc.feature_info.reduction()
        assert red[0] == 2 and len(ch) == 5, f"expects 5 encoder stages starting at stride 2, got {red}"
        self.tr = BottleneckTR(ch[-1]) if tr else nn.Identity()
        dec = [256, 128, 64, 32]; self.gates = nn.ModuleList(); self.blocks = nn.ModuleList(); prev = ch[-1]
        for i, d in enumerate(dec):
            skip = ch[-2 - i]
            self.gates.append(AttnGate(prev, skip, max(16, skip // 2)) if gates else nn.Identity())
            self.blocks.append(nn.Sequential(cbr(prev + skip, d), cbr(d, d))); prev = d
        self.stem = nn.Sequential(cbr(3, 16), cbr(16, 16))
        self.final = nn.Sequential(cbr(prev + 16, 32), nn.Dropout2d(0.1))
        self.head_mem = nn.Conv2d(32, 1, 1); self.head_ev = nn.Conv2d(32, 1, 1)
        self.register_buffer("mean", torch.tensor(IMNET_MEAN).view(1, 3, 1, 1)); self.register_buffer("std", torch.tensor(IMNET_STD).view(1, 3, 1, 1))
        self.use_gates = gates
    def forward(self, x):
        xn = (x - self.mean) / self.std; f = self.enc(xn); y = self.tr(f[-1])
        for i, blk in enumerate(self.blocks):
            s = f[-2 - i]; y = F.interpolate(y, size=s.shape[-2:], mode="bilinear", align_corners=False)
            s = self.gates[i](y, s) if self.use_gates else s
            y = blk(torch.cat([y, s], 1))
        y = F.interpolate(y, size=x.shape[-2:], mode="bilinear", align_corners=False)
        y = self.final(torch.cat([y, self.stem(x)], 1))
        return self.head_mem(y)[:, 0], self.head_ev(y)[:, 0]

def seg_loss(lm, le, lab, mode):
    ml, mc, el, ec = lab[..., 0], lab[..., 1].float() / 255, lab[..., 2], lab[..., 3].float() / 255
    def weights(l, c):
        valid = (l != 255).float()
        if mode == "naive": return torch.ones_like(valid), (l == 1).float()
        if mode == "ignore": return valid, (l == 1).float()
        return valid * c, (l == 1).float()
    wm, tm = weights(ml, mc); we, te = weights(el, ec)
    bce_m = (wm * F.binary_cross_entropy_with_logits(lm, tm, reduction="none")).sum() / wm.sum().clamp(min=1)
    p = torch.sigmoid(lm); dice = 1 - (2 * (wm * p * tm).sum() + 1) / ((wm * p).sum() + (wm * tm).sum() + 1)
    bce_e = (we * F.binary_cross_entropy_with_logits(le, te, reduction="none")).sum() / we.sum().clamp(min=1)
    return bce_m + dice + CFG.W_EV * bce_e

@torch.no_grad()
def seg_agreement(model, dl):
    model.eval(); tp = fp = fn = 0; tph = fph = fnh = 0; ev_s, ev_y = [], []
    for img, lab in dl:
        img = img.to(DEVICE).permute(0, 3, 1, 2).float() / 255; lab = lab.to(DEVICE)
        with autocast(): lm, le = model(img)
        pm = torch.sigmoid(lm.float()) >= 0.5; ml, mc = lab[..., 0], lab[..., 1]
        v = ml != 255; t = ml == 1; hc = v & (mc >= 191)
        tp += (pm & t & v).sum().item(); fp += (pm & ~t & v).sum().item(); fn += (~pm & t & v).sum().item()
        tph += (pm & t & hc).sum().item(); fph += (pm & ~t & hc).sum().item(); fnh += (~pm & t & hc).sum().item()
        el = lab[..., 2]; ve = el != 255
        if ve.any():
            s = torch.sigmoid(le.float())[ve]; y = (el[ve] == 1)
            k = torch.randperm(len(s), device=s.device)[:20000]; ev_s.append(s[k].cpu().numpy()); ev_y.append(y[k].cpu().numpy())
    d = lambda a, b, c: 2 * a / max(2 * a + b + c, 1)
    ev_s = np.concatenate(ev_s) if ev_s else np.array([]); ev_y = np.concatenate(ev_y) if ev_y else np.array([])
    return dict(agree_Dice=d(tp, fp, fn), agree_IoU=tp / max(tp + fp + fn, 1), agree_precision=tp / max(tp + fp, 1), agree_recall=tp / max(tp + fn, 1),
                agree_Dice_highconf=d(tph, fph, fnh), evidence_AUROC_vs_pseudo=roc_auc_score(ev_y, ev_s) if len(set(ev_y.tolist())) > 1 else np.nan)

cv2.setNumThreads(0)   # OpenCV's own thread pool inside forked DataLoader workers can dead-lock -> disable it

def _seg_loader(df, shuffle, workers):
    kw = dict(batch_size=CFG.SEG_BS, shuffle=shuffle, drop_last=shuffle, num_workers=workers)
    if workers > 0:   # a frozen worker raises after 180 s instead of hanging forever
        kw.update(timeout=180, worker_init_fn=lambda _: cv2.setNumThreads(0), persistent_workers=False)
    return torch.utils.data.DataLoader(SegCropDS(df), **kw)

# Epoch selection (v2.1.2). v2.1 kept the epoch with the best agreement on HIGH-CONFIDENCE pixels only. Those are the easiest
# pixels, so an under-trained epoch 1 could win (it did on Kaggle: epoch 1 of 10). The rule is now: best agreement over ALL
# labelled held-out pixels (rounded to 3 decimals), with high-confidence agreement only as a tie-break. A model trained under the
# old rule is reused only if the new rule picks the same epoch; otherwise it is retrained (deterministically) once.
SEL_RULE = "overall-agreement-v2"
def _sel_key(h): return (round(float(h["agree_Dice"]), 3), float(h["agree_Dice_highconf"]))
def _old_key(h): return float(h["agree_Dice_highconf"])

def train_seg(name, gates, tr, mode, epochs, snapshot_at=None):
    """Checkpoints every epoch (resumes after any interruption); keeps the epoch with the best overall pseudo-label
    agreement on held-out fold-0 crops (tie-break: high-confidence agreement); single-process loading."""
    path = OUT / "models" / f"seg_{slug(name)}.pt"; mpath = CACHE / f"seg_{slug(name)}_metrics.json"
    ckpt_path = CACHE / f"seg_{slug(name)}_ckpt.pt"
    seed_everything(_img_seed(name))                         # same initial weights whether or not earlier stages were cached
    model = SegNet(CFG.SEG_ENCODER, gates, tr).to(DEVICE)
    best_path = OUT / "models" / f"seg_{slug(name)}_best.pt"
    if path.exists() and mpath.exists():
        res = json.load(open(mpath))
        if res.get("selection") != SEL_RULE:                       # trained under the v2.1 rule
            new_ep = max(res["history"], key=_sel_key)["epoch"]
            if new_ep == res["final"]["epoch"]:
                res["selection"] = SEL_RULE; save_json(res, mpath)
                print(f"  {name[:40]}: new selection rule picks the same epoch ({new_ep}); trained model reused")
            else:
                print(f"  {name[:40]}: new selection rule picks epoch {new_ep}, not {res['final']['epoch']}; retraining once")
                res = None
                for f_ in (path, mpath, ckpt_path, best_path): f_.unlink(missing_ok=True)
        if res is not None:
            model.load_state_dict(torch.load(path, map_location=DEVICE)); return model.eval(), res
    tr_df, va_df = CROPS[CROPS.fold != 0], CROPS[CROPS.fold == 0]
    workers = 0   # single-process loading: the v1 run froze twice inside forked loader workers
    dl, dlv = _seg_loader(tr_df, True, workers), _seg_loader(va_df, False, workers)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.SEG_LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG.SEG_LR, total_steps=max(1, epochs * len(dl)), pct_start=0.1)
    scaler = (torch.amp.GradScaler("cuda") if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler") else torch.cuda.amp.GradScaler()) if DEVICE.type == "cuda" else torch.cuda.amp.GradScaler(enabled=False)
    hist, start, best = [], 0, (-1.0, -1.0)
    if ckpt_path.exists():
        ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        ok = True
        if ck.get("rule") != SEL_RULE and ck["hist"]:               # checkpoint written under the v2.1 rule
            old_ep = ck.get("best_epoch") or max(ck["hist"], key=_old_key)["epoch"]
            ok = max(ck["hist"], key=_sel_key)["epoch"] == old_ep   # saved best weights are valid only if both rules agree
        if ok:
            model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
            hist, start = ck["hist"], ck["epoch"]; best = _sel_key(max(hist, key=_sel_key)) if hist else (-1.0, -1.0)
            print(f"  resuming {name[:40]} from epoch {start + 1}/{epochs} (best overall agreement so far {best[0]:.3f})")
        else:
            print(f"  {name[:40]}: checkpoint used the old selection rule and the best epoch differs; restarting this model")
            ckpt_path.unlink(missing_ok=True); best_path.unlink(missing_ok=True)
            seed_everything(_img_seed(name)); model = SegNet(CFG.SEG_ENCODER, gates, tr).to(DEVICE)
            opt = torch.optim.AdamW(model.parameters(), lr=CFG.SEG_LR, weight_decay=1e-4)
            sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG.SEG_LR, total_steps=max(1, epochs * len(dl)), pct_start=0.1)
    ep = start
    while ep < epochs:
        seed_everything(_img_seed(name) + 1000 + ep)          # per-epoch seed: shuffling/augmentation identical after a resume
        model.train(); tl = []
        try:
            for img, lab in tqdm(dl, desc=f"{name[:28]} ep{ep + 1}/{epochs}", leave=False):
                img = img.to(DEVICE, non_blocking=True).permute(0, 3, 1, 2).float() / 255; lab = lab.to(DEVICE, non_blocking=True).permute(0, 3, 1, 2)
                img, lab = gpu_augment(img, lab); lab = lab.permute(0, 2, 3, 1)
                with autocast():
                    lm, le = model(img)
                loss = seg_loss(lm.float(), le.float(), lab, mode)
                opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update()
                if sched.last_epoch < sched.total_steps - 1: sched.step()
                tl.append(loss.item())
            m = seg_agreement(model, dlv)
        except RuntimeError as e:
            if workers == 0 or "DataLoader" not in str(e) and "timed out" not in str(e): raise
            print(f"  !! data-loader worker froze ({str(e)[:80]}); switching to single-process loading and repeating epoch {ep + 1}")
            workers = 0; dl, dlv = _seg_loader(tr_df, True, 0), _seg_loader(va_df, False, 0); continue
        m["epoch"] = ep + 1; m["train_loss"] = float(np.mean(tl)); hist.append(m)
        print(f"  {name[:40]} ep{ep + 1}: loss {m['train_loss']:.3f} | pseudo-agreement Dice {m['agree_Dice']:.3f} (high-conf {m['agree_Dice_highconf']:.3f}) | evidence AUROC {m['evidence_AUROC_vs_pseudo']:.3f}", flush=True)
        if _sel_key(m) > best:
            best = _sel_key(m); atomic_torch_save(model.state_dict(), best_path)
        ep += 1
        atomic_torch_save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), scaler=scaler.state_dict(),
                        hist=hist, epoch=ep, best=best, best_epoch=max(hist, key=_sel_key)["epoch"], rule=SEL_RULE), ckpt_path)
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    best_m = max(hist, key=_sel_key)
    print(f"  {name[:40]}: keeping epoch {best_m['epoch']} of {len(hist)} (overall agreement {best_m['agree_Dice']:.3f}, "
          f"high-conf {best_m['agree_Dice_highconf']:.3f})")
    atomic_torch_save(model.state_dict(), path)
    res = dict(final=best_m, selection=SEL_RULE, last_epoch=hist[-1], history=hist, same_budget=hist[min(len(hist), snapshot_at or len(hist)) - 1])
    save_json(res, mpath); ckpt_path.unlink(missing_ok=True)
    best_path.unlink(missing_ok=True)   # identical to the final file; saves ~100 MB per model
    return model.eval(), res

SEG_EXPS = {
    "MSA-U-Net-TR + ignore + confidence (proposed)": dict(gates=True, tr=True, mode="conf", epochs=CFG.SEG_EPOCHS),
    "U-Net + ignore + confidence": dict(gates=False, tr=False, mode="conf", epochs=CFG.SEG_ABL_EPOCHS),
    "MSA-U-Net-TR + naive hard labels": dict(gates=True, tr=True, mode="naive", epochs=CFG.SEG_ABL_EPOCHS),
    "MSA-U-Net-TR + ignore only": dict(gates=True, tr=True, mode="ignore", epochs=CFG.SEG_ABL_EPOCHS),
}
SEG_PROPOSED = "MSA-U-Net-TR + ignore + confidence (proposed)"
SEG_MODELS, SEG_METRICS = {}, {}
for name, e in SEG_EXPS.items():
    SEG_MODELS[name], SEG_METRICS[name] = train_seg(name, e["gates"], e["tr"], e["mode"], e["epochs"],
                                                    snapshot_at=CFG.SEG_ABL_EPOCHS if name == SEG_PROPOSED else None)
SEGT = pd.DataFrame([{"model": n, **{f"same-budget {k}": v for k, v in m["same_budget"].items() if k.startswith(("agree", "evidence"))},
                      "final_epochs": m["final"]["epoch"], "final agree_Dice": m["final"]["agree_Dice"]} for n, m in SEG_METRICS.items()])
SEGT.to_csv(TAB / "seg_pseudo_agreement.csv", index=False)
print("Held-out agreement with PSEUDO-labels (not accuracy):\n", SEGT.round(3).to_string(index=False))
section_time("segmentation students")

## 14. Cell-level membrane morphometry and core-level features (same protocol for train and test)
Each core contributes `FEAT_CROPS_PER_CORE` interior crops. When cluster naming is available, the crops come from tumour clusters only.
For each crop:
1. **Nuclei** are haematoxylin-**dominant** blobs (H ≥ `t_h` and H ≥ DAB), split by a distance-transform watershed.
2. **Cells (v2, membrane-guided)** come from a marker watershed. The nuclei are the seeds, the smoothed membrane map is the elevation, and growth is limited to `CELL_MAX_UM`. Cell outlines follow stained membranes where they exist.
3. **Membrane coverage** is the fraction of each territory boundary that lies on predicted membranous DAB, together with the mean DAB OD on that
   covered boundary.

Coverage is reported in bins (≥0.3, 0.5, 0.7). These bins describe a distribution. They are **not** a re-implementation of "complete/basolateral"
scoring. Territories are approximations, and without a tumour mask stromal cells are included.

The same features are computed from the **rule-based** membrane map and from the **student**. This lets the analysis test whether distillation adds
anything downstream. Only membrane-derived features are used for the clinical classifier. The evidence head saw the labels, so it is excluded.

In [ ]:
from multiprocessing import Pool
CAL_T = dict(t_h=CALIB["t_h"], t_weak=CALIB["t_weak"], t_mod=CALIB["t_mod"], t_strong=CALIB["t_strong"])
def cell_analysis(args):
    """Membrane-guided cell morphometry (v2). Nuclei (haematoxylin-dominant) are watershed seeds; the membrane map is the
    barrier, so cell outlines follow stained membranes where present and stay compact where absent. Coverage = fraction
    of each cell's boundary that lies on membranous DAB (complete, basolateral and lateral patterns give high/partial values)."""
    Hc, Dc, tis, mems = args
    hs, ds = ndi.gaussian_filter(Hc, 1.0), ndi.gaussian_filter(Dc, 1.0)
    nucm = ndi.binary_opening((hs >= CAL_T["t_h"]) & (hs >= ds) & tis, iterations=1)
    if nucm.sum() < 20:
        return [(np.zeros(0), np.zeros(0)) for _ in mems]
    dist = ndi.distance_transform_edt(nucm)
    lab0, _ = ndi.label(nucm)
    pk = peak_local_max(dist, min_distance=max(2, int(um2px(CFG.CELL_NUC_MIN_UM))), labels=lab0, exclude_border=False)
    mk = np.zeros_like(lab0); mk[tuple(pk.T)] = np.arange(1, len(pk) + 1)
    nuc = watershed(-dist, mk, mask=nucm)
    area = np.bincount(nuc.ravel()); small = np.where(area < math.pi * um2px(1.5) ** 2)[0]
    nuc[np.isin(nuc, small)] = 0
    territory = (expand_labels(nuc, distance=um2px(CFG.CELL_MAX_UM)) > 0) & tis
    out = []
    for mem in mems:
        elev = ndi.gaussian_filter(mem.astype(np.float32), 1.0)
        cells = watershed(elev, markers=nuc, mask=territory, compactness=0.02)
        bnd = find_boundaries(cells, mode="inner") & (cells > 0) & (nuc == 0)
        n = cells.max() + 1; tot = np.bincount(cells[bnd], minlength=n)
        md = ndi.binary_dilation(mem, iterations=2); hit = bnd & md
        cov = np.bincount(cells[hit], minlength=n); dsum = np.bincount(cells[hit], weights=Dc[hit], minlength=n)
        ok = tot >= 8; ok[0] = False
        out.append((cov[ok] / tot[ok], np.where(cov[ok] > 0, dsum[ok] / np.maximum(cov[ok], 1), 0.0)))
    return out

def summarize_cells(cov, dab, prefix):
    n = len(cov); f = lambda m: float(m.mean()) if n else 0.0
    return {f"{prefix}n_cells": n, f"{prefix}cov>=0.3": f(cov >= 0.3), f"{prefix}cov>=0.5": f(cov >= 0.5), f"{prefix}cov>=0.7": f(cov >= 0.7),
            f"{prefix}cov>=0.5&mod": f((cov >= 0.5) & (dab >= CAL_T["t_mod"])), f"{prefix}cov>=0.5&strong": f((cov >= 0.5) & (dab >= CAL_T["t_strong"])),
            f"{prefix}median_cov_pos": float(np.median(cov[cov > 0])) if (cov > 0).any() else 0.0}

def remove_small(mask_np, min_px):
    n, lab, st, _ = cv2.connectedComponentsWithStats(mask_np.astype(np.uint8), 8)
    keep = np.zeros(n, bool); keep[1:] = st[1:, cv2.CC_STAT_AREA] >= min_px
    return keep[lab]

@torch.no_grad()
def seg_predict(model, x, tta=1):
    """x: (B,3,H,W) float [0,1]. Returns p_mem, p_ev, sd_mem (B,H,W)."""
    pm, pe = [], []
    tfs = [(lambda z: z, lambda z: z), (lambda z: z.flip(-1), lambda z: z.flip(-1)), (lambda z: z.flip(-2), lambda z: z.flip(-2)),
           (lambda z: torch.rot90(z, 1, (-2, -1)), lambda z: torch.rot90(z, -1, (-2, -1)))][:max(1, tta)]
    for f, finv in tfs:
        with autocast(): lm, le = model(f(x))
        pm.append(finv(torch.sigmoid(lm.float()))); pe.append(finv(torch.sigmoid(le.float())))
    pm = torch.stack(pm); pe = torch.stack(pe)
    return pm.mean(0), pe.mean(0), pm.std(0) if len(tfs) > 1 else torch.zeros_like(pm[0])

FEAT_PATH = CACHE / "core_features.csv"
SEG_P = SEG_MODELS[SEG_PROPOSED]
if FEAT_PATH.exists():
    COREF = pd.read_csv(FEAT_PATH)
else:
    S = CFG.SEG_CROP
    flog = ProgressLog("core_features")                   # restart skips cores already measured
    both = pd.concat([TR, TE], ignore_index=True)
    both = both[[i not in flog for i in both.image_id]].reset_index(drop=True)
    print(f"core & cell features: {len(flog)} cores done earlier, {len(both)} to go")
    pool = Pool(CFG.NUM_WORKERS) if CFG.NUM_WORKERS > 1 and len(both) else None
    dl = torch.utils.data.DataLoader(DecodeDS(both), batch_size=1, num_workers=CFG.NUM_WORKERS, collate_fn=lambda b: b[0])
    for i, work in tqdm(dl, total=len(both), desc="core & cell features"):
        r = both.iloc[i]; iid = r.image_id
        if work is None or work.shape[0] < S or work.shape[1] < S:
            flog.add(iid, dict(_skip=True)); continue
        rng = np.random.default_rng(_img_seed(iid) + 7)
        mz = np.load(MAP_DIR / f"{iid}.npz")
        mp = ENGINE.maps(work, mz["I0"], mz["tissue"], mz["rim"])
        tis = mp["tissue"][0, 0]; interior = tis & ~mp["rim"][0, 0]
        b = BAGS[iid]; T = CFG.TILE
        cand = b["coords"] if TUMOUR_CLUSTERS is None else b["coords"][np.isin(b["tc"], list(TUMOUR_CLUSTERS))]
        if len(cand) == 0: cand = b["coords"]
        sel = cand[rng.choice(len(cand), min(CFG.FEAT_CROPS_PER_CORE, len(cand)), replace=False)]
        H, W = work.shape[:2]
        boxes = [(int(np.clip(y + T // 2 - S // 2, 0, H - S)), int(np.clip(x + T // 2 - S // 2, 0, W - S))) for y, x in sel]
        xb = torch.stack([torch.from_numpy(work[y:y + S, x:x + S]) for y, x in boxes]).to(DEVICE).permute(0, 3, 1, 2).float() / 255
        pmem, _, _ = seg_predict(SEG_P, xb, tta=1)
        args, acc = [], defaultdict(float)
        for j, (y, x) in enumerate(boxes):
            sl = (slice(y, y + S), slice(x, x + S))
            ti = interior[sl].cpu().numpy(); d = mp["DAB"][0, 0][sl].cpu().numpy(); hh = mp["H"][0, 0][sl].cpu().numpy()
            stu = remove_small((pmem[j].cpu().numpy() >= 0.5) & ti, CFG.MIN_MEM_OBJ_PX)
            rule = mp["mem_weak"][0, 0][sl].cpu().numpy() & ti
            acc["tissue_px"] += ti.sum()
            for nm, mk in [("stu", stu), ("rule", rule)]:
                for k in ("weak", "mod", "strong"): acc[f"{nm}_mem_{k}"] += (mk & (d >= CALIB[f"t_{k}"])).sum()
            args.append((hh, d, ti, [stu, rule]))
        res = None
        if pool is not None:
            try:
                res = pool.map_async(cell_analysis, args).get(timeout=600)   # never hang: fall back to serial
            except Exception as e:
                print(f"!! parallel cell analysis failed ({type(e).__name__}); continuing single-process"); pool.terminate(); pool = None
        if res is None: res = [cell_analysis(a) for a in args]
        rec = dict(image_id=iid, split=r.split, group=r.group, ihc_score=r.ihc_score, her2_status=r.her2_status)
        for k, v in acc.items():
            if k != "tissue_px": rec[k + "_frac"] = v / max(acc["tissue_px"], 1)
        for mi, nm in enumerate(["stu", "rule"]):
            cov = np.concatenate([rr[mi][0] for rr in res]); dd = np.concatenate([rr[mi][1] for rr in res])
            rec.update(summarize_cells(cov, dd, f"{nm}_cell_"))
        flog.add(iid, rec); del mp
    if pool: pool.close()
    COREF = pd.DataFrame([{k: v for k, v in rec.items() if k != "_key"} for rec in flog.records() if not rec.get("_skip")])
    atomic_csv(COREF, FEAT_PATH, index=False)
section_time("core & cell features")

## 15. Whole-core test inference: membrane masks, intensity grades, evidence maps, and uncertainty
The student runs as a sliding window with Hann blending and `TTA` flips/rotations. It outputs:
* **membrane** = P ≥ 0.5 inside tissue, with small objects removed and no morphological closing, so thin structures survive
* an **intensity grade** for each membrane pixel, *measured* from absolute DAB OD rather than predicted
* an **evidence region** map
* **uncertainty**, as the SD across TTA

The stain-perturbation self-consistency check re-segments a subset with a fixed ±5% stain shift.

In [ ]:
TEST_OUT = OUT / "test_outputs"
@torch.no_grad()
def segment_core(model, work, tissue, tta):
    S = CFG.SEG_CROP; stride = S - S // 4; H, W = work.shape[:2]
    ys, xs = torch.nonzero(tissue, as_tuple=True)
    if len(ys) == 0: z = np.zeros((H, W), np.float32); return z, z, z, []
    y0, y1, x0, x1 = int(ys.min()), int(ys.max()), int(xs.min()), int(xs.max())
    gy = list(range(max(0, y0 - S // 4), max(1, min(H - S, y1) + 1), stride)) or [0]
    gx = list(range(max(0, x0 - S // 4), max(1, min(W - S, x1) + 1), stride)) or [0]
    if gy[-1] + S < min(H, y1 + 1): gy.append(H - S)
    if gx[-1] + S < min(W, x1 + 1): gx.append(W - S)
    win = torch.hann_window(S, periodic=False, device=DEVICE); win = (0.1 + 0.9 * win[:, None] * win[None, :])
    acc = torch.zeros(3, H, W, device=DEVICE); wsum = torch.zeros(H, W, device=DEVICE)
    xg = torch.from_numpy(work).to(DEVICE).permute(2, 0, 1)
    boxes = [(y, x) for y in gy for x in gx if tissue[y:y + S, x:x + S].float().mean() > 0.02]
    for b in range(0, len(boxes), CFG.SEG_BS):
        bb = boxes[b:b + CFG.SEG_BS]
        xb = torch.stack([xg[:, y:y + S, x:x + S] for y, x in bb]).float() / 255
        pm, pe, sd = gpu_safe(lambda t: seg_predict(model, t, tta), xb)
        for j, (y, x) in enumerate(bb):
            acc[0, y:y + S, x:x + S] += pm[j] * win; acc[1, y:y + S, x:x + S] += pe[j] * win; acc[2, y:y + S, x:x + S] += sd[j] * win
            wsum[y:y + S, x:x + S] += win
    acc = acc / wsum.clamp(min=1e-6)
    return acc[0].cpu().numpy(), acc[1].cpu().numpy(), acc[2].cpu().numpy(), boxes

def dice_np(a, b):
    s = a.sum() + b.sum(); return float(2 * (a & b).sum() / s) if s else 1.0

TEST_SEG_PATH = CACHE / "test_seg_summary.csv"; ROI_PATH = CACHE / "test_roi_candidates.csv"
if TEST_SEG_PATH.exists():
    TSEG = pd.read_csv(TEST_SEG_PATH); ROIC = pd.read_csv(ROI_PATH)
else:
    perturb_ids = set(TE.image_id.sample(min(40, len(TE)), random_state=CFG.SEED))   # fixed set, independent of restarts
    slog = ProgressLog("test_seg")
    todo = TE[[i not in slog for i in TE.image_id]].reset_index(drop=True)
    print(f"test segmentation: {len(slog)} cores done earlier, {len(todo)} to go")
    dl = torch.utils.data.DataLoader(DecodeDS(todo), batch_size=1, num_workers=CFG.NUM_WORKERS, collate_fn=lambda b: b[0])
    for i, work in tqdm(dl, total=len(todo), desc="test whole-core segmentation"):
        r = todo.iloc[i]; iid = r.image_id
        if work is None or min(work.shape[:2]) < CFG.SEG_CROP:
            slog.add(iid, dict(_skip=True)); continue
        t_core = time.time(); mz = np.load(MAP_DIR / f"{iid}.npz")
        mp = ENGINE.maps(work, mz["I0"], mz["tissue"], mz["rim"])
        tis_t = mp["tissue"][0, 0]; tis = tis_t.cpu().numpy(); rim = mp["rim"][0, 0].cpu().numpy(); dab = mp["DAB"][0, 0].cpu().numpy()
        pm, pe, sd, boxes = segment_core(SEG_P, work, tis_t, CFG.TTA)
        mem = remove_small((pm >= 0.5) & tis, CFG.MIN_MEM_OBJ_PX)
        grade = np.zeros(mem.shape, np.uint8)
        for g, k in enumerate(("weak", "mod", "strong"), 1): grade[mem & (dab >= CALIB[f"t_{k}"])] = g
        rule = mp["mem_weak"][0, 0].cpu().numpy() & ~rim
        interior = tis & ~rim; nt = max(interior.sum(), 1)
        rec = dict(image_id=iid, group=r.group, ihc_score=r.ihc_score, her2_status=r.her2_status,
                   stu_mem_any=float((mem & interior).sum() / nt), stu_mem_weak=float(((grade >= 1) & interior).sum() / nt),
                   stu_mem_mod=float(((grade >= 2) & interior).sum() / nt), stu_mem_strong=float(((grade >= 3) & interior).sum() / nt),
                   rule_mem_weak=float((rule & interior).sum() / nt), stu_vs_rule_dice=dice_np(mem & interior, rule & interior),
                   rim_to_interior_mem_ratio=float(((grade >= 1) & rim).sum() / max(rim.sum(), 1) / max(((grade >= 1) & interior).sum() / nt, 1e-6)),
                   evidence_area=float(((pe >= 0.5) & tis).sum() / max(tis.sum(), 1)), mean_tta_sd=float(sd[tis].mean()) if tis.any() else np.nan,
                   subthreshold_membrane=float((mem & (grade == 0) & interior).sum() / nt))
        rec["seg_seconds"] = float(time.time() - t_core)          # maps + segmentation + features per core (decode excluded; TTA included; stain-shift rerun excluded)
        if iid in perturb_ids:
            xs = torch.from_numpy(work).to(DEVICE).permute(2, 0, 1)[None].float() / 255
            od = -torch.log(xs.clamp(min=1 / 255)); c = torch.einsum("bchw,cd->bdhw", od, M_STAIN_INV)
            c = c * torch.tensor([0.95, 1.05, 1.0], device=DEVICE).view(1, 3, 1, 1)
            wp = (torch.exp(-torch.einsum("bdhw,dc->bchw", c, M_STAIN)).clamp(0, 1)[0].permute(1, 2, 0) * 255).byte().cpu().numpy()
            pm2, _, _, _ = segment_core(SEG_P, wp, tis_t, 1)
            rec["stain_shift_dice"] = dice_np(mem, remove_small((pm2 >= 0.5) & tis, CFG.MIN_MEM_OBJ_PX))
        core_rois = []; S = CFG.SEG_CROP
        for (y, x) in boxes:
            ti = interior[y:y + S, x:x + S]
            if ti.mean() < 0.4: continue
            core_rois.append(dict(image_id=iid, y=int(y), x=int(x), tissue=float(ti.mean()), tta_sd=float(sd[y:y + S, x:x + S][ti].mean()),
                             evidence=float(pe[y:y + S, x:x + S][ti].mean()), mem=float(mem[y:y + S, x:x + S][ti].mean())))
        if CFG.SAVE_FULLRES_MASKS and free_disk_gb() < 3:
            CFG.SAVE_FULLRES_MASKS = False; print("!! free disk < 3 GB: full-resolution test masks are no longer saved (overlays still are)")
        if CFG.SAVE_FULLRES_MASKS:
            cv2.imwrite(str(TEST_OUT / f"{iid}_membrane_grade.png"), grade)
            cv2.imwrite(str(TEST_OUT / f"{iid}_p_membrane.png"), (pm * 255).astype(np.uint8))
            cv2.imwrite(str(TEST_OUT / f"{iid}_p_evidence.png"), (pe * 255).astype(np.uint8))
        R = 4; small = cv2.resize(work, (work.shape[1] // R, work.shape[0] // R), interpolation=cv2.INTER_AREA)
        gs = cv2.resize(grade, (small.shape[1], small.shape[0]), interpolation=cv2.INTER_NEAREST)
        ov = small.copy()
        for g, col in [(1, (42, 120, 214)), (2, (235, 104, 52)), (3, (227, 73, 72))]: ov[gs == g] = col
        cv2.imwrite(str(TEST_OUT / f"{iid}_overlay.jpg"), cv2.cvtColor(np.concatenate([small, ov], 1), cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 88])
        del mp
        slog.add(iid, dict(rec=rec, rois=core_rois))              # written last: overlay/masks above are idempotent rewrites
    done = [x for x in slog.records() if not x.get("_skip")]
    TSEG = pd.DataFrame([x["rec"] for x in done])
    ROIC = pd.DataFrame([roi for x in done for roi in x["rois"]],
                        columns=["image_id", "y", "x", "tissue", "tta_sd", "evidence", "mem"])
    atomic_csv(TSEG, TEST_SEG_PATH, index=False); atomic_csv(ROIC, ROI_PATH, index=False)
section_time("whole-core test segmentation")

# ablation students on the same test proxies (crop-level, cheaper): IHC-0 membranous area >= moderate, TTA stability
@torch.no_grad()
def student_proxy_on_test(model, n_cores=60, tta=CFG.TTA):
    sub = TE.sample(min(n_cores, len(TE)), random_state=CFG.SEED).reset_index(drop=True); out = []
    S = CFG.SEG_CROP; rng = np.random.default_rng(1)
    for _, r in sub.iterrows():
        work = read_rgb(r.path, CFG.WORK_DS); mz = np.load(MAP_DIR / f"{r.image_id}.npz")
        mp = ENGINE.maps(work, mz["I0"], mz["tissue"], mz["rim"], need_ridge=False)
        c = BAGS[r.image_id]["coords"]; c = c[rng.choice(len(c), min(4, len(c)), replace=False)]
        H, W = work.shape[:2]; bx = [(int(np.clip(y, 0, H - S)), int(np.clip(x, 0, W - S))) for y, x in c]
        xb = torch.stack([torch.from_numpy(work[y:y + S, x:x + S]) for y, x in bx]).to(DEVICE).permute(0, 3, 1, 2).float() / 255
        pm, _, sd = seg_predict(model, xb, tta)
        d = torch.stack([mp["DAB"][0, 0, y:y + S, x:x + S] for y, x in bx]); t = torch.stack([mp["tissue"][0, 0, y:y + S, x:x + S] for y, x in bx])
        m = (pm >= 0.5) & t
        out.append(dict(group=r.group, mem_mod=float((m & (d >= CALIB["t_mod"])).sum() / t.sum().clamp(min=1)), tta_sd=float(sd[t].mean()) if t.any() else np.nan))
    o = pd.DataFrame(out)
    return dict(IHC0_membranous_area_ge_mod=float(o[o.group == "0"].mem_mod.mean()) if (o.group == "0").any() else np.nan,
                IHC3_membranous_area_ge_mod=float(o[o.group == "3"].mem_mod.mean()) if (o.group == "3").any() else np.nan,
                mean_TTA_SD=float(o.tta_sd.mean()))
SEG_PROXY_PATH = CACHE / "seg_test_proxies.csv"
SEG_PROXY = pd.read_csv(SEG_PROXY_PATH) if SEG_PROXY_PATH.exists() else None
if SEG_PROXY is None or set(SEG_PROXY.model) != set(SEG_MODELS):
    SEG_PROXY = pd.DataFrame([{"model": n, **student_proxy_on_test(m)} for n, m in SEG_MODELS.items()])
    SEG_PROXY["IHC3_to_IHC0_contrast"] = SEG_PROXY.IHC3_membranous_area_ge_mod / SEG_PROXY.IHC0_membranous_area_ge_mod.clip(lower=1e-6)
    atomic_csv(SEG_PROXY, SEG_PROXY_PATH, index=False)
SEG_PROXY.to_csv(TAB / "seg_test_proxies.csv", index=False); print(SEG_PROXY.round(4).to_string(index=False))
print("Test whole-core summary by IHC group:\n", TSEG.groupby("group")[["stu_mem_weak", "stu_mem_mod", "stu_mem_strong", "rule_mem_weak",
      "stu_vs_rule_dice", "rim_to_interior_mem_ratio", "mean_tta_sd"]].median().reindex(GROUP_ORDER).round(4))
if "stain_shift_dice" in TSEG: print("stain-shift self-consistency Dice (median):", round(float(TSEG.stain_shift_dice.median()), 3))

## 16. Clinical consistency of the segmentation-derived features (the downstream proxy)
If the membrane maps capture diagnostically relevant structure, the features derived from them should do three things:
1. increase monotonically with the IHC score (Spearman ρ with a bootstrap CI, plus Kruskal–Wallis)
2. predict FISH-resolved status on the **independent test set**, when an interpretable logistic model is fitted on the train cores
3. beat the stain-intensity rule

Features from the rule map, the student map, and the student plus cell morphometry are compared. This is evidence of *clinical plausibility*,
not of pixel accuracy.

In [ ]:
FEATSETS = {
    "rule membrane fractions": [c for c in COREF.columns if c.startswith("rule_mem_")],
    "student membrane fractions": [c for c in COREF.columns if c.startswith("stu_mem_")],
    "student + cell morphometry": [c for c in COREF.columns if c.startswith("stu_")],
    "rule + cell morphometry": [c for c in COREF.columns if c.startswith("rule_")],
}
cf_tr = COREF[COREF.split == "train"].merge(TR[["image_id", "fold"]], on="image_id")
cf_te = COREF[COREF.split == "test"].set_index("image_id").reindex(TE.image_id).reset_index()
CLIN = []
for fs, cols in FEATSETS.items():
    Xtr = cf_tr[cols].fillna(0).values; Xte = cf_te[cols].fillna(0).values
    mdl = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=3000, C=0.5)).fit(Xtr, cf_tr.her2_status)
    p = mdl.predict_proba(Xte)[:, 1]; ok = ~np.isnan(cf_te[cols].values).any(1)
    auc = roc_auc_score(cf_te.her2_status[ok], p[ok]) if len(set(cf_te.her2_status[ok])) > 1 else np.nan
    lo, hi = boot_ci(cf_te.her2_status[ok].values, p[ok], 0.5)
    CLIN.append(dict(feature_set=fs, n_features=len(cols), test_AUROC=auc, lo=lo["AUROC"], hi=hi["AUROC"]))
CLIN.append(dict(feature_set="Stain intensity rule (Pisula 2023)", n_features=1, test_AUROC=CLS[CLS.method == "Stain intensity rule (Pisula 2023)"].AUROC.iloc[0],
                 lo=CLS[CLS.method == "Stain intensity rule (Pisula 2023)"].AUROC_lo.iloc[0], hi=CLS[CLS.method == "Stain intensity rule (Pisula 2023)"].AUROC_hi.iloc[0]))
CLIN = pd.DataFrame(CLIN); CLIN.to_csv(TAB / "clinical_feature_models.csv", index=False); print(CLIN.round(3).to_string(index=False))

TREND = []
key_feats = ["rule_mem_mod_frac", "stu_mem_mod_frac", "stu_mem_strong_frac", "stu_cell_cov>=0.5", "stu_cell_cov>=0.5&mod", "stu_cell_median_cov_pos"]
for split, d in [("train", cf_tr), ("test", cf_te)]:
    for f in key_feats:
        if f not in d: continue
        x, y = d[f].values, d.ihc_score.values; ok = ~np.isnan(x)
        rho = stats.spearmanr(y[ok], x[ok]).correlation; br = []
        r0 = np.random.default_rng(0)
        for _ in range(500):
            ii = r0.choice(ok.sum(), ok.sum()); br.append(stats.spearmanr(y[ok][ii], x[ok][ii]).correlation)
        groups = [x[ok][d.group.values[ok] == g] for g in GROUP_ORDER if (d.group.values[ok] == g).sum() > 1]
        kw = stats.kruskal(*groups).pvalue if len(groups) > 1 else np.nan
        med = {f"median_{g}": float(np.median(x[ok][d.group.values[ok] == g])) if (d.group.values[ok] == g).any() else np.nan for g in GROUP_ORDER}
        TREND.append(dict(split=split, feature=f, spearman_rho=rho, rho_lo=np.nanquantile(br, 0.025), rho_hi=np.nanquantile(br, 0.975), kruskal_p=kw, **med))
TREND = pd.DataFrame(TREND); TREND.to_csv(TAB / "clinical_feature_trends.csv", index=False); print(TREND.round(3).to_string(index=False))

# ---- H3: do membrane / cell features separate the clinically hard pairs? ----
# (a) IHC 3+ vs IHC 2+ (FISH- and FISH+ pooled): the visual 2+/3+ boundary that the ASCO/CAP score rests on.
# (b) 2+/FISH+ vs 2+/FISH-: whether morphometry carries any signal for the equivocal cores that currently need ISH.
# Single-feature AUROC (direction fixed on TRAIN, so the test AUROC is not optimistically flipped), stratified bootstrap CI,
# Holm correction across features within each comparison x split. Small groups are reported, never hidden.
def strat_boot_auc(y, x, n=2000, seed=0):
    """Stratified bootstrap CI of AUROC, vectorised via the Mann-Whitney rank identity (ties get mid-ranks)."""
    r0 = np.random.default_rng(seed); x1, x0 = x[y == 1], x[y == 0]; n1, n0 = len(x1), len(x0)
    X = np.concatenate([x1[r0.integers(0, n1, (n, n1))], x0[r0.integers(0, n0, (n, n0))]], 1)
    R = stats.rankdata(X, axis=1)
    auc = (R[:, :n1].sum(1) - n1 * (n1 + 1) / 2) / (n1 * n0)
    return np.quantile(auc, [0.025, 0.975])
def mw_p(y, x):
    return stats.mannwhitneyu(x[y == 1], x[y == 0], alternative="two-sided").pvalue
H3_FEATS = [c for c in COREF.columns if c.startswith(("stu_", "rule_")) and COREF[c].dtype.kind in "fi"]
H3 = []
for comp, pos, neg in [("3+ vs 2+", ["3"], ["2-", "2+"]), ("2+/FISH+ vs 2+/FISH-", ["2+"], ["2-"])]:
    dtr = cf_tr[cf_tr.group.isin(pos + neg)]; dte = cf_te[cf_te.group.isin(pos + neg)]
    for f in H3_FEATS:
        xtr = dtr[f].values.astype(float); ytr = dtr.group.isin(pos).values.astype(int); ok = ~np.isnan(xtr)
        if ok.sum() < 6 or len(set(ytr[ok])) < 2 or np.nanstd(xtr) == 0: continue
        sign = 1.0 if roc_auc_score(ytr[ok], xtr[ok]) >= 0.5 else -1.0          # direction learned on train only
        for split, d in [("train", dtr), ("test", dte)]:
            x = sign * d[f].values.astype(float); y = d.group.isin(pos).values.astype(int); ok = ~np.isnan(x)
            if ok.sum() < 4 or len(set(y[ok])) < 2: continue
            lo, hi = strat_boot_auc(y[ok], x[ok], n=1000 if split == "train" else 2000)
            H3.append(dict(comparison=comp, split=split, feature=f, direction="+" if sign > 0 else "-", n_pos=int(y[ok].sum()),
                           n_neg=int((1 - y[ok]).sum()), AUROC=roc_auc_score(y[ok], x[ok]), lo=lo, hi=hi, p_mannwhitney=mw_p(y[ok], x[ok])))
H3 = pd.DataFrame(H3)
if len(H3):
    H3["p_holm"] = np.nan
    for _, idx in H3.groupby(["comparison", "split"]).groups.items():
        p = H3.loc[idx, "p_mannwhitney"].values; o = np.argsort(p); m = len(p); adj = np.empty(m)
        adj[o] = np.minimum(1, np.maximum.accumulate((m - np.arange(m)) * p[o])); H3.loc[idx, "p_holm"] = adj
    H3 = H3.sort_values(["comparison", "split", "AUROC"], ascending=[True, False, False])
    H3.to_csv(TAB / "h3_feature_separation_3v2_and_fish.csv", index=False)
    for (comp, split), d in H3.groupby(["comparison", "split"], sort=False):
        print(f"\nH3 {comp} [{split}] (n+={d.n_pos.iloc[0]}, n-={d.n_neg.iloc[0]}) top features:")
        print(d.head(6)[["feature", "direction", "AUROC", "lo", "hi", "p_holm"]].round(3).to_string(index=False))
    # figure: train vs test AUROC of every feature for the 3+ vs 2+ comparison (features ranked on train)
    d = H3[H3.comparison == "3+ vs 2+"].pivot_table(index="feature", columns="split", values=["AUROC", "lo", "hi"])
    if ("AUROC", "test") in d.columns:
        d = d.sort_values(("AUROC", "train"), ascending=False).head(15)
        fig, ax = plt.subplots(figsize=(7, 0.32 * len(d) + 1.2)); yy = np.arange(len(d))
        for off, sp, col in [(-0.15, "train", "#2a78d6"), (0.15, "test", "#e34948")]:
            ax.errorbar(d[("AUROC", sp)], yy + off, xerr=[d[("AUROC", sp)] - d[("lo", sp)], d[("hi", sp)] - d[("AUROC", sp)]],
                        fmt="o", ms=4, color=col, label=sp, capsize=2)
        ax.axvline(0.5, color="#888", ls="--", lw=1); ax.set_yticks(yy); ax.set_yticklabels(d.index, fontsize=7); ax.invert_yaxis()
        ax.set_xlabel("AUROC, IHC 3+ vs 2+ (95% stratified bootstrap CI)"); ax.legend(fontsize=8)
        savefig(fig, "fig12_h3_3plus_vs_2plus")
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
for ax, f in zip(axes, ["stu_mem_mod_frac", "stu_cell_cov>=0.5", "stu_cell_cov>=0.5&mod", "rule_mem_mod_frac"]):
    for k, g in enumerate(GROUP_ORDER):
        v = cf_te[cf_te.group == g][f].dropna().values if f in cf_te else []
        if len(v) == 0: v = cf_tr[cf_tr.group == g][f].dropna().values; mk = "x"
        else: mk = "o"
        ax.scatter(np.full(len(v), k) + np.random.default_rng(k).uniform(-0.18, 0.18, len(v)), v, s=8, color=GROUP_COL[g], alpha=0.6, marker=mk)
        if len(v): ax.hlines(np.median(v), k - 0.3, k + 0.3, color="#0b0b0b", lw=1.5)
    ax.set_xticks(range(len(GROUP_ORDER))); ax.set_xticklabels(GROUP_ORDER); ax.set_xlabel("IHC group"); ax.set_title(f, fontsize=8)
axes[0].set_ylabel("fraction per core")
savefig(fig, "fig10_clinical_consistency")
section_time("clinical consistency")

## 17. True segmentation accuracy, available only once expert masks exist
Point `CFG.REF_MASK_DIR` to a folder of binary PNG masks, one per worklist ROI: `<roi_id>.png` for the primary annotator and
`<roi_id>__<initials>.png` for a second annotator. Whole-core masks (`<image stem>.png`) are also accepted.

**Why the metric set is more than Dice.** HER2 membranes are 2–4 px wide at 0.46 µm/px. On such thin structures, plain pixel Dice
punishes a correct line drawn 1 px to the side as if it were wrong. Expert masks are drawn as lines, which makes this worse. So the
main metrics are the ones used for thin, curvilinear structures:
* **clDice** (centreline Dice): is each centreline covered by the other mask?
* **Tolerance F1 at 1 µm**: a pixel counts as correct if it lies within 1 µm of the other mask.
* **Boundary F1 at 2 µm** and **HD95** (µm).

Plain Dice and IoU are still reported, for comparability with other papers.

**ROIs with no membrane** (common in IHC 0) have no defined Dice. They are scored separately as the predicted membrane area, which is a
false-positive measure, so they never inflate or deflate the Dice.

**Statistics.** The core is the independent unit: metrics are averaged per core, then given 95% bootstrap CIs over cores. Only the primary
annotator's masks go into the main estimate.

**Inter-observer benchmark.** When a second annotator has scored a subset, the code compares model-vs-annotator agreement with
annotator-vs-annotator agreement on the same ROIs, with a paired bootstrap. Reviewers judge the model against that ceiling.

In [ ]:
from skimage.morphology import skeletonize
def boundary(m): return m ^ ndi.binary_erosion(m, iterations=1, border_value=0)
def seg_metrics(pred, ref, mpp=WORK_MPP, tol_um=1.0, btol_um=2.0, empty_frac=0.005):
    pred, ref = pred.astype(bool), ref.astype(bool)
    out = dict(ref_frac=float(ref.mean()), pred_frac=float(pred.mean()))
    if not ref.any():   # membrane-free ROI: Dice undefined -> report false-positive area only
        out.update(ref_empty=True, pred_empty=bool(pred.mean() < empty_frac))
        return out
    out["ref_empty"] = False
    tp = (pred & ref).sum(); fp = (pred & ~ref).sum(); fn = (~pred & ref).sum()
    out.update(Dice=2 * tp / max(2 * tp + fp + fn, 1), IoU=tp / max(tp + fp + fn, 1), Sensitivity=tp / max(tp + fn, 1), Precision=tp / max(tp + fp, 1))
    if not pred.any():
        out.update(clDice=0.0, TolF1=0.0, BoundaryF1=0.0, HD95_um=np.nan); return out
    d_ref = ndi.distance_transform_edt(~ref) * mpp; d_pred = ndi.distance_transform_edt(~pred) * mpp
    tp_p = (d_ref[pred] <= tol_um).mean(); tp_r = (d_pred[ref] <= tol_um).mean()            # tolerance precision / recall
    out["TolF1"] = 2 * tp_p * tp_r / max(tp_p + tp_r, 1e-9)
    sp, sr = skeletonize(pred), skeletonize(ref)                                              # clDice (Shit et al., CVPR 2021)
    t_prec = (sp & ref).sum() / max(sp.sum(), 1); t_sens = (sr & pred).sum() / max(sr.sum(), 1)
    out["clDice"] = 2 * t_prec * t_sens / max(t_prec + t_sens, 1e-9)
    bp, br = boundary(pred), boundary(ref)
    dp = ndi.distance_transform_edt(~br)[bp] * mpp; dr = ndi.distance_transform_edt(~bp)[br] * mpp
    out["HD95_um"] = float(max(np.percentile(dp, 95), np.percentile(dr, 95)))
    prc, rec = (dp <= btol_um).mean(), (dr <= btol_um).mean(); out["BoundaryF1"] = 2 * prc * rec / max(prc + rec, 1e-9)
    return out

def _read_mask(p, shape=None):
    m = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    if m is None: return None
    m = m > 0
    if shape is not None and m.shape != shape: m = cv2.resize(m.astype(np.uint8), shape[::-1], interpolation=cv2.INTER_NEAREST) > 0
    return m

def parse_ref_name(stem):
    """'<roi_id>' -> (roi_id, None); '<roi_id>__<initials>' -> (roi_id, initials). ROI ids themselves end in '__roi<k>'."""
    parts = stem.split("__")
    if len(parts) >= 2 and not re.fullmatch(r"roi\d+", parts[-1]): return "__".join(parts[:-1]), parts[-1]
    return stem, None

def pred_lookup(stem):
    f = OUT / "annotation_worklist" / "pred_masks" / f"{stem}.png"
    if f.exists(): return _read_mask(f)
    f = TEST_OUT / f"{stem}_membrane_grade.png"
    return _read_mask(f) if f.exists() else None

METRICS = ["Dice", "IoU", "Sensitivity", "Precision", "clDice", "TolF1", "BoundaryF1", "HD95_um"]
def core_boot(df, cols, n=None, seed=0):
    core = df.groupby("core")[cols].mean(); r0 = np.random.default_rng(seed); n = n or CFG.N_BOOT
    bs = pd.DataFrame([core.iloc[r0.integers(0, len(core), len(core))].mean() for _ in range(n)])
    return pd.DataFrame({"mean_over_cores": core.mean(), "lo": bs.quantile(0.025), "hi": bs.quantile(0.975), "n_cores": len(core)})

if CFG.REF_MASK_DIR and Path(CFG.REF_MASK_DIR).exists():
    refs = {}
    for p in sorted(Path(CFG.REF_MASK_DIR).glob("*.png")):
        unit, annot = parse_ref_name(p.stem); refs.setdefault(unit, {})[annot or "primary"] = p
    rows, pair_rows = [], []
    for unit, ann in refs.items():
        pred = pred_lookup(unit)
        if pred is None: print(f"  no prediction found for {unit}; skipped"); continue
        core = unit.split("__roi")[0]
        for who, p in ann.items():
            ref = _read_mask(p, pred.shape)
            if ref is not None: rows.append(dict(unit=unit, core=core, annotator=who, **seg_metrics(pred, ref)))
        names = sorted(ann)                                   # annotator-vs-annotator on the same ROI (inter-observer ceiling)
        for i in range(len(names)):
            for j in range(i + 1, len(names)):
                a, b = _read_mask(ann[names[i]], pred.shape), _read_mask(ann[names[j]], pred.shape)
                if a is not None and b is not None:
                    pair_rows.append(dict(unit=unit, core=core, pair=f"{names[i]} vs {names[j]}", **seg_metrics(a, b)))
    REFM = pd.DataFrame(rows); atomic_csv(REFM, TAB / "reference_mask_metrics_per_roi.csv", index=False)
    prim = REFM[(REFM.annotator == "primary") & ~REFM.ref_empty.astype(bool)]
    neg = REFM[(REFM.annotator == "primary") & REFM.ref_empty.astype(bool)]
    print(f"reference masks: {REFM.unit.nunique()} ROIs, {REFM.core.nunique()} cores, annotators: {sorted(REFM.annotator.unique())}")
    if len(prim):
        SUMM = core_boot(prim, [c for c in METRICS if c in prim])
        atomic_csv(SUMM.reset_index().rename(columns={"index": "metric"}), TAB / "reference_mask_metrics_summary.csv", index=False)
        print("Model vs primary expert, membrane-present ROIs (mean over cores, 95% bootstrap CI over cores):"); print(SUMM.round(3))
        _wl = OUT / "annotation_worklist" / "worklist.csv"
        if _wl.exists():
            g = prim.merge(pd.read_csv(_wl)[["roi_id", "ihc_group"]], left_on="unit", right_on="roi_id", how="left")
            print("by IHC group (median clDice / TolF1):\n", g.groupby("ihc_group")[["clDice", "TolF1"]].median().round(3))
    if len(neg):
        NEGS = dict(n_rois=len(neg), n_cores=neg.core.nunique(), median_pred_membrane_frac=float(neg.pred_frac.median()),
                    rois_with_pred_below_0_5pct=float(neg.pred_empty.mean()))
        save_json(NEGS, TAB / "reference_mask_membrane_free_rois.json"); print("membrane-free ROIs (false-positive check):", NEGS)
    PAIRS = pd.DataFrame(pair_rows)
    if len(PAIRS):
        PAIRS = PAIRS[~PAIRS.ref_empty.astype(bool)]; atomic_csv(PAIRS, TAB / "inter_observer_per_roi.csv", index=False)
        sec = [a for a in REFM.annotator.unique() if a != "primary"]
        both = PAIRS[PAIRS.pair.str.contains("primary")].unit.unique()
        cmp_rows = []
        for m_ in ["clDice", "TolF1", "Dice"]:
            mp_ = REFM[(REFM.annotator == "primary") & REFM.unit.isin(both)].groupby("core")[m_].mean()
            ms_ = REFM[REFM.annotator.isin(sec) & REFM.unit.isin(both)].groupby("core")[m_].mean()
            io_ = PAIRS[PAIRS.unit.isin(both)].groupby("core")[m_].mean()
            model_ = pd.concat([mp_, ms_], axis=1).mean(axis=1).reindex(io_.index)
            d = (model_ - io_).dropna(); r0 = np.random.default_rng(1)
            bs = [d.iloc[r0.integers(0, len(d), len(d))].mean() for _ in range(CFG.N_BOOT)] if len(d) else [np.nan]
            cmp_rows.append(dict(metric=m_, n_cores=len(d), model_vs_experts=float(model_.mean()), expert_vs_expert=float(io_.mean()),
                                 diff=float(d.mean()) if len(d) else np.nan, lo=float(np.quantile(bs, 0.025)), hi=float(np.quantile(bs, 0.975))))
        IOBS = pd.DataFrame(cmp_rows); atomic_csv(IOBS, TAB / "model_vs_interobserver.csv", index=False)
        print("Model agreement with experts vs expert-expert agreement (same ROIs; diff CI including 0 = within inter-observer variability):")
        print(IOBS.round(3).to_string(index=False))
else:
    print("No expert reference masks supplied. NO segmentation accuracy (Dice/IoU vs truth) is claimed by this notebook.")

## 18. Smallest useful expert-annotation set (active worklist)
The goal is the fewest annotations that give a usable CI on Dice. The worklist samples cores from three groups:
* **all** test IHC 2+ (both FISH outcomes) and IHC 3+ cores
* the IHC-0 cores with the most predicted membranous DAB, which are likely false positives
* the highest-uncertainty remaining cores

Within each core, 512×512 ROIs (about 235 µm) are chosen by uncertainty, evidence, and at random for an unbiased estimate. Each ROI is exported as a
PNG together with the model's predicted mask, a CSV in **native** pixel coordinates, and a QuPath-importable GeoJSON.

**Sample-size rationale.** The CI is computed over **cores**, not ROIs, because ROIs within a core are correlated. Assuming a per-core Dice SD
of about 0.15, n cores gives a 95% CI half-width of about 1.96 × 0.15 / √n. That is ±0.05 at about 35 cores and ±0.04 at about 55 cores.

In [ ]:
WL = OUT / "annotation_worklist"; (WL / "rois").mkdir(exist_ok=True); (WL / "pred_masks").mkdir(exist_ok=True)
t = TSEG.merge(TE_PRED[["image_id", "entropy"]], on="image_id", how="left")
sel = list(t[t.ihc_score.isin([2, 3])].image_id)
sel += list(t[(t.ihc_score == 0) & ~t.image_id.isin(sel)].sort_values("stu_mem_mod", ascending=False).image_id[:12])
sel += list(t[~t.image_id.isin(sel)].sort_values("mean_tta_sd", ascending=False).image_id[:10])
rng = np.random.default_rng(CFG.SEED); wl_rows, feats = [], []
path_of = dict(zip(TE.image_id, TE.path)); S = CFG.SEG_CROP
for iid in tqdm(sel, desc="worklist ROIs"):
    rc = ROIC[ROIC.image_id == iid]
    if rc.empty: continue
    picks = []
    for col in ["tta_sd", "evidence"]:
        c = rc[~rc.index.isin([p.name for p in picks])].sort_values(col, ascending=False)
        if len(c): picks.append(c.iloc[0])
    c = rc[~rc.index.isin([p.name for p in picks])]
    if len(c): picks.append(c.iloc[rng.integers(len(c))])
    work = read_rgb(path_of[iid], CFG.WORK_DS)
    gfile = TEST_OUT / f"{iid}_membrane_grade.png"; grade = cv2.imread(str(gfile), cv2.IMREAD_GRAYSCALE) if gfile.exists() else None
    for k, p in enumerate(picks[:CFG.N_ANNOT_ROIS_PER_CORE]):
        y, x = int(p.y), int(p.x); roi = f"{iid}__roi{k}"
        cv2.imwrite(str(WL / "rois" / f"{roi}.png"), cv2.cvtColor(work[y:y + S, x:x + S], cv2.COLOR_RGB2BGR))
        if grade is not None: cv2.imwrite(str(WL / "pred_masks" / f"{roi}.png"), ((grade[y:y + S, x:x + S] > 0) * 255).astype(np.uint8))
        X0, Y0, side = x * CFG.WORK_DS, y * CFG.WORK_DS, S * CFG.WORK_DS
        why = ["highest uncertainty", "highest evidence", "random (unbiased)"][k] if k < 3 else "extra"
        g = TE.set_index("image_id").loc[iid]
        wl_rows.append(dict(roi_id=roi, image_id=iid, filename=g.filename, ihc_group=g.group, roi_reason=why,
                            x_native=X0, y_native=Y0, w_native=side, h_native=side, work_level_mpp=WORK_MPP))
        feats.append({"_file": g.filename, "type": "Feature", "properties": {"name": roi, "objectType": "annotation", "classification": {"name": "ROI to annotate"}},
                      "geometry": {"type": "Polygon", "coordinates": [[[X0, Y0], [X0 + side, Y0], [X0 + side, Y0 + side], [X0, Y0 + side], [X0, Y0]]]}})
WLDF = pd.DataFrame(wl_rows)
# second-annotator subset: about 21 ROIs, stratified by IHC group and ROI reason, one ROI per core (cores are the independent unit)
if len(WLDF):
    one = WLDF.groupby("image_id", group_keys=False).apply(lambda d: d.sample(1, random_state=CFG.SEED))
    IO_SET = one.groupby("ihc_group", group_keys=False).apply(lambda d: d.sample(min(len(d), max(3, round(21 * len(d) / len(one)))),
                                                                                    random_state=CFG.SEED)).head(21)
    WLDF["second_annotator"] = WLDF.roi_id.isin(IO_SET.roi_id)
WLDF.to_csv(WL / "worklist.csv", index=False)
# one GeoJSON per core image: QuPath imports objects into the image that is open, so a combined file would place every core's
# squares on every image
QP = WL / "qupath_geojson"; QP.mkdir(exist_ok=True)
for fname in sorted({f["_file"] for f in feats}):
    fc = [{k: v for k, v in f.items() if k != "_file"} for f in feats if f["_file"] == fname]
    json.dump({"type": "FeatureCollection", "features": fc}, open(QP / f"{Path(fname).stem}.geojson", "w"), indent=1)
(WL / "export_masks_qupath.groovy").write_text(r"""// Export HER2 membrane reference masks: one PNG per worklist ROI, 0.46 um/px, same size as the ROI PNGs (512 x 512).
// QuPath 0.4 or later. Run for the whole project with Automate > Script editor > Run > Run for project.
// Check the first exported mask visually (membrane lines white on black) before exporting everything.
import qupath.lib.images.servers.LabeledImageServer
import qupath.lib.regions.RegionRequest

def INITIALS   = ''      // '' for the primary annotator; the second annotator puts initials here, e.g. 'AB'
def DOWNSAMPLE = 2.0     // native 0.23 um/px -> 0.46 um/px (the resolution used by the model)
def OUT_DIR = buildFilePath(PROJECT_BASE_DIR, 'reference_masks' + (INITIALS ? '_' + INITIALS : ''))
mkdirs(OUT_DIR)

def labelServer = new LabeledImageServer.Builder(getCurrentImageData())
    .backgroundLabel(0)
    .downsample(DOWNSAMPLE)
    .addLabel('Membrane', 255)
    .lineThickness(2)
    .multichannelOutput(false)
    .build()

def rois = getAnnotationObjects().findAll { it.getName() != null && it.getName().contains('__roi') }
for (r in rois) {
    def request = RegionRequest.createInstance(labelServer.getPath(), DOWNSAMPLE, r.getROI())
    def name = r.getName() + (INITIALS ? '__' + INITIALS : '') + '.png'
    writeImageRegion(labelServer, request, buildFilePath(OUT_DIR, name))
    print 'wrote ' + name
}
""")
n_cores = WLDF.image_id.nunique() if len(WLDF) else 0
print(f"worklist: {len(WLDF)} ROIs from {n_cores} cores | expected Dice CI half-width ≈ ±{1.96 * 0.15 / np.sqrt(max(n_cores, 1)):.3f}")
(WL / "ANNOTATION_PROTOCOL.md").write_text(f"""# HER2 membrane annotation protocol (gastro-oesophageal adenocarcinoma, Ventana 4B5)
Purpose: reference masks for measuring the accuracy of an automated membrane segmentation. Your masks are the reference standard,
so annotate what you see. Please do not try to match the model.

## Material
* `worklist.csv`: {len(WLDF)} ROIs from {n_cores} test cores. Each ROI is {S}x{S} px at {WORK_MPP:.2f} µm/px (about {S * WORK_MPP:.0f} µm square).
* `qupath_geojson/<core>.geojson`: the ROI squares for each core, in native pixel coordinates of that core's JPEG.
* `rois/`: the same ROIs as PNGs (for a quick look).
* Do **not** open `pred_masks/` before annotating, so that the model does not anchor you.

## What to draw (QuPath 0.4 or later)
1. Create a QuPath project with the listed core JPEGs. For each image, import its own `qupath_geojson/<core>.geojson`
   (File > Import objects, or drag the file onto the image). Each ROI square carries its roi_id as its name.
2. Inside each square, trace **membranous** DAB of **tumour cells only** with the *Polyline* tool (or the brush for thick segments).
   Give each line the class **Membrane**.
   * Count complete, basolateral and lateral (U-shaped) staining. In gastric/GEA cancer, incomplete membranes are scored (Hofmann/Rüschoff).
   * Include weak membranes you would count when scoring: faint, discontinuous lines visible at x40.
   * Exclude cytoplasmic, nuclear and luminal-secretion DAB, stroma, normal glands, and edge or crush artefact.
3. If an ROI contains **no** tumour membrane staining, leave it empty. An empty mask is a valid and important answer.
4. Optional: add an area annotation of class **Tumor** around the tumour epithelium in the ROI.
5. Expected time: about 1 min for an IHC 0 ROI, 3–5 min for a 2+/3+ ROI. Allow roughly 4–6 h in total, split over sessions.

## Second annotator
ROIs with `second_annotator = True` in `worklist.csv` (about 21) should be annotated **independently** by a second pathologist who has not
seen the first annotator's masks. Their masks are saved with an `__<initials>` suffix.

## Export
Run the script `export_masks_qupath.groovy` (Automate > Script editor > Run for project). It writes one PNG per ROI:
`<roi_id>.png`, or `<roi_id>__<initials>.png` when you set INITIALS in the script. Lines are rasterised 2 px wide at 0.46 µm/px.
Put all PNGs in one folder, set `CFG.REF_MASK_DIR` to it, and rerun Section 17.

## Reporting
State the annotators' experience, the tools, that annotation was blind to model output, and the time spent.
""")

## 18b. Training convergence and computational cost
Reviewers of SCIE journals routinely ask whether the models converged and what they cost to run. This cell plots the
MIL training loss for each fold, and the segmentation students' loss and pseudo-label agreement per epoch.
It also tabulates the trainable parameters, the measured per-core inference time and the wall-clock time per pipeline section.
Interrupted-and-resumed sessions are summed.

In [ ]:
def n_params(m, trainable=True): return int(sum(p.numel() for p in m.parameters() if (p.requires_grad or not trainable)))
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
for i, n in enumerate([PROPOSED, "SANC − negative constraint", MAIN_BASELINE, "MaxPool"]):
    L = [l for l in MIL_RES.get(n, {}).get("loss", []) if len(l)]
    if not L: continue
    E = min(map(len, L)); A = np.array([l[:E] for l in L])
    ax[0].plot(np.arange(1, E + 1), A.mean(0), color=PAL[i], label=n); ax[0].fill_between(np.arange(1, E + 1), A.min(0), A.max(0), color=PAL[i], alpha=0.15)
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("training loss (mean, range over folds)"); ax[0].set_title("MIL")
for i, (n, m) in enumerate(SEG_METRICS.items()):
    h = pd.DataFrame(m["history"])
    ax[1].plot(h.epoch, h.train_loss, "-o", ms=3, color=PAL[i], label=n[:32])
    ax[2].plot(h.epoch, h.agree_Dice, "-o", ms=3, color=PAL[i], label=n[:32])
    ax[2].plot(h.epoch, h.agree_Dice_highconf, ":", lw=1, color=PAL[i])
    b = h[h.epoch == m["final"]["epoch"]].iloc[0]; ax[2].scatter([b.epoch], [b.agree_Dice], s=60, facecolor="none", edgecolor=PAL[i])
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("training loss"); ax[1].set_title("Segmentation students")
ax[2].set_xlabel("epoch"); ax[2].set_ylabel("pseudo-label agreement, Dice (solid: all pixels; dotted: high-conf)"); ax[2].set_title("Held-out agreement (circle = selected epoch)")
ax[2].legend(fontsize=7)
from matplotlib.ticker import MaxNLocator
for a in ax: a.xaxis.set_major_locator(MaxNLocator(integer=True))
ax[0].legend(fontsize=7, loc="upper right", framealpha=0.9)
savefig(fig, "figS2_convergence")

COST = []
use_embedding("emb")
for n, v in VARIANTS.items():
    try: COST.append(dict(component="MIL head", model=n, trainable_params=n_params(build(v))))
    except Exception as e: print("param count failed:", n, e)
for n, m in SEG_MODELS.items(): COST.append(dict(component="segmentation student", model=n, trainable_params=n_params(m)))
try: COST.append(dict(component="CNN baseline (Grad-CAM)", model=CFG.CAM_ARCH, trainable_params=n_params(timm.create_model(CFG.CAM_ARCH, pretrained=False, num_classes=1))))
except Exception as e: print("CAM param count failed:", e)
try:
    _e = timm.create_model(CFG.ENCODER.split(":", 1)[-1], pretrained=False, features_only=True, out_indices=(CFG.ENCODER_OUT_INDEX,))
    COST.append(dict(component="frozen tile encoder (not trained)", model=CFG.ENCODER, trainable_params=n_params(_e, trainable=False))); del _e
except Exception as e: print("encoder param count failed:", e)
COST = pd.DataFrame(COST); atomic_csv(COST, TAB / "compute_parameters.csv", index=False)
if "seg_seconds" in TSEG:
    _t = TSEG.seg_seconds.dropna()
    SEG_TIME = dict(n_cores=int(len(_t)), median_s=float(_t.median()), q25_s=float(_t.quantile(0.25)), q75_s=float(_t.quantile(0.75)),
                    tta=CFG.TTA, gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
else: SEG_TIME = {}
save_json(SEG_TIME, TAB / "test_segmentation_time_per_core.json")
try: TIMES = pd.read_csv(OUT / "timings.csv", on_bad_lines="skip")
except Exception: TIMES = pd.DataFrame(columns=["section", "seconds", "session"])
if "session" not in TIMES: TIMES = pd.DataFrame(columns=["section", "seconds", "session"])   # pre-v2.1 log format
_agg = dict(minutes=("seconds", lambda x: x.sum() / 60), sessions=("session", "nunique"))
for c, how in [("ram_gb", "max"), ("peak_gpu_gb", "max"), ("output_gb", "max")]:
    if c in TIMES: _agg[{"ram_gb": "max_RAM_GB", "peak_gpu_gb": "peak_GPU_GB", "output_gb": "output_folder_GB"}[c]] = (c, how)
TIMES = TIMES.groupby("section", sort=False).agg(**_agg).reset_index()
atomic_csv(TIMES, TAB / "compute_section_times.csv", index=False)
print(COST.to_string(index=False)); print("\nwhole-core test segmentation per core:", SEG_TIME); print(TIMES.round(1).to_string(index=False))

## 18c. Threshold-free localisation trade-off, publication figures, clean compute table
**Why.** A single 0.5 threshold can mislead. A model with near-zero scores everywhere (MaxPool) has almost no false positives at 0.5
simply because it rarely calls any tile positive. The fair comparison sweeps the threshold. For each model it traces the IHC-0 tile
false-positive rate against the fraction of tiles called positive in IHC-3 cores, then reads off the IHC-3 positivity at a fixed
false-positive budget (1% and 0.5%). CIs come from a core-level bootstrap (IHC-0 and IHC-3 cores resampled separately).

Tiles in IHC-3 cores are not all tumour, so "IHC-3 tile positivity" is a detection proxy, not a sensitivity. The curve is computed on the
test set (284 IHC-0 vs 8 IHC-3 cores) and on out-of-fold training cores (about 1,326 vs 114), which give much tighter intervals.

This cell also redraws Fig 8, Fig 10, Fig 12 and Fig S2 with readable labels and white backgrounds, and writes a de-duplicated
compute table.

In [ ]:
def _tile_sets(res, split):
    """Per-core tile probabilities for IHC-0 and IHC-3 cores (fold-averaged on test; OOF fold model on train)."""
    if split == "test":
        sc = per_fold_tile_scores(res, "inst")
        if sc is None: return None
        P = [sig(x.mean(0)) for x in sc]
        return [P[b] for b in neg0], [P[b] for b in ihc3]
    oof = res.get("oof")
    if oof is None or oof[0] is None or oof[0].get("inst") is None: return None
    n0 = np.where(TR.ihc_score.values == 0)[0]; n3 = np.where(TR.ihc_score.values == 3)[0]
    return [sig(oof[j]["inst"]) for j in n0], [sig(oof[j]["inst"]) for j in n3]

def _pos_at_fp(neg_cores, pos_cores, fp):
    t = np.quantile(np.concatenate(neg_cores), 1 - fp)                 # threshold giving the target pooled IHC-0 tile FP rate
    return float(np.mean(np.concatenate(pos_cores) > t))

OPC_METHODS = [m for m in [PROPOSED, "SANC − negative constraint", "SANC − stain features", "MaxPool", "CLAM-SB", "MeanPool",
                           "Grad-CAM CNN (per-image normalised CAM)"] if m in {**MIL_RES, **CAM_RES}]
OPC_ROWS, OPC_CURVES = [], {}
fp_grid = np.logspace(-4, np.log10(0.3), 60)
for split in ("test", "oof"):
    for m in OPC_METHODS:
        ts = _tile_sets({**MIL_RES, **CAM_RES}[m], split)
        if ts is None: continue
        neg, pos = ts
        OPC_CURVES[(m, split)] = [_pos_at_fp(neg, pos, f) for f in fp_grid]
        rb = np.random.default_rng(0); row = dict(method=m, split=split, n_IHC0_cores=len(neg), n_IHC3_cores=len(pos))
        for fp in (0.01, 0.005):
            v = _pos_at_fp(neg, pos, fp)
            bs = [_pos_at_fp([neg[i] for i in rb.integers(0, len(neg), len(neg))], [pos[i] for i in rb.integers(0, len(pos), len(pos))], fp)
                  for _ in range(500)]
            row.update({f"IHC3_tile_pos@FP{fp:g}": v, f"lo@FP{fp:g}": float(np.quantile(bs, 0.025)), f"hi@FP{fp:g}": float(np.quantile(bs, 0.975))})
        y = np.r_[np.ones(sum(map(len, pos))), np.zeros(sum(map(len, neg)))]
        row["tile_AUROC_IHC3_vs_IHC0"] = float(roc_auc_score(y, np.r_[np.concatenate(pos), np.concatenate(neg)]))
        OPC_ROWS.append(row)
OPC = pd.DataFrame(OPC_ROWS); atomic_csv(OPC, TAB / "localisation_operating_points.csv", index=False)
print("IHC-3 tile positivity at a fixed IHC-0 tile false-positive budget (core-bootstrap 95% CI):")
print(OPC.round(3).to_string(index=False))

LABEL = {PROPOSED: "SANC-MIL (proposed)", "SANC − negative constraint": "SANC without negative constraint",
         "SANC − stain features": "SANC without stain features", "MaxPool": "MaxPool", "CLAM-SB": "CLAM-SB", "MeanPool": "MeanPool",
         "Grad-CAM CNN (per-image normalised CAM)": "Grad-CAM CNN"}
fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.9), sharey=True)
def _n(split, col):
    v = OPC[OPC.split == split][col]; return int(v.iloc[0]) if len(v) else 0
for j, (split, ttl) in enumerate([("test", f"Test set ({_n('test', 'n_IHC0_cores')} IHC 0 vs {_n('test', 'n_IHC3_cores')} IHC 3 cores)"),
                                  ("oof", f"Out-of-fold training cores ({_n('oof', 'n_IHC0_cores')} vs {_n('oof', 'n_IHC3_cores')})")]):
    for i, m in enumerate(OPC_METHODS):
        if (m, split) not in OPC_CURVES: continue
        ax[j].plot(fp_grid * 100, np.array(OPC_CURVES[(m, split)]) * 100, color=PAL[i % len(PAL)], lw=2.2 if m == PROPOSED else 1.4,
                   ls="-" if m != "SANC − negative constraint" else "--", label=LABEL.get(m, m))
    ax[j].set_xscale("log"); ax[j].axvline(1, color="#888", lw=0.8, ls=":"); ax[j].set_title(ttl, fontsize=9)
    ax[j].set_xlabel("IHC 0 tiles called positive (%, log scale)")
ax[0].set_ylabel("IHC 3 tiles called positive (%)")
_h, _l = ax[0].get_legend_handles_labels()
fig.legend(_h, _l, loc="lower center", ncol=4, fontsize=7.5, bbox_to_anchor=(0.5, -0.12))
fig.suptitle("Localisation trade-off across all thresholds (dotted line: 1% false-positive budget)", fontsize=10)
savefig(fig, "fig11b_localisation_operating_curve")

# ---------- readable feature names for figures and tables ----------
def pretty_feature(f):
    src = "student" if f.startswith("stu_") else ("rule" if f.startswith("rule_") else "")
    g = re.sub(r"^(stu|rule)_", "", f)
    names = {"mem_weak_frac": "membranous DAB area, ≥weak", "mem_mod_frac": "membranous DAB area, ≥moderate",
             "mem_strong_frac": "membranous DAB area, ≥strong", "cell_cov>=0.3": "cells ≥30% membrane-covered",
             "cell_cov>=0.5": "cells ≥50% membrane-covered", "cell_cov>=0.7": "cells ≥70% membrane-covered",
             "cell_cov>=0.5&mod": "cells ≥50% covered at ≥moderate DAB", "cell_cov>=0.5&strong": "cells ≥50% covered at ≥strong DAB",
             "cell_median_cov_pos": "median coverage of stained cells", "cell_n_cells": "number of cells"}
    return f"{names.get(g, g)} ({src})" if src else names.get(g, g)

# ---------- Fig 10 redrawn ----------
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
for ax_, f in zip(axes, ["stu_mem_mod_frac", "stu_cell_cov>=0.5", "stu_cell_cov>=0.5&mod", "rule_mem_mod_frac"]):
    for k, g in enumerate(GROUP_ORDER):
        v = cf_te[cf_te.group == g][f].dropna().values if f in cf_te else []
        mk = "o"
        if len(v) == 0: v = cf_tr[cf_tr.group == g][f].dropna().values; mk = "x"
        ax_.scatter(np.full(len(v), k) + np.random.default_rng(k).uniform(-0.18, 0.18, len(v)), v, s=9, color=GROUP_COL[g], alpha=0.6, marker=mk)
        if len(v): ax_.hlines(np.median(v), k - 0.3, k + 0.3, color="#0b0b0b", lw=1.5)
    ax_.set_xticks(range(len(GROUP_ORDER))); ax_.set_xticklabels(["0", "1+", "2+ FISH−", "2+ FISH+", "3+"], fontsize=7)
    ax_.set_title(pretty_feature(f), fontsize=8)
axes[0].set_ylabel("fraction per core")
fig.text(0.5, -0.02, "Test cores (●); IHC 1+ shown from training cores (×) because the test set has none. Bars = medians.", ha="center", fontsize=7.5)
savefig(fig, "fig10_clinical_consistency")

# ---------- Fig 12 redrawn ----------
d = H3[H3.comparison == "3+ vs 2+"].pivot_table(index="feature", columns="split", values=["AUROC", "lo", "hi"])
if ("AUROC", "test") in d.columns:
    d = d.sort_values(("AUROC", "train"), ascending=False).head(12)
    fig, ax_ = plt.subplots(figsize=(7.5, 0.36 * len(d) + 1.3)); yy = np.arange(len(d))
    for off, sp, col, lab in [(-0.15, "train", PAL[0], "training cores (73 vs 114)"), (0.15, "test", PAL[7], "test cores (15 vs 8)")]:
        ax_.errorbar(d[("AUROC", sp)], yy + off, xerr=[d[("AUROC", sp)] - d[("lo", sp)], d[("hi", sp)] - d[("AUROC", sp)]],
                     fmt="o", ms=4, color=col, label=lab, capsize=2)
    ax_.axvline(0.5, color="#888", ls="--", lw=1); ax_.set_yticks(yy); ax_.set_yticklabels([pretty_feature(f) for f in d.index], fontsize=7)
    ax_.invert_yaxis(); ax_.set_xlabel("AUROC, IHC 3+ vs IHC 2+ (95% stratified bootstrap CI)"); ax_.legend(fontsize=7, loc="lower left")
    savefig(fig, "fig12_h3_3plus_vs_2plus")

# ---------- Fig 8 redrawn: white background, sequential maps that start at white, colour bars ----------
def heatmap_figure_pub(n_per=1, seed=CFG.SEED):
    r_ = np.random.default_rng(seed); picks = []
    for g in ["0", "2-", "2+", "3"]:
        s_ = TE[TE.group == g]
        if len(s_): picks += [s_.index[i] for i in r_.choice(len(s_), min(n_per, len(s_)), replace=False)]
    inst_s = per_fold_tile_scores(PROPOSED, "inst"); att_a = per_fold_tile_scores(MAIN_BASELINE, "attn")
    cam_n = per_fold_tile_scores(CAM_RES["Grad-CAM CNN (per-image normalised CAM)"], "inst") if CAM_RES else None
    cols = [("Core", None, None), ("DAB optical density", "YlOrBr", (0, CALIB["t_strong"] * 1.2)),
            ("Membranous DAB ≥ moderate\n(fraction per pixel block)", "Reds", None), ("Grad-CAM\n(per-image normalised)", "viridis", (0, 1)),
            ("ABMIL attention\n(scaled to max)", "viridis", (0, 1)), ("SANC-MIL P(positive tile)", "RdBu_r", (0, 1)), ("SANC-MIL uncertainty\n(SD over 5 folds)", "Greys", (0, 0.35))]
    fig, axes_all = plt.subplots(len(picks) + 1, len(cols), figsize=(2.3 * len(cols), 2.35 * len(picks) + 0.7),
                                 gridspec_kw={"height_ratios": [1] * len(picks) + [0.05]})
    axes, caxes = axes_all[:-1], axes_all[-1]                                  # last row holds the colour bars
    ims = [None] * len(cols)
    for r, b in enumerate(picks):
        iid = TE.image_id[b]; mpz = np.load(MAP_DIR / f"{iid}.npz"); th = cv2.cvtColor(cv2.imread(str(THUMB_DIR / f"{iid}.jpg")), cv2.COLOR_BGR2RGB)
        shp = mpz["dab"].shape
        tmask = cv2.resize(mpz["tissue"].astype(np.uint8), (shp[1], shp[0]), interpolation=cv2.INTER_NEAREST) > 0
        inside = lambda a: np.where(tmask, a.astype(np.float32), np.nan)
        pr = sig(inst_s[b]); att = att_a[b].mean(0); att = att / (att.max() + 1e-9)
        camv = tile_grid_map(iid, sig(cam_n[b]).mean(0), shp) if cam_n is not None else np.full(shp, np.nan, np.float32)
        mm = inside(mpz["mem_mod"])
        data = [th, inside(mpz["dab"]), mm, camv, tile_grid_map(iid, att, shp), tile_grid_map(iid, pr.mean(0), shp), tile_grid_map(iid, pr.std(0), shp)]
        for c, ((title, cmap, lim), img) in enumerate(zip(cols, data)):
            a = axes[r, c]; a.axis("off")
            if cmap is None: a.imshow(img)
            else:
                cm_ = plt.get_cmap(cmap).copy(); cm_.set_bad("white")
                vmin, vmax = lim if lim else (0, max(0.05, float(np.nanpercentile(img, 99.5)) if np.isfinite(img).any() else 0.05))
                ims[c] = a.imshow(cv2.resize(img, (th.shape[1], th.shape[0]), interpolation=cv2.INTER_NEAREST), cmap=cm_, vmin=vmin, vmax=vmax)
            if r == 0: a.set_title(title, fontsize=8)
        lab = {"0": "IHC 0", "2-": "IHC 2+, FISH−", "2+": "IHC 2+, FISH+", "3": "IHC 3+"}[TE.group[b]]
        axes[r, 0].text(5, 25, f"{lab}   P(HER2+)={pu['pm'][b]:.2f}", color="k", fontsize=7, bbox=dict(fc="w", ec="none", alpha=0.85))
    for c, im in enumerate(ims):
        if im is not None:
            cb = fig.colorbar(im, cax=caxes[c], orientation="horizontal"); cb.ax.tick_params(labelsize=6)
        else: caxes[c].axis("off")
    savefig(fig, "fig08_localization_maps")
heatmap_figure_pub()

# ---------- Fig S2 label fix ----------
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
for i, n in enumerate([PROPOSED, "SANC − negative constraint", MAIN_BASELINE, "MaxPool"]):
    L = [l for l in MIL_RES.get(n, {}).get("loss", []) if len(l)]
    if not L: continue
    E = min(map(len, L)); A = np.array([l[:E] for l in L])
    ax[0].plot(np.arange(1, E + 1), A.mean(0), color=PAL[i], label=LABEL.get(n, n)); ax[0].fill_between(np.arange(1, E + 1), A.min(0), A.max(0), color=PAL[i], alpha=0.15)
for i, (n, m_) in enumerate(SEG_METRICS.items()):
    h = pd.DataFrame(m_["history"])
    ax[1].plot(h.epoch, h.train_loss, "-o", ms=3, color=PAL[i], label=n.replace(" (proposed)", ""))
    ax[2].plot(h.epoch, h.agree_Dice, "-o", ms=3, color=PAL[i]); ax[2].plot(h.epoch, h.agree_Dice_highconf, ":", lw=1, color=PAL[i])
    b_ = h[h.epoch == m_["final"]["epoch"]].iloc[0]; ax[2].scatter([b_.epoch], [b_.agree_Dice], s=60, facecolor="none", edgecolor=PAL[i])
ax[0].set(xlabel="epoch", ylabel="training loss", title="MIL (mean, range over 5 folds)"); ax[0].legend(fontsize=7)
ax[1].set(xlabel="epoch", ylabel="training loss", title="Segmentation students"); ax[1].legend(fontsize=6.5)
ax[2].set(xlabel="epoch", ylabel="agreement with pseudo-labels (Dice)", title="Held-out agreement (solid: all pixels;\ndotted: high-confidence; circle: kept epoch)")
from matplotlib.ticker import MaxNLocator
for a in ax: a.xaxis.set_major_locator(MaxNLocator(integer=True))
savefig(fig, "figS2_convergence")

# ---------- clean compute table: a section logged twice in one session = a re-executed cell, not new work ----------
try:
    _T = pd.read_csv(OUT / "timings.csv", on_bad_lines="skip")
    if "session" in _T:
        _T["_n"] = _T.groupby(["section", "session"]).cumcount()
        dup = _T[_T._n > 0]
        clean = _T[_T._n == 0].groupby("section", sort=False).agg(minutes=("seconds", lambda x: x.sum() / 60), sessions=("session", "nunique"),
                                                                  peak_GPU_GB=("peak_gpu_gb", "max"), max_RAM_GB=("ram_gb", "max")).reset_index()
        clean["note"] = np.where(clean.sessions > 1, "summed over sessions (a resumed session adds only the unfinished work)", "")
        atomic_csv(clean, TAB / "compute_section_times_clean.csv", index=False)
        print("\nCompute per section (re-executed cells removed):\n", clean.round(1).to_string(index=False))
        if len(dup): print("removed duplicate entries:", list(dup.section))
except Exception as e:
    print("compute table not rebuilt:", e)
section_time("figures + operating curve")

## 19. Results summary and claims ledger
The summary below is generated automatically from the tables. The ledger separates what the evidence **supports** from what remains **unverified**.

In [ ]:
def md(df):
    try: return df.to_markdown(index=False)
    except Exception: return "```\n" + df.to_string(index=False) + "\n```"
def fmt_ci(r, k): return f"{r[k]:.3f} (95% CI {r[k + '_lo']:.3f}–{r[k + '_hi']:.3f})"
pr = CLS[CLS.method == PROPOSED].iloc[0]; bl = CLS[CLS.method == MAIN_BASELINE].iloc[0]; ir = CLS[CLS.method == "Stain intensity rule (Pisula 2023)"].iloc[0]
d_ab = DELTA[DELTA.comparison.str.endswith(MAIN_BASELINE)].iloc[0]
locp = LOC.set_index("method")
# ---- evidence-gated claim wording: a hypothesis is called "supported" only when the pre-stated criterion is met ----
_h1 = H1[(H1.A == PROPOSED) & (H1.B == "SANC − negative constraint")] if len(H1) else pd.DataFrame()
H1_OK = bool(len(_h1)) and bool((_h1.hi < 0).all())
H1_TXT = ("SUPPORTED: in every seed and on both the test set and OOF train, the CI of the per-core FP difference lies below 0 "
          f"(mean diff {_h1['diff'].mean():+.3f})." if H1_OK else
          "NOT consistently supported: at least one seed/split has a CI including 0 — report as a trend, see table.") if len(_h1) else "not computed"
_h3 = H3[H3.comparison == "3+ vs 2+"] if len(H3) else pd.DataFrame()
if len(_h3) and (_h3.split == "test").any():
    _best = _h3[_h3.split == "train"].sort_values("AUROC", ascending=False).feature.iloc[0]       # chosen on TRAIN, evaluated on test
    _bt = _h3[(_h3.split == "test") & (_h3.feature == _best)]
    H3_TXT = (f"train-selected feature `{_best}`: test AUROC {_bt.AUROC.iloc[0]:.3f} (95% CI {_bt.lo.iloc[0]:.3f}–{_bt.hi.iloc[0]:.3f}, "
              f"n={_bt.n_pos.iloc[0]} vs {_bt.n_neg.iloc[0]}) — " + ("SUPPORTED (CI excludes 0.5)." if _bt.lo.iloc[0] > 0.5 else "NOT supported (CI includes 0.5).")) if len(_bt) else "not computable on test"
else: H3_TXT = "not computable"
lines = [
    "# Results summary (auto-generated)", "",
    f"*Run mode:* `{RUN_MODE}` | encoder `{CFG.ENCODER}` (pretrained={ENCODER_PRETRAINED}) | train cores {len(TR)} | test cores {len(TE)} "
    f"({int(ystat_te.sum())} HER2+) | test groups: " + ", ".join(f"{g}: {int((TE.group == g).sum())}" for g in GROUP_ORDER), "",
    "## Classification (temporally independent test set)", "", md(CLS_PRETTY), "",
    f"Paired bootstrap ΔAUROC, {PROPOSED} vs {MAIN_BASELINE}: {d_ab.dAUC:+.3f} (95% CI {d_ab.lo:+.3f} to {d_ab.hi:+.3f}, p={d_ab.p_boot:.3f}).", "",
    "All paired comparisons with the proposed model (bootstrap p, Holm-corrected):", "", md(DELTA.round(4)), "",
    "Leakage sensitivity (OOF AUROC, random vs proxy-grouped CV):", "", md(LEAK.round(4)), "",
    "Abstention / equivocal routing:", "", "```", json.dumps(ab, indent=1, default=str), "```", "",
    "## Localisation (grounded checks + proxies)", "", md(LOC.round(3)), "",
    f"Variability over {CFG.N_SEEDS} repeated CV seeds (mean, SD):", "", "```", SEED_SUMMARY.to_string(), "```", "",
    "### H1: paired per-core tests of IHC-0 false-positive tile fraction (A − B < 0 favours A)", "", md(H1.round(4)), "",
    "### Threshold-free trade-off: IHC-3 tile positivity at a fixed IHC-0 tile false-positive budget", "", md(OPC.round(3)), "",
    "## Segmentation student (agreement with PSEUDO-labels on held-out cores — not accuracy)", "", md(SEGT.round(3)), "",
    "Test-set proxies:", "", md(SEG_PROXY.round(4)), "",
    "## Clinical consistency of segmentation-derived features", "", md(CLIN.round(3)), "", md(TREND.round(3)), "",
    "### H3: single-feature separation of 3+ vs 2+ and of 2+/FISH+ vs 2+/FISH− (direction fixed on train; top 5 per block)", "",
    md(H3.groupby(["comparison", "split"], sort=False).head(5).round(3)) if len(H3) else "(not computable)", "",
    "## Computational cost", "", md(COST), "", f"Whole-core test segmentation: {SEG_TIME}", "", md(TIMES.round(1)), "",
    "## Claims ledger", "",
    "**Supported by this run (with the stated uncertainty):**",
    f"- Image-level HER2 status on the independent test set: SANC-MIL AUROC {fmt_ci(pr, 'AUROC')}; ABMIL {fmt_ci(bl, 'AUROC')}; stain-intensity rule {fmt_ci(ir, 'AUROC')}.",
    "- Instance positivity on IHC-0 test cores (label-grounded specificity), seed mean ± SD: " + "; ".join(
        f"{m}: FP@0.5 {SEED_SUMMARY.loc[m, ('IHC0_tile_FP_rate@0.5', 'mean')]:.3f} ± {SEED_SUMMARY.loc[m, ('IHC0_tile_FP_rate@0.5', 'std')]:.3f}"
        for m in SEED_SUMMARY.index if ("IHC0_tile_FP_rate@0.5", "mean") in SEED_SUMMARY.columns and not np.isnan(SEED_SUMMARY.loc[m, ("IHC0_tile_FP_rate@0.5", "mean")])) + ".",
    "- Grad-CAM baseline (per-image normalised, as in the original draft): IHC-0 tile FP@0.5 = "
    + (f"{locp.loc['Grad-CAM CNN (per-image normalised CAM)', 'IHC0_tile_FP_rate@0.5']:.3f}" if "Grad-CAM CNN (per-image normalised CAM)" in locp.index else "n/a") + ".",
    f"- ISH routing (threshold tuned on OOF for {CFG.ISH_ROUTE_TARGET_SENS:.0%} sensitivity): test sensitivity for IHC 2 = {ab['ISH_routing_sensitivity_for_true_IHC2']:.2f}, routing rate in non-IHC 2 = {ab['ISH_routing_rate_in_non_IHC2']:.2f}.",
    f"- H1 (negative-bag constraint reduces IHC-0 false-positive tiles): {H1_TXT}",
    f"- H3 (membrane/cell features separate IHC 3+ from 2+): {H3_TXT}",
    "- Classification differences between MIL variants: see Holm-corrected table; a difference is claimed only where p_holm < 0.05.",
    "- Monotonic association of membrane-derived features with IHC score (see Spearman ρ / Kruskal–Wallis table).",
    "- QC and leakage analyses: see qc_summary.json and leakage_sensitivity.csv.", "",
    "**NOT established (do not claim):**",
    "- Pixel-level segmentation accuracy of membranes, tumour regions or cells: no expert masks were available"
    + (" — EXCEPT the reference-mask results in reference_mask_metrics_summary.csv." if CFG.REF_MASK_DIR and Path(CFG.REF_MASK_DIR).exists() else "."),
    "- That 'pseudo-label agreement' Dice equals accuracy — it measures reproduction of the training signal.",
    "- That attention / evidence maps mark the cells a pathologist would score — they mark what drives the model.",
    "- Any IHC 1+ test performance (the test set has no 1+ cores); IHC 2+ results rest on n≈15 cores.",
    "- That the model can replace FISH for IHC 2+: equivocal cores are routed to ISH by design.",
    "- Generalisation across institutions/scanners: single centre, single scanner here.", "",
    f"Next step for validated segmentation: annotate `annotation_worklist/` ({len(WLDF)} ROIs, {n_cores} cores; expected Dice CI ±{1.96 * 0.15 / np.sqrt(max(n_cores, 1)):.2f}).",
]
(OUT / "RESULTS_SUMMARY.md").write_text("\n".join(lines))
section_time("report")
env = dict(version="v2.1", run_mode=RUN_MODE, torch=torch.__version__, timm=timm.__version__, numpy=np.__version__, sklearn=__import__("sklearn").__version__,
           skimage=skimage.__version__, cv2=cv2.__version__, gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
           encoder=CFG.ENCODER, encoder_pretrained=ENCODER_PRETRAINED, encoder2=CFG.ENCODER2, seed=CFG.SEED,
           cfg={k: v for k, v in vars(CFG).items() if not k.startswith("_") and isinstance(v, (int, float, str, tuple, bool, type(None)))})
save_json(env, OUT / "environment_and_config.json")
print("\n".join(lines[:12]))
print(f"\nAll outputs in {OUT}: tables/, figures/ (PNG 300 dpi + PDF), test_outputs/, annotation_worklist/, models/, RESULTS_SUMMARY.md")

## 20. Export a review package
This cell zips the tables, figures, summaries, timings and environment, plus a stratified sample of masks and pseudo-labels. Download
`her2_v2_results_for_review.zip` from the Output panel.

In [ ]:
import glob, zipfile
Z = Path("/kaggle/working/her2_v2_results_for_review.zip" if IS_KAGGLE else OUT.parent / "her2_v2_results_for_review.zip")
files = glob.glob(f"{OUT}/tables/*") + glob.glob(f"{OUT}/figures/*.png") + glob.glob(f"{OUT}/*.md") + glob.glob(f"{OUT}/*.json") + glob.glob(f"{OUT}/*.csv")
files += glob.glob(f"{CACHE}/*.json") + glob.glob(f"{CACHE}/*.csv")
files += glob.glob(f"{OUT}/annotation_worklist/*.csv") + glob.glob(f"{OUT}/annotation_worklist/*.md") + glob.glob(f"{OUT}/annotation_worklist/*.groovy")
files += glob.glob(f"{OUT}/annotation_worklist/qupath_geojson/*.geojson")
p_ = pd.read_csv(TAB / "test_predictions.csv"); ids = list(p_[p_.group.isin(["2-", "2+", "3"])].image_id)
s_ = pd.read_csv(CACHE / "test_seg_summary.csv"); z_ = s_[s_.group == "0"]
ids += list(z_.sort_values("stu_mem_mod", ascending=False).image_id[:12]) + list(z_.sample(min(6, len(z_)), random_state=0).image_id)
files += [f for i in ids for f in glob.glob(f"{OUT}/test_outputs/{i}_overlay.jpg")]
c_ = pd.read_csv(CACHE / "seg_crops_index.csv").groupby("group", group_keys=False).apply(lambda d: d.sample(min(4, len(d)), random_state=0))
files += list(c_.crop) + list(c_.label)
files += sorted(glob.glob(f"{OUT}/annotation_worklist/rois/*.png"))[:6] + sorted(glob.glob(f"{OUT}/annotation_worklist/pred_masks/*.png"))[:6]
with zipfile.ZipFile(Z, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(set(files)):
        if os.path.exists(f): zf.write(f, os.path.relpath(f, OUT.parent))
print(f"{Z}  {Z.stat().st_size / 1e6:.1f} MB  ({len(set(files))} files)")

## 21. External validation and ablation on an independent dataset (optional)
**Purpose.** This section tests whether the models and measurements trained on the gastro-oesophageal TMA cores transfer to
data from another centre, scanner and organ. Nothing is re-trained: the stain matrix, DAB thresholds, tile encoder, MIL fold
models and segmentation student are applied **frozen**.

**Supported datasets** (the file layout is detected automatically; a generic CSV also works):
| Dataset | Content | What it tests |
|---|---|---|
| Warwick HER2 Scoring Contest (breast, HER2 IHC whole slides, NanoZoomer) | HER2 score 0–3+ and the pathologists' **percentage of cells with complete membrane staining** (PCMS) | Membrane-coverage features against an expert membrane measure; status; H1 on IHC 0 slides |
| ADEL (breast, HER2 IHC images or tiles, CC BY 4.0) | HER2 expression level per image | Status and monotonicity with score; H1 on IHC 0 images |
| Generic | a CSV with `path`, `ihc_score` and optionally `her2_status`, `pcms`, `mpp` | Same, as available |

**Analyses** (all pre-specified here, before seeing external results):
1. **Status**: AUROC of SANC-MIL, ABMIL, the stain-intensity rule and the membrane-feature model. Positive = IHC 3+ or 2+/ISH+;
   negative = IHC 0/1+. IHC 2+ without an ISH result is excluded from status metrics.
2. **Monotonicity**: Spearman ρ between IHC score and P(HER2+), the expected ordinal score, and membrane features.
3. **H1 replication (ablation)**: tile false-positive rate in external IHC 0 images, SANC-MIL vs the same model trained without the
   negative-bag constraint; paired Wilcoxon and bootstrap CI.
4. **Membrane measure vs experts** (Warwick only): Spearman ρ between PCMS and the fraction of cells with ≥70% membrane coverage
   (primary, the closest analogue of "complete"); other coverage bins as secondary.
5. **Stain-calibration ablation**: every analysis is run twice, with the training stain matrix (frozen) and with a stain matrix
   re-estimated on the external images (no labels used). DAB thresholds stay frozen in both.

**Caveat to state in the paper.** Breast HER2 scoring requires complete, circumferential membrane staining; gastric scoring also
counts basolateral and lateral staining. The external results are therefore a cross-organ transfer test, not a validation of
gastric scoring rules.

**How to use.** Attach the dataset with *Add Input*, set `EXT_ROOT` (and `EXT_MPP` for plain images without scale metadata),
then run this cell. Results go to `tables/external_<name>_*.csv`, `figures/fig13_external_<name>.png` and
`external_<name>_results.zip`. The cell resumes per image if interrupted.

In [ ]:
try: import openslide                                     # whole-slide reader (Warwick .ndpi); not needed for plain images
except ImportError:
    try: _pip("openslide-bin"); _pip("openslide-python")
    except Exception as e: print("openslide unavailable (whole slides cannot be read):", e)
EXT_ROOT = os.environ.get("HER2_EXT_ROOT")          # e.g. "/kaggle/input/warwick-her2" or "/kaggle/input/adel-her2"
EXT_NAME = os.environ.get("HER2_EXT_NAME")          # "warwick" | "adel" | "generic" (auto-detected when None)
EXT_CSV = os.environ.get("HER2_EXT_CSV")            # generic manifest (path, ihc_score[, her2_status, pcms, mpp])
EXT_MPP = float(os.environ["HER2_EXT_MPP"]) if os.environ.get("HER2_EXT_MPP") else None   # µm/px of plain images, if known
EXT_STAIN_MODES = ("frozen", "refit")
EXT_MAX_REGIONS = 16 if RUN_MODE != "debug" else 4  # regions of 6x6 tiles sampled per slide (whole slides only)
EXT_FEAT_CROPS = CFG.FEAT_CROPS_PER_CORE
EXT_WSI_MIN_PX = int(os.environ.get("HER2_EXT_WSI_MIN_PX", 12000))   # a .tif larger than this (per side) is read as a whole slide

def _find_cols(df, *keys):
    """First column whose normalised name contains all keys (case/space/punctuation-insensitive)."""
    norm = {c: re.sub(r"[^a-z0-9]", "", str(c).lower()) for c in df.columns}
    for c, n in norm.items():
        if all(k in n for k in keys): return c
    return None

def _score_to_int(v):
    s = str(v).strip().lower().replace("+", "")
    m = re.search(r"[0-3]", s)
    return int(m.group()) if m else np.nan

WSI_EXT = {".ndpi", ".svs", ".mrxs", ".scn", ".vms", ".bif", ".tiff", ".tif", ".kfb"}
IMG_EXT2 = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

def _read_table(p):
    p = Path(p)
    return pd.read_excel(p) if p.suffix.lower() in (".xlsx", ".xls") else pd.read_csv(p)

def build_manifest(root, name=None, csv=None):
    """Return a DataFrame with path, ihc_score, her2_status (NaN if unknown), pcms (NaN if absent), mpp (NaN if unknown)."""
    root = Path(root)
    if csv:
        m = _read_table(csv)
        pc = _find_cols(m, "path") or _find_cols(m, "file") or _find_cols(m, "image")
        out = pd.DataFrame({"path": [str(root / str(x)) if not os.path.isabs(str(x)) else str(x) for x in m[pc]],
                            "ihc_score": m[_find_cols(m, "score") or _find_cols(m, "her2")].map(_score_to_int)})
        for col, keys in [("her2_status", ("status",)), ("pcms", ("percent",)), ("mpp", ("mpp",))]:
            c = _find_cols(m, *keys); out[col] = pd.to_numeric(m[c], errors="coerce") if c else np.nan
        return out.assign(dataset=name or "generic")
    tables = [p for p in root.rglob("*") if p.suffix.lower() in (".csv", ".xlsx", ".xls")]
    files = [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in (WSI_EXT | IMG_EXT2)]
    # --- Warwick: whole slides named like "<case>_HER2.ndpi" (H&E slides are ignored) + a ground-truth table with score + percentage ---
    for t in tables:
        g = _read_table(t)
        sc, pc = _find_cols(g, "score"), _find_cols(g, "percent")
        if sc and pc:
            case = _find_cols(g, "case") or g.columns[0]
            her2 = [p for p in files if p.suffix.lower() in WSI_EXT and re.search(r"her2", p.name, re.I)]
            rows = []
            for _, r in g.iterrows():
                cid = re.sub(r"\D", "", str(r[case]))
                key = lambda p: re.sub(r"\D", "", re.split(r"her2", p.stem, flags=re.I)[0]) or re.sub(r"\D", "", p.parent.name)
                hit = [p for p in her2 if key(p).lstrip("0") == cid.lstrip("0") and cid]
                if hit: rows.append(dict(path=str(hit[0]), ihc_score=_score_to_int(r[sc]), her2_status=np.nan,
                                         pcms=pd.to_numeric(r[pc], errors="coerce"), mpp=np.nan))
            if rows:
                print(f"external: Warwick-style layout ({t.name}): {len(rows)} HER2 slides with score + percentage")
                return pd.DataFrame(rows).assign(dataset=name or "warwick")
    # --- ADEL / generic image folders: a labels table naming image files (or folders of tiles) + a HER2 level ---
    for t in tables:
        g = _read_table(t)
        lc = _find_cols(g, "her2") or _find_cols(g, "label") or _find_cols(g, "score") or _find_cols(g, "class")
        fc = _find_cols(g, "file") or _find_cols(g, "image") or _find_cols(g, "name") or _find_cols(g, "id")
        if not (lc and fc): continue
        by_stem = defaultdict(list)
        for p in files: by_stem[p.stem].append(p)
        dirs = {p.name: p for p in root.rglob("*") if p.is_dir()}
        rows = []
        for _, r in g.iterrows():
            key = Path(str(r[fc])).stem
            if key in by_stem: path = str(by_stem[key][0])
            elif key in dirs: path = str(dirs[key])                         # a folder of tiles from one slide
            else: continue
            rows.append(dict(path=path, ihc_score=_score_to_int(r[lc]), her2_status=np.nan, pcms=np.nan, mpp=np.nan))
        if rows:
            print(f"external: labelled image layout ({t.name}): {len(rows)} images or tile folders")
            return pd.DataFrame(rows).assign(dataset=name or "adel")
    raise RuntimeError(f"No usable label table found under {root}. Provide HER2_EXT_CSV with columns path, ihc_score[, her2_status, pcms, mpp].")

# ---------------- reading: whole slide, single image, or folder of tiles -> work-level regions ----------------
def _open_slide(path):
    import openslide
    return openslide.OpenSlide(str(path))

def _is_wsi(path):
    p = Path(path)
    if p.is_dir() or p.suffix.lower() not in WSI_EXT: return False
    if p.suffix.lower() in (".tif", ".tiff"):
        try:
            Image.MAX_IMAGE_PIXELS = None
            with Image.open(p) as im: return im.size[0] * im.size[1] > EXT_WSI_MIN_PX ** 2
        except Exception: return True
    return True

def estimate_mpp_from_nuclei(rgb, M):
    """Fallback scale estimate when an image carries no µm/px: median equivalent diameter of haematoxylin-dominant nuclei,
    assuming 7 µm for a typical carcinoma nucleus. Reported, never silently trusted."""
    od = od_np(rgb, estimate_I0(rgb)).reshape(-1, 3)
    c = np.clip(od @ np.linalg.pinv(np.asarray(M, np.float32)), 0, None).reshape(rgb.shape[0], rgb.shape[1], 3)
    hmask = (c[..., 0] > np.percentile(c[..., 0], 85)) & (c[..., 0] > c[..., 1])
    hmask = ndi.binary_opening(hmask, iterations=1)
    lab, n = ndi.label(hmask)
    if n < 20: return np.nan
    areas = np.bincount(lab.ravel())[1:]; areas = areas[(areas > 10) & (areas < np.percentile(areas, 95))]
    return 7.0 / (2 * np.sqrt(np.median(areas) / np.pi)) if len(areas) else np.nan

def iter_regions(row, rng, region_px=6 * CFG.TILE):
    """Yield (work-level RGB region, I0) for one external item; at most EXT_MAX_REGIONS regions per slide."""
    path = Path(row.path)
    if path.is_dir():                                           # folder of tiles: each tile is a region
        tiles = sorted([p for p in path.rglob("*") if p.suffix.lower() in IMG_EXT2])
        if len(tiles) > EXT_MAX_REGIONS * 9: tiles = [tiles[i] for i in np.sort(rng.choice(len(tiles), EXT_MAX_REGIONS * 9, replace=False))]
        imgs = [np.asarray(Image.open(t).convert("RGB")) for t in tiles]
        mpp = row.mpp if np.isfinite(row.mpp) else EXT_MPP
        I0 = estimate_I0(np.concatenate([i.reshape(-1, 3) for i in imgs])[None])
        for im in imgs:
            f = mpp / WORK_MPP
            yield cv2.resize(im, (max(1, int(im.shape[1] * f)), max(1, int(im.shape[0] * f))), interpolation=cv2.INTER_AREA), I0
        return
    if _is_wsi(path):
        sl = _open_slide(path)
        mpp = float(sl.properties.get("openslide.mpp-x", "nan")) if np.isnan(row.mpp) else row.mpp
        if not np.isfinite(mpp): mpp = EXT_MPP or 0.25
        W0, H0 = sl.dimensions
        th = np.asarray(sl.get_thumbnail((max(1, W0 // 32), max(1, H0 // 32))).convert("RGB")); sx = W0 / th.shape[1]
        I0 = estimate_I0(th)
        om = od_np(th, I0).max(-1); thr = float(np.clip(np.percentile(om, 10) + 0.045, *CFG.OD_TISSUE_RANGE))
        tis = cv2.morphologyEx((om > thr).astype(np.uint8), cv2.MORPH_OPEN, np.ones((3, 3), np.uint8)) > 0
        side0 = region_px * WORK_MPP / mpp                       # region side in level-0 pixels
        step = max(1, int(side0 / sx))
        cand = [(y, x) for y in range(0, tis.shape[0] - step, step) for x in range(0, tis.shape[1] - step, step)
                if tis[y:y + step, x:x + step].mean() > 0.6]
        if not cand: return
        cand = [cand[i] for i in np.sort(rng.choice(len(cand), min(EXT_MAX_REGIONS, len(cand)), replace=False))]
        level = sl.get_best_level_for_downsample(max(1.0, WORK_MPP / mpp))
        ds = sl.level_downsamples[level]
        for y, x in cand:
            X0, Y0 = int(x * sx), int(y * sx); sz = int(round(side0 / ds))
            reg = np.asarray(sl.read_region((X0, Y0), level, (sz, sz)).convert("RGB"))
            yield cv2.resize(reg, (region_px, region_px), interpolation=cv2.INTER_AREA), I0
        sl.close(); return
    Image.MAX_IMAGE_PIXELS = None
    im = np.asarray(Image.open(path).convert("RGB"))             # single image (tile, core or large field)
    mpp = row.mpp if np.isfinite(row.mpp) else EXT_MPP
    if mpp is None or not np.isfinite(mpp):
        raise RuntimeError("Plain images need a scale: set HER2_EXT_MPP (µm/px) or an mpp column (see the scale check printed above).")
    f = mpp / WORK_MPP; I0 = estimate_I0(im)
    work = cv2.resize(im, (max(1, int(im.shape[1] * f)), max(1, int(im.shape[0] * f))), interpolation=cv2.INTER_AREA); del im
    H, W = work.shape[:2]
    if H <= 2 * region_px and W <= 2 * region_px:
        yield work, I0; return
    om = od_np(thumb_of(work), I0).max(-1) > 0.1                # large image: sample tissue regions like a whole slide
    R = CFG.THUMB_REL; step = region_px // R
    cand = [(y, x) for y in range(0, om.shape[0] - step, step) for x in range(0, om.shape[1] - step, step) if om[y:y + step, x:x + step].mean() > 0.6]
    for i in np.sort(rng.choice(len(cand), min(EXT_MAX_REGIONS, len(cand)), replace=False)) if cand else []:
        y, x = cand[i]; yield work[y * R:y * R + region_px, x * R:x * R + region_px], I0

# ---------------- per-region features, identical in definition to pass 1 (Section 4) ----------------
@torch.no_grad()
def region_tile_features(engine, work, I0):
    T = CFG.TILE; H, W = work.shape[:2]
    if H < T or W < T: return None
    mp = engine.maps(work, I0, None, None)
    ny, nx = H // T, W // T
    cut = lambda m: m.float()[..., :ny * T, :nx * T]
    pool = lambda m: F.avg_pool2d(cut(m), T, T)[0, 0]
    tis = mp["tissue"]; tf = pool(tis); tisN = tf.clamp(min=1e-6)
    frac = lambda m: pool(m & tis) / tisN
    dab = mp["DAB"] * tis; dmean = pool(dab) / tisN; dsq = pool(dab * dab) / tisN
    lap = F.conv2d(F.pad(mp["gray"], (1, 1, 1, 1), mode="reflect"), LAP); focus = pool(lap * lap) - pool(lap) ** 2
    feats = torch.stack([tf, dmean, torch.sqrt((dsq - dmean ** 2).clamp(min=0)),
                         frac(mp["DAB"] >= CALIB["t_weak"]), frac(mp["DAB"] >= CALIB["t_mod"]), frac(mp["DAB"] >= CALIB["t_strong"]),
                         frac(mp["mem_weak"]), frac(mp["mem_mod"]), frac(mp["mem_strong"]), frac(mp["nonmem"]),
                         frac(mp["nuc"] & (mp["DAB"] >= CALIB["t_weak"])), frac(mp["nuc"]), focus, torch.zeros_like(tf)], -1)
    assert feats.shape[-1] == len(HAND_NAMES), "feature definition drifted from pass 1"
    keep = torch.nonzero(tf >= CFG.MIN_TILE_TISSUE).cpu().numpy()
    return mp, feats[keep[:, 0], keep[:, 1]].cpu().numpy().astype(np.float32) if len(keep) else np.zeros((0, len(HAND_NAMES)), np.float32), keep * T

@torch.no_grad()
def process_item(row, engine, enc, rng):
    """Tiles -> embeddings + stain features (for MIL) and 512-px crops -> membrane / cell features (as in Section 14)."""
    embs, hands, S = [], [], CFG.SEG_CROP; acc = defaultdict(float); cov = {"stu": [[], []], "rule": [[], []]}; n_crops = 0
    for work, I0 in iter_regions(row, rng):
        out = region_tile_features(engine, work, I0)
        if out is None: continue
        mp, hand, coords = out
        if len(coords):
            xg = torch.from_numpy(np.ascontiguousarray(work)).to(DEVICE).permute(2, 0, 1); T = CFG.TILE
            for b in range(0, len(coords), CFG.ENC_BATCH):
                cb = coords[b:b + CFG.ENC_BATCH]
                batch = torch.stack([xg[:, y:y + T, x:x + T] for y, x in cb]).float() / 255
                with autocast(): embs.append(gpu_safe(enc, batch).float().cpu().numpy().astype(np.float16))
            hands.append(hand)
        H, W = work.shape[:2]
        if n_crops < EXT_FEAT_CROPS and H >= S and W >= S:      # one 512-px crop per region for cell morphometry
            tis = mp["tissue"][0, 0]
            y, x = (H - S) // 2, (W - S) // 2
            ti = tis[y:y + S, x:x + S].cpu().numpy()
            if ti.mean() > 0.3:
                xb = torch.from_numpy(np.ascontiguousarray(work[y:y + S, x:x + S]))[None].to(DEVICE).permute(0, 3, 1, 2).float() / 255
                pmem, _, _ = seg_predict(SEG_P, xb, tta=1)
                d = mp["DAB"][0, 0, y:y + S, x:x + S].cpu().numpy(); hh = mp["H"][0, 0, y:y + S, x:x + S].cpu().numpy()
                stu = remove_small((pmem[0].cpu().numpy() >= 0.5) & ti, CFG.MIN_MEM_OBJ_PX)
                rule = mp["mem_weak"][0, 0, y:y + S, x:x + S].cpu().numpy() & ti
                acc["tissue_px"] += ti.sum()
                for nm, mk in [("stu", stu), ("rule", rule)]:
                    for k in ("weak", "mod", "strong"): acc[f"{nm}_mem_{k}"] += (mk & (d >= CALIB[f"t_{k}"])).sum()
                res = cell_analysis((hh, d, ti, [stu, rule]))
                for mi, nm in enumerate(["stu", "rule"]): cov[nm][0].append(res[mi][0]); cov[nm][1].append(res[mi][1])
                n_crops += 1
        del mp
    if not hands: return None
    hand = np.concatenate(hands); emb = np.concatenate(embs)
    if len(hand) > CFG.MAX_TILES_PER_IMAGE:
        k = np.sort(rng.choice(len(hand), CFG.MAX_TILES_PER_IMAGE, replace=False)); hand, emb = hand[k], emb[k]
    rec = {k + "_frac": v / max(acc["tissue_px"], 1) for k, v in acc.items() if k != "tissue_px"}
    for nm in ("stu", "rule"):
        c = np.concatenate(cov[nm][0]) if cov[nm][0] else np.zeros(0); dd = np.concatenate(cov[nm][1]) if cov[nm][1] else np.zeros(0)
        rec.update(summarize_cells(c, dd, f"{nm}_cell_"))
    if n_crops == 0:                                            # images too small for a 512-px crop: no membrane/cell features
        rec = {k: np.nan for k in rec}
    return dict(emb=emb, hand=hand, feats=rec, n_tiles=len(hand), n_crops=n_crops)

def _bag_tensors(item):
    h = ((transform_hand(item["hand"]) - HAND_MU) / HAND_SD).astype(np.float32)
    e = torch.from_numpy(item["emb"].astype(np.float32))[None].to(DEVICE); hd = torch.from_numpy(h)[None].to(DEVICE)
    return e, hd, torch.ones(1, e.shape[1], dtype=torch.bool, device=DEVICE)

@torch.no_grad()
def predict_external(models, item, T=1.0):
    e, hd, mk = _bag_tensors(item); outs = [m.eval()(e, hd, mk) for m in models]
    logit = float(np.mean([o["logit"].item() for o in outs])) / T
    inst = np.mean([o["inst"][0].cpu().numpy() for o in outs], 0) if outs[0]["inst"] is not None else None
    ordl = np.mean([o["ord"][0].cpu().numpy() for o in outs], 0) if "ord" in outs[0] else None
    return logit, inst, ordl

import hashlib
def run_external():
    if not EXT_ROOT or not Path(EXT_ROOT).exists():
        print("External validation skipped: set HER2_EXT_ROOT to an attached dataset (see the markdown above)."); return None
    man = build_manifest(EXT_ROOT, EXT_NAME, EXT_CSV)
    name = man.dataset.iloc[0]; man = man[man.ihc_score.notna()].reset_index(drop=True)
    man["item_id"] = [re.sub(r"[^A-Za-z0-9]+", "_", Path(p).stem)[:80] + "_" + hashlib.md5(str(p).encode()).hexdigest()[:6] for p in man.path]   # stable if rows are added
    print(f"external dataset '{name}': {len(man)} labelled items | score counts {man.ihc_score.value_counts().sort_index().to_dict()}")
    # scale check for plain images: estimated µm/px from nuclear size, printed so a wrong EXT_MPP is caught early
    first = Path(man.path.iloc[0])
    if not _is_wsi(first) and EXT_MPP is None and man.mpp.isna().all():   # only when no scale is given (the estimate is crude)
        f0 = first if first.is_file() else next(p for p in first.rglob("*") if p.suffix.lower() in IMG_EXT2)
        est = estimate_mpp_from_nuclei(np.asarray(Image.open(f0).convert("RGB")), CALIB["M"])
        print(f"scale check: nuclear-size estimate {est:.3f} µm/px (assumes 7 µm nuclei; UNRELIABLE on pale HER2 IHC nuclei - give the true scale)")
        if EXT_MPP is None and man.mpp.isna().all():
            print("!! No µm/px given; using the nuclear-size estimate. Set HER2_EXT_MPP for a reproducible result.")
            man["mpp"] = est
    use_embedding("emb")
    enc = TileEncoder(CFG.ENCODER).to(DEVICE).eval()
    variants = {"SANC-MIL": PROPOSED, "SANC without negative constraint": "SANC − negative constraint", "ABMIL": MAIN_BASELINE}
    MODELS = {k: fold_models(v) for k, v in variants.items()}   # unsaved variants are re-trained deterministically (fold seed = fold index)
    TEMPS = {k: float(CLS[CLS.method == v].temp.iloc[0]) for k, v in variants.items()}
    # interpretable comparators fitted on ALL training cores (as in Sections 8 and 16), applied unchanged
    rule_lr = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000)).fit(XS_tr[:, :1], ystat_tr)
    feat_cols = FEATSETS["student + cell morphometry"]
    feat_lr = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=3000, C=0.5)).fit(cf_tr[feat_cols].fillna(0).values, cf_tr.her2_status)
    engines = {"frozen": ENGINE}
    if "refit" in EXT_STAIN_MODES:                                 # unsupervised stain re-estimation on external thumbnails
        thumbs = []
        for _, r in man.sample(min(40, len(man)), random_state=CFG.SEED).iterrows():
            try:
                reg = next(iter_regions(r, np.random.default_rng(0)))
                thumbs.append((thumb_of(reg[0]), reg[1]))
            except Exception as e: print("refit: skipped", r.path, type(e).__name__)
        M_ext, minfo = macenko_global(thumbs)
        engines["refit"] = StainEngine(M_ext, CALIB); print("refit stain matrix vs Ruifrok:", minfo)
    out_rows = []
    for mode, eng in engines.items():
        log = ProgressLog(f"external_{name}_{mode}")
        for _, r in tqdm(man.iterrows(), total=len(man), desc=f"external {name} [{mode}]"):
            if r.item_id in log: continue
            rng = np.random.default_rng(_img_seed(r.item_id))
            try: item = process_item(r, eng, enc, rng)
            except Exception as e: print(f"  {r.item_id}: {type(e).__name__}: {str(e)[:120]}"); item = None
            if item is None: log.add(r.item_id, dict(_skip=True)); continue
            rec = dict(item_id=r.item_id, mode=mode, ihc_score=int(r.ihc_score), her2_status=r.her2_status, pcms=r.pcms, n_tiles=item["n_tiles"],
                       n_crops=item["n_crops"], **item["feats"])
            for k, ms in MODELS.items():
                lg, inst, ordl = predict_external(ms, item, TEMPS[k])
                rec[f"logit::{k}"] = lg
                if inst is not None: rec[f"tileFP::{k}"] = float(np.mean(sig(inst) > 0.5)); rec[f"tileMax::{k}"] = float(sig(inst).max())
                if ordl is not None: rec[f"exp_score::{k}"] = float((coral_probs(ordl[None]) * np.arange(4)).sum())
            h = item["hand"]; g = lambda n: h[:, HAND_NAMES.index(n)]
            rec["logit::Stain-intensity rule"] = float(rule_lr.decision_function([[g("dab_mean").max()]])[0])
            fv = [item["feats"].get(c, np.nan) for c in feat_cols]
            rec["logit::Membrane + cell features"] = float(feat_lr.decision_function([fv])[0]) if np.all(np.isfinite(fv)) else np.nan
            log.add(r.item_id, rec)
        out_rows += [x for x in log.records() if not x.get("_skip")]
    EXT = pd.DataFrame(out_rows).drop(columns=["_key"], errors="ignore")
    atomic_csv(EXT, TAB / f"external_{name}_per_item.csv", index=False)
    return name, EXT

if "pretty_feature" not in globals():          # defined in section 18c; fallback so this section also runs when 18c was skipped
    def pretty_feature(f):
        src = "student" if f.startswith("stu_") else ("rule" if f.startswith("rule_") else "")
        g = re.sub(r"^(stu|rule)_", "", f)
        names = {"mem_weak_frac": "membranous DAB area, ≥weak", "mem_mod_frac": "membranous DAB area, ≥moderate",
                 "mem_strong_frac": "membranous DAB area, ≥strong", "cell_cov>=0.3": "cells ≥30% membrane-covered",
                 "cell_cov>=0.5": "cells ≥50% membrane-covered", "cell_cov>=0.7": "cells ≥70% membrane-covered",
                 "cell_cov>=0.5&mod": "cells ≥50% covered at ≥moderate DAB", "cell_cov>=0.5&strong": "cells ≥50% covered at ≥strong DAB",
                 "cell_median_cov_pos": "median coverage of stained cells", "cell_n_cells": "number of cells"}
        return f"{names.get(g, g)} ({src})" if src else names.get(g, g)

def analyse_external(name, EXT):
    rows, h1_rows, pc_rows = [], [], []
    for mode, d in EXT.groupby("mode"):
        st = d.her2_status.where(d.her2_status.notna(), np.where(d.ihc_score == 3, 1.0, np.where(d.ihc_score <= 1, 0.0, np.nan)))
        ok = st.notna().values; y = st.values[ok].astype(int)
        for c in [c for c in d.columns if c.startswith("logit::")]:
            m = c.split("::")[1]; p = d[c].values.astype(float); fin = ok & np.isfinite(p)
            yy = st.values[fin].astype(int)
            r = dict(mode=mode, method=m, n_pos=int(yy.sum()), n_neg=int((1 - yy).sum()))
            if len(set(yy)) > 1:
                r["status_AUROC"] = roc_auc_score(yy, p[fin]); lo, hi = strat_boot_auc(yy, p[fin]); r["AUROC_lo"], r["AUROC_hi"] = lo, hi
            r["spearman_vs_score"] = stats.spearmanr(d.ihc_score, p, nan_policy="omit").correlation
            rows.append(r)
        for c in [c for c in d.columns if c.startswith("exp_score::")]:   # ordinal head: monotonicity AND status discrimination
            r = dict(mode=mode, method=c.split("::")[1] + " (expected ordinal score)", spearman_vs_score=stats.spearmanr(d.ihc_score, d[c]).correlation,
                     QWK_vs_score=cohen_kappa_score(d.ihc_score, np.clip(np.rint(d[c]), 0, 3).astype(int), weights="quadratic"))
            p = d[c].values.astype(float); fin = ok & np.isfinite(p); yy = st.values[fin].astype(int)
            if len(set(yy)) > 1:
                r.update(n_pos=int(yy.sum()), n_neg=int((1 - yy).sum()), status_AUROC=roc_auc_score(yy, p[fin]))
                r["AUROC_lo"], r["AUROC_hi"] = strat_boot_auc(yy, p[fin])
            rows.append(r)
        for f in ["stu_mem_mod_frac", "stu_mem_strong_frac", "stu_cell_cov>=0.5", "stu_cell_cov>=0.7", "stu_cell_cov>=0.5&mod"]:
            if f in d and d[f].notna().sum() > 3:
                rows.append(dict(mode=mode, method=f"feature: {pretty_feature(f)}", spearman_vs_score=stats.spearmanr(d.ihc_score, d[f], nan_policy="omit").correlation))
        z = d[d.ihc_score == 0]                                  # H1 replication on external IHC 0 items
        a, b = "tileFP::SANC-MIL", "tileFP::SANC without negative constraint"
        if len(z) >= 5 and a in z and b in z:
            h1_rows.append(dict(mode=mode, **paired_test(z[a].values, z[b].values),
                                any_tile_gt09_SANC=float((z["tileMax::SANC-MIL"] > 0.9).mean()),
                                any_tile_gt09_noconstraint=float((z["tileMax::SANC without negative constraint"] > 0.9).mean())))
        if d.pcms.notna().sum() >= 10:                            # Warwick: membrane coverage vs expert percentage
            q = d[d.pcms.notna() & d.get("stu_cell_cov>=0.7", pd.Series(np.nan, index=d.index)).notna()]
            for f in ["stu_cell_cov>=0.7", "stu_cell_cov>=0.5", "stu_cell_cov>=0.3", "rule_cell_cov>=0.7", "stu_cell_cov>=0.5&mod"]:
                if f not in q: continue
                rho = stats.spearmanr(q.pcms, q[f]).correlation; rb = np.random.default_rng(0)
                bs = [stats.spearmanr(*q.iloc[i][["pcms", f]].values.T).correlation for i in (rb.integers(0, len(q), len(q)) for _ in range(1000))]
                pc_rows.append(dict(mode=mode, feature=pretty_feature(f), primary=f == "stu_cell_cov>=0.7", n=len(q), spearman_rho=rho,
                                    lo=np.nanquantile(bs, 0.025), hi=np.nanquantile(bs, 0.975)))
    R, H1E, PC = pd.DataFrame(rows), pd.DataFrame(h1_rows), pd.DataFrame(pc_rows)
    atomic_csv(R, TAB / f"external_{name}_summary.csv", index=False)
    if len(H1E): atomic_csv(H1E, TAB / f"external_{name}_h1_negative_constraint.csv", index=False)
    if len(PC): atomic_csv(PC, TAB / f"external_{name}_pcms_agreement.csv", index=False)
    print(R.round(3).to_string(index=False))
    if len(H1E): print("\nH1 on external IHC 0 items (A = SANC-MIL, B = without negative constraint):\n", H1E.round(4).to_string(index=False))
    if len(PC): print("\nMembrane coverage vs pathologists' % of completely stained cells:\n", PC.round(3).to_string(index=False))
    # figure: score-wise distributions of P(HER2+) and the primary membrane feature, frozen stain mode
    d = EXT[EXT["mode"] == "frozen"]
    fig, ax = plt.subplots(1, 3 if len(PC) else 2, figsize=(12 if len(PC) else 8.5, 3.4))
    for j, (col, lab) in enumerate([("logit::SANC-MIL", "SANC-MIL P(HER2+)"), ("stu_cell_cov>=0.7", "cells ≥70% membrane-covered")]):
        for s in range(4):
            v = d[d.ihc_score == s][col].dropna().values
            if col.startswith("logit"): v = sig(v)
            ax[j].scatter(np.full(len(v), s) + np.random.default_rng(s).uniform(-0.18, 0.18, len(v)), v, s=10, alpha=0.6, color=PAL[s])
            if len(v): ax[j].hlines(np.median(v), s - 0.3, s + 0.3, color="#0b0b0b", lw=1.5)
        ax[j].set_xticks(range(4)); ax[j].set_xticklabels(["0", "1+", "2+", "3+"]); ax[j].set_xlabel("external HER2 score"); ax[j].set_title(lab, fontsize=9)
    if len(PC):
        q = d[d.pcms.notna()]; ax[2].scatter(q.pcms, q["stu_cell_cov>=0.7"] * 100, s=12, color=PAL[0], alpha=0.7)
        ax[2].set_xlabel("pathologists' % completely stained cells"); ax[2].set_ylabel("% cells ≥70% membrane-covered (model)")
        ax[2].set_title(f"ρ = {stats.spearmanr(q.pcms, q['stu_cell_cov>=0.7']).correlation:.2f}", fontsize=9)
    fig.suptitle(f"External dataset: {name} (frozen models, frozen stain calibration)", fontsize=10)
    savefig(fig, f"fig13_external_{name}")
    import zipfile
    zp = OUT / f"external_{name}_results.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in list(TAB.glob(f"external_{name}_*")) + list(FIG.glob(f"fig13_external_{name}*")):
            zf.write(f, f.relative_to(OUT.parent))
    print(f"\nexternal results: {zp}")
    return R, H1E, PC

_ext = run_external()
if _ext is not None:
    EXT_RESULTS = analyse_external(*_ext)
    section_time(f"external validation ({_ext[0]})")